In [ ]:
# 1. Reproducible Colab Free setup (retain the runtime's CUDA-enabled torch)
import os, sys, subprocess, importlib.util
os.environ['WANDB_DISABLED']='true'
os.environ['TOKENIZERS_PARALLELISM']='false'
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==4.57.1','datasets==4.3.0','accelerate==1.11.0','sentencepiece==0.2.1','protobuf','safetensors','scikit-learn','pandas','numpy','matplotlib','seaborn'])
import torch
assert torch.cuda.is_available(), 'GPU unavailable: select a free T4 runtime before continuing.'
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,'CUDA:',torch.version.cuda)
from pathlib import Path
ROOT=Path('/content/latent_commitment'); ROOT.mkdir(exist_ok=True)
RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
MODEL_DIR=ROOT/'best_model'
print('Setup complete. Upload the original dataset ZIP to /content using the Files panel.')


In [ ]:
# 2. Preserve original partitions; fail closed on leakage or schema errors
import json, zipfile, hashlib, itertools, time, random, warnings
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback, set_seed
from datasets import Dataset
set_seed(42)
LABELS=['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT']
NAMES=['No Commitment','Latent Commitment','Qualified Commitment','Explicit Commitment']
label2id={name:i for i,name in enumerate(LABELS)}
id2label={i:name for name,i in label2id.items()}
zip_path=Path('/content/latent_commitment_leakage_free_splits.zip')
# Accept either the original ZIP or byte-identical extracted CSVs.
if not zip_path.exists():
    expected={'train':'b1f16c3a7e90a2a5f6b9b79954bb34afad3f17bd24a914a02d74886324c55d0b','validation':'46f6faf70fc96546211bb791fc0284ed6bab89da775555fecadc1469083c6ca5','test':'999cba094817ee3027b9dfc5e498bb842ae4e1cd5f7e018a0c656284f7e749f7'}
    for s,h in expected.items():
        p=Path('/content')/('latent_commitment_'+s+'.csv')
        assert p.exists() and hashlib.sha256(p.read_bytes()).hexdigest()==h, 'Missing or modified original CSV: '+s
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
        for s in expected: z.write(Path('/content')/('latent_commitment_'+s+'.csv'),arcname='latent_commitment_'+s+'.csv')
    print('Verified all three CSV hashes against the supplied archive; packaged without modifying data.')
data={}; audit={}
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
    for split,n in [('train',7000),('validation',1000),('test',2000)]:
        with z.open('latent_commitment_'+split+'.csv') as f:
            df=pd.read_csv(f)
        assert len(df)==n, (split,len(df))
        required=['model_input_text','commitment_label','target_utterance','contrast_set_id']
        assert all(c in df for c in required)
        assert not df[required].isna().any().any(), 'Missing required data'
        assert df['model_input_text'].str.strip().ne('').all()
        assert set(df.commitment_label)==set(LABELS), 'Unexpected labels'
        assert df.contrast_set_id.astype(str).str.strip().ne('').all()
        assert df.target_utterance.str.strip().ne('').all()
        df['labels']=df.commitment_label.map(label2id).astype(int)
        data[split]=df
        audit[split]={'rows':len(df),'class_counts':df.commitment_label.value_counts().to_dict()}
for a,b in itertools.combinations(data,2):
    audit[a+'_'+b]={}
    for col in ['target_utterance','contrast_set_id']:
        overlap=set(data[a][col].astype(str).str.strip()) & set(data[b][col].astype(str).str.strip())
        audit[a+'_'+b][col]=len(overlap)
        assert not overlap, (a,b,col,'overlap',len(overlap))
audit['dataset_sha256']=hashlib.sha256(zip_path.read_bytes()).hexdigest()
(RESULTS/'dataset_validation.json').write_text(json.dumps(audit,indent=2))
print('ALL DATA CHECKS PASSED',json.dumps(audit,indent=2))
def metrics(y,p):
    precision,recall,f1,_=precision_recall_fscore_support(y,p,labels=range(4),average='macro',zero_division=0)
    return dict(accuracy=accuracy_score(y,p),macro_precision=precision,macro_recall=recall,macro_f1=f1,weighted_f1=f1_score(y,p,labels=range(4),average='weighted',zero_division=0))
def compute_metrics(pred):
    logits=pred.predictions[0] if isinstance(pred.predictions,tuple) else pred.predictions
    return metrics(pred.label_ids,np.argmax(logits,axis=-1))
baseline_rows=[]
for column in ['target_utterance','model_input_text']:
    baseline=make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=100000,sublinear_tf=True),LogisticRegression(C=1.0,max_iter=2000,random_state=42))
    baseline.fit(data['train'][column],data['train'].labels)
    row={'model':'TF-IDF + Logistic Regression','input':column,'split':'validation',**metrics(data['validation'].labels,baseline.predict(data['validation'][column]))}
    baseline_rows.append(row)
pd.DataFrame(baseline_rows).to_csv(RESULTS/'baseline_comparison.csv',index=False)
print('VALIDATION BASELINES'); display(pd.DataFrame(baseline_rows))
tokenizer=AutoTokenizer.from_pretrained('microsoft/deberta-v3-base',use_fast=False)
truncation=[]
for split,df in data.items():
    lengths=[]
    for start in range(0,len(df),256):
        lengths.extend(map(len,tokenizer(df.model_input_text.iloc[start:start+256].tolist(),truncation=False)['input_ids']))
    row={'split':split,'examples':len(df),'truncated':int(np.sum(np.array(lengths)>256)),'truncated_fraction':float(np.mean(np.array(lengths)>256)),'max_tokens':int(max(lengths))}
    truncation.append(row)
    if row['truncated_fraction']>0.05: warnings.warn(str(row)+'; retaining requested max_length=256.')
pd.DataFrame(truncation).to_csv(RESULTS/'truncation_report.csv',index=False)
print('TRUNCATION AT 256 TOKENS'); display(pd.DataFrame(truncation))
# Only train/validation are tokenized for model selection. Test prediction is deferred.
def make_dataset(split):
    ds=Dataset.from_dict({'text':data[split].model_input_text.tolist(),'labels':data[split].labels.tolist()})
    return ds.map(lambda b:tokenizer(b['text'],truncation=True,max_length=256),batched=True,remove_columns=['text'])
train_ds=make_dataset('train'); val_ds=make_dataset('validation')
print('READY FOR TRAINING')


In [ ]:
# 3. Train exclusively on train; select exclusively by validation macro-F1
import shutil, gc
assert 'train_ds' in globals(), 'Run the data validation cell successfully first.'
assert torch.cuda.is_available()
class PersistProgress(TrainerCallback):
    def on_log(self,args,state,control,**kwargs):
        (RESULTS/'trainer_log.json').write_text(json.dumps(state.log_history,indent=2))
    def on_save(self,args,state,control,**kwargs):
        (RESULTS/'training_state.json').write_text(json.dumps({'step':state.global_step,'epoch':state.epoch,'best_model_checkpoint':state.best_model_checkpoint,'best_metric':state.best_metric},indent=2))
        # Epoch checkpoints persist in /content; no paid services or Drive authorization.
        print('Checkpoint safely written at step',state.global_step,flush=True)
model=AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base',num_labels=4,id2label=id2label,label2id=label2id)
args=TrainingArguments(output_dir=str(ROOT/'checkpoints'),num_train_epochs=3,learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=0.01,warmup_ratio=0.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=2,logging_steps=25,seed=42,data_seed=42,report_to='none',dataloader_num_workers=2,save_safetensors=True)
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8),compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2),PersistProgress()])
started=time.time()
train_output=trainer.train()
training_seconds=time.time()-started
trainer.save_model(str(MODEL_DIR)); tokenizer.save_pretrained(str(MODEL_DIR))
(MODEL_DIR/'label_mappings.json').write_text(json.dumps({'label2id':label2id,'id2label':id2label,'display_names':NAMES},indent=2))
logs=trainer.state.log_history
pd.DataFrame([r for r in logs if 'loss' in r]).to_csv(RESULTS/'training_metrics.csv',index=False)
pd.DataFrame([r for r in logs if 'eval_loss' in r]).to_csv(RESULTS/'validation_metrics.csv',index=False)
val_metrics=trainer.evaluate(val_ds)
selected={'model':'DeBERTa-v3-base','input':'model_input_text','split':'validation',**{k.removeprefix('eval_'):v for k,v in val_metrics.items() if k.removeprefix('eval_') in metrics([0,1,2,3],[0,1,2,3])}}
pd.DataFrame(baseline_rows+[selected]).to_csv(RESULTS/'baseline_comparison.csv',index=False)
run_info={'training_seconds':training_seconds,'gpu':torch.cuda.get_device_name(0),'best_checkpoint':trainer.state.best_model_checkpoint,'best_validation_macro_f1':trainer.state.best_metric,'global_step':trainer.state.global_step,'completed_epochs':trainer.state.epoch,'seed':42,'model':'microsoft/deberta-v3-base','max_length':256,'fp16':True,'gradient_accumulation':2,'train_metrics':train_output.metrics}
(RESULTS/'run_info.json').write_text(json.dumps(run_info,indent=2))
assert trainer.state.best_model_checkpoint is not None
assert np.isfinite(trainer.state.best_metric)
print('TRAINING COMPLETED',json.dumps(run_info,indent=2)); display(pd.DataFrame(baseline_rows+[selected]))


In [ ]:
# 4. One final held-out test pass; persist predictions to prevent accidental repeats
import matplotlib.pyplot as plt, seaborn as sns
assert (MODEL_DIR/'config.json').exists(), 'Complete training first.'
assert 'run_info' in globals()
cache=RESULTS/'heldout_predictions.npz'
if cache.exists():
    cached=np.load(cache); test_logits=cached['logits']; y_true=cached['labels']
    print('Reusing saved held-out predictions; no second model test pass.')
else:
    test_ds=make_dataset('test')
    test_output=trainer.predict(test_ds,metric_key_prefix='test')
    test_logits=test_output.predictions; y_true=test_output.label_ids
    np.savez_compressed(cache,logits=test_logits,labels=y_true)
y_pred=test_logits.argmax(axis=-1)
test_metrics=metrics(y_true,y_pred)
pd.DataFrame([test_metrics]).to_csv(RESULTS/'test_metrics.csv',index=False)
report=classification_report(y_true,y_pred,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
pd.DataFrame(report).T.to_csv(RESULTS/'classification_report.csv')
cm=confusion_matrix(y_true,y_pred,labels=range(4)); pd.DataFrame(cm,index=NAMES,columns=NAMES).to_csv(RESULTS/'confusion_matrix.csv')
sns.set_theme(style='whitegrid')
def line_chart(rows,key,title,ylabel,filename):
    fig,ax=plt.subplots(figsize=(8,5)); ax.plot([r.get('epoch',r['step']) for r in rows],[r[key] for r in rows],marker='o',markersize=3)
    ax.set(xlabel='Epoch',ylabel=ylabel,title=title); fig.tight_layout(); fig.savefig(RESULTS/filename,dpi=160); plt.close(fig)
line_chart([r for r in logs if 'loss' in r],'loss','DeBERTa training loss','Loss','training_loss.png')
line_chart([r for r in logs if 'eval_loss' in r],'eval_loss','Validation loss','Loss','validation_loss.png')
line_chart([r for r in logs if 'eval_macro_f1' in r],'eval_macro_f1','Validation macro-F1','Macro-F1','validation_macro_f1.png')
fig,ax=plt.subplots(figsize=(9,7)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES,ax=ax)
ax.set(xlabel='Predicted class',ylabel='True class',title='Held-out test confusion matrix'); plt.xticks(rotation=25,ha='right'); plt.yticks(rotation=0); fig.tight_layout(); fig.savefig(RESULTS/'confusion_matrix.png',dpi=160); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,5)); ax.bar(NAMES,[report[n]['f1-score'] for n in NAMES]); ax.set(ylim=(0,1.05),ylabel='F1',title='Held-out test per-class F1'); plt.xticks(rotation=20,ha='right'); fig.tight_layout(); fig.savefig(RESULTS/'per_class_f1.png',dpi=160); plt.close(fig)
confusions=sorted([(int(cm[i,j]),NAMES[i],NAMES[j]) for i in range(4) for j in range(4) if i!=j],reverse=True)
print('TEST METRICS',test_metrics); print('LATENT COMMITMENT',report[NAMES[1]]); print('Largest off-diagonal confusions',confusions[:4])
(RESULTS/'error_analysis.json').write_text(json.dumps({'latent_commitment':report[NAMES[1]],'confusions':[{'count':n,'true':a,'predicted':b} for n,a,b in confusions]},indent=2))
# Reload saved weights and tokenizer, then compare logits on a validation example.
trainer.model.to('cpu'); gc.collect(); torch.cuda.empty_cache()
reloaded_tokenizer=AutoTokenizer.from_pretrained(MODEL_DIR,use_fast=False)
reloaded_model=AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to('cuda').eval()
probe=reloaded_tokenizer(data['validation'].model_input_text.iloc[0],return_tensors='pt',truncation=True,max_length=256)
with torch.no_grad():
    original_logits=trainer.model(**probe).logits.cpu()
    reload_logits=reloaded_model(**{k:v.to('cuda') for k,v in probe.items()}).logits.cpu()
assert torch.allclose(original_logits,reload_logits,atol=2e-4,rtol=2e-4), 'Reload logits mismatch'
run_info['reload_verified']=True
print('SAVED MODEL RELOAD VERIFIED')


In [ ]:
# 5. Illustrative inference, reproducibility manifest, and verified exports
import importlib.metadata as metadata
from google.colab import _message, files

def predict_commitment(context,target_utterance):
    text=f'[CONTEXT] {context} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target_utterance}'
    batch=reloaded_tokenizer(text,return_tensors='pt',truncation=True,max_length=256)
    reloaded_model.eval()
    with torch.no_grad():
        probs=torch.softmax(reloaded_model(**{k:v.to('cuda') for k,v in batch.items()}).logits,dim=-1)[0].cpu().numpy()
    k=int(probs.argmax())
    return {'predicted_category':k,'predicted_label':NAMES[k],'confidence':float(probs[k]),'probabilities':{name:float(p) for name,p in zip(NAMES,probs)}}
examples=[
(0,'[lead]: Can you arrange the customer demo?','I cannot take responsibility for the demo this week.'),
(0,'[manager]: Who will update the security checklist?','I can explain the checklist, but I am not agreeing to update it.'),
(0,'[colleague]: Can you cover the reception desk on Tuesday?','Tuesday is unavailable for me; please find someone else.'),
(1,'[lead]: We need someone to investigate the billing discrepancy.','The invoices are open on my screen and my afternoon is clear.'),
(1,'[manager]: The new hire still needs an onboarding guide.','I have the previous guide beside me and a quiet hour before lunch.'),
(1,'[colleague]: Nobody has looked into the stalled deployment yet.','I am already in the deployment logs, following the last successful run.'),
(2,'[lead]: Can you finish the supplier comparison by Thursday?','I can finish it by Thursday if procurement sends the prices today.'),
(2,'[manager]: Will you host the retrospective?','I will host it provided we move it after the client call.'),
(2,'[colleague]: Can you prepare the board charts?','I can prepare the charts once finance approves the source figures.'),
(3,'[lead]: Who will send the release summary?','I will write and send the release summary by four today.'),
(3,'[manager]: Please arrange a handover for the support team.','I will schedule the handover and send the agenda tomorrow morning.'),
(3,'[colleague]: Can you check the migration report?','Yes, I will review the report and return my comments by noon.')]
known=set(pd.concat([d.target_utterance for d in data.values()]).str.strip())
rows=[]
for intended,context,target in examples:
    assert target.strip() not in known
    result=predict_commitment(context,target)
    rows.append({'context':context,'target_utterance':target,'illustrative_intended_label':NAMES[intended],**{k:v for k,v in result.items() if k!='probabilities'},**{'probability_'+k:v for k,v in result['probabilities'].items()}})
pd.DataFrame(rows).to_csv(RESULTS/'sample_predictions.csv',index=False)
print('ILLUSTRATIVE PREDICTIONS: NOT TEST-SET PERFORMANCE'); display(pd.DataFrame(rows))
packages=['torch','transformers','datasets','accelerate','scikit-learn','pandas','numpy','matplotlib','seaborn','sentencepiece','protobuf','safetensors']
(RESULTS/'requirements.txt').write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in packages))
run_info['inference_verified']=True; run_info['test_evaluated']=True
run_info['test_metrics']=test_metrics
run_info['dataset_sha256']=audit['dataset_sha256']
run_info['base_model_revision']=getattr(model.config,'_commit_hash',None)
(RESULTS/'run_info.json').write_text(json.dumps(run_info,indent=2))
comparison=pd.read_csv(RESULTS/'baseline_comparison.csv')
readme=f'''# Latent Commitment Inference

Trained microsoft/deberta-v3-base on the original 7000/1000/2000 partitions.
Labels: 0 No Commitment; 1 Latent Commitment; 2 Qualified Commitment; 3 Explicit Commitment.
Input format: [CONTEXT] context [TARGET SPEAKER] speaker [TARGET UTTERANCE] utterance.
The two-argument inference helper defaults the speaker to employee.
Sequence length 256, dynamic padding, seed 42, AdamW, learning rate 2e-5, batch 8, accumulation 2, eval batch 16, weight decay .01, warmup .1, FP16, up to 3 epochs, early stopping patience 2.
Best checkpoint selected exclusively by validation macro-F1; test predicted once and cached.
GPU: {run_info['gpu']}; training duration: {training_seconds:.2f} seconds.
Best validation macro-F1: {run_info['best_validation_macro_f1']}.
Test metrics: {json.dumps(test_metrics)}.
Latent Commitment class: {json.dumps(report[NAMES[1]])}.
Validation comparisons: {comparison.to_json(orient='records')}.
Top off-diagonal confusions: {confusions[:4]}.
The baseline comparison measures the effect of context for TF-IDF. A causal context benefit for DeBERTa requires a separate target-only DeBERTa ablation, which was not trained.
The supplied split checks establish exact target and contrast-set separation, not absence of template or semantic similarity. Performance may not generalize to naturally occurring workplace messages.
Illustrative predictions are 12 newly authored examples, not a benchmark. Confidence is uncalibrated softmax.
Model reload and inference were verified. Check run_info.json, truncation_report.csv, dataset_validation.json, and trainer_log.json for provenance.
No paid service was enabled. Drive backup is optional and requires separate authorization; runtime checkpoints are ephemeral.
To rerun: select a free T4, upload the original ZIP to /content, and run cells in order.
To load: AutoTokenizer.from_pretrained(model_directory,use_fast=False) and AutoModelForSequenceClassification.from_pretrained(model_directory).
requirements.txt records the exact installed versions; retain Colab's CUDA-enabled torch rather than replacing it unnecessarily.
'''
(RESULTS/'README.md').write_text(readme)
# Export this actual notebook using Colab's frontend, including measured outputs.
notebook=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(RESULTS/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(notebook,indent=1))
required=['Latent_Commitment_DeBERTa_Training.ipynb','training_metrics.csv','validation_metrics.csv','test_metrics.csv','classification_report.csv','confusion_matrix.png','training_loss.png','validation_loss.png','validation_macro_f1.png','per_class_f1.png','baseline_comparison.csv','sample_predictions.csv','requirements.txt','README.md']
assert all((RESULTS/f).is_file() and (RESULTS/f).stat().st_size>0 for f in required)
assert (MODEL_DIR/'model.safetensors').exists()
results_zip=shutil.make_archive(str(ROOT/'latent_commitment_results'),'zip',RESULTS)
model_zip=shutil.make_archive(str(ROOT/'latent_commitment_deberta_model'),'zip',MODEL_DIR)
for archive in [results_zip,model_zip]:
    with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('VERIFIED EXPORTS:',results_zip,model_zip)
print('ALL WORKFLOW CHECKS PASSED; training seconds:',training_seconds,'GPU:',run_info['gpu'])
files.download(results_zip)
files.download(model_zip)


In [ ]:
# 6. Final verification summary
assert run_info['reload_verified'] and run_info['inference_verified']
assert run_info['completed_epochs']==3.0 and run_info['test_evaluated']
assert all((RESULTS/f).is_file() for f in required)
for a in [results_zip,model_zip]:
    with zipfile.ZipFile(a) as z: assert z.testzip() is None
print('COMPLETE: 3 epochs, Tesla T4, %.2f seconds' % training_seconds)
print('Validation macro-F1:',run_info['best_validation_macro_f1'])
print('Held-out test (2000 examples):',test_metrics)
print('Latent Commitment:',report['Latent Commitment'])
print('Model reload and 12 illustrative predictions verified.')
print('Both ZIP archives verified. Required files:',len(required))
print('Results ZIP bytes:',Path(results_zip).stat().st_size,'Model ZIP bytes:',Path(model_zip).stat().st_size)


In [ ]:
# 7. Finalize epoch-only charts and the complete executed notebook snapshot
# A post-selection validation of the restored epoch-1 checkpoint is not an epoch-3 training measurement.
epoch_validation=pd.read_csv(RESULTS/'validation_metrics.csv').to_dict('records')
assert len(epoch_validation)==3
line_chart(epoch_validation,'eval_loss','Validation loss by training epoch','Loss','validation_loss.png')
line_chart(epoch_validation,'eval_macro_f1','Validation macro-F1 by training epoch','Macro-F1','validation_macro_f1.png')
print('Epoch-only charts verified; no additional test prediction was performed.')
notebook=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(RESULTS/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(notebook,indent=1))
results_zip=shutil.make_archive(str(ROOT/'latent_commitment_results_final'),'zip',RESULTS)
with zipfile.ZipFile(results_zip) as z: assert z.testzip() is None
print('FINAL RESULTS ARCHIVE VERIFIED:',results_zip)
files.download(results_zip)


In [ ]:
# EXPERIMENT B — template-disjoint split construction (original experiment remains above)
EXPROOT=Path('/content/latent_commitment_template_independent_v1')
EXPROOT.mkdir(exist_ok=True)
SPLITDIR=EXPROOT/'splits'; ERESULTS=EXPROOT/'results'; ERESULTS.mkdir(exist_ok=True)
"""Deterministic grouping only. Does not change example text or labels."""
import csv, io, zipfile, collections, re, unicodedata, itertools, difflib, random, hashlib, json
from pathlib import Path

def build_template_splits(zip_path, destination):
    destination=Path(destination); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        rows=[]
        for split in ['train','validation','test']:
            rows.extend(csv.DictReader(io.TextIOWrapper(z.open('latent_commitment_'+split+'.csv'),encoding='utf-8-sig')))
    original_fields=list(rows[0]); assert len(rows)==10000
    assert len({r['sample_id'] for r in rows})==10000
    groups=collections.defaultdict(list)
    for r in rows: groups[r['contrast_set_id']].append(r)
    def canon(text):
        text=unicodedata.normalize('NFKC',text).lower().replace('’',"'")
        return ' '.join(re.findall(r"[a-z]+(?:'[a-z]+)?|<\w+>",text))
    def replace_slot(text,value,slot):
        return re.sub(r'(?<!\w)'+re.escape(value)+r'(?!\w)',slot,text,flags=re.I)
    for gid,rs in groups.items():
        assert len(rs)==4 and len({r['commitment_label'] for r in rs})==4
        actions=sorted({r['committed_action'] for r in rs if r['committed_action']},key=len,reverse=True)
        deadlines=sorted({r['deadline'] for r in rs if r['deadline']},key=len,reverse=True)
        for r in rs:
            target=r['target_utterance']; context=re.sub(r'\[[^\]]+\]','<speaker>',r['conversation_context'])
            for value in actions:
                target=replace_slot(target,value,'<action>');context=replace_slot(context,value,'<action>')
            for value in deadlines:
                target=replace_slot(target,value,'<deadline>');context=replace_slot(context,value,'<deadline>')
            r['_target_template']=canon(target);r['_context_template']=canon(context)
    # Collapse cosmetic prefixes/suffixes using the shortest observed contained core.
    # Neither model predictions nor evaluation metrics influence this grouping.
    core_map={};cores=[]
    for t in sorted({r['_target_template'] for r in rows},key=lambda t:(len(t.split()),t)):
        matches=[c for c in cores if ' '+c+' ' in ' '+t+' ']
        if matches:core_map[t]=min(matches,key=lambda c:(len(c.split()),c))
        else:cores.append(t);core_map[t]=t
    for r in rows:r['_target_core']=core_map[r['_target_template']]
    parent={g:g for g in groups}
    def find(g):
        while parent[g]!=g:parent[g]=parent[parent[g]];g=parent[g]
        return g
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b:parent[max(a,b)]=min(a,b)
    # Always preserve contrast sets; also protect source IDs and raw exact duplicates.
    for field in ['_target_core','_context_template','source_sample_id','target_utterance','conversation_context']:
        owners={}
        for r in rows:
            key=r[field].strip()
            if not key:continue
            gid=r['contrast_set_id']
            if key in owners:union(gid,owners[key])
            else:owners[key]=gid
    near_edges=[]
    for field in ['_target_core','_context_template']:
        owners={r[field]:r['contrast_set_id'] for r in rows}
        for a,b in itertools.combinations(sorted(owners),2):
            similarity=difflib.SequenceMatcher(None,a.split(),b.split(),autojunk=False).ratio()
            if similarity>=0.8:
                union(owners[a],owners[b])
                near_edges.append({'field':field,'left':a,'right':b,'similarity':similarity})
    components=collections.defaultdict(list)
    for r in rows:components[find(r['contrast_set_id'])].append(r)
    keys=sorted(components);random.Random(42).shuffle(keys)
    def select_exact(candidates,n):
        # First seeded combination matching the requested size; never search on performance.
        for k in range(1,len(candidates)+1):
            for selected in itertools.combinations(candidates,k):
                if sum(len(components[g]) for g in selected)==n:return set(selected)
        raise ValueError('Cannot achieve group-preserving requested size '+str(n))
    validation_groups=select_exact(keys,1200)
    test_groups=select_exact([g for g in keys if g not in validation_groups],2000)
    assignment={g:('validation' if g in validation_groups else 'test' if g in test_groups else 'train') for g in keys}
    partitions={s:[] for s in ['train','validation','test']}
    for r in rows:
        r['_template_group']=find(r['contrast_set_id']);r['_new_split']=assignment[r['_template_group']]
        partitions[r['_new_split']].append(r)
    assert [len(partitions[s]) for s in partitions]==[6800,1200,2000]
    audits=[]
    fields=['sample_id','contrast_set_id','source_sample_id','target_utterance','conversation_context','_target_template','_target_core','_context_template','_template_group']
    for a,b in itertools.combinations(partitions,2):
        for field in fields:
            overlap={r[field] for r in partitions[a]} & {r[field] for r in partitions[b]}
            assert not overlap,(a,b,field,len(overlap))
            audits.append({'split_a':a,'split_b':b,'check':field,'overlap':0})
        for field in ['_target_core','_context_template']:
            aa={r[field] for r in partitions[a]};bb={r[field] for r in partitions[b]}
            maximum=max(difflib.SequenceMatcher(None,x.split(),y.split(),autojunk=False).ratio() for x in aa for y in bb)
            assert maximum<0.8,(field,maximum)
            audits.append({'split_a':a,'split_b':b,'check':field+'_max_token_similarity','maximum':maximum})
    for split,rs in partitions.items():
        counts=collections.Counter(r['commitment_label'] for r in rs)
        assert len(counts)==4 and len(set(counts.values()))==1
        with (destination/('template_independent_'+split+'.csv')).open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=original_fields);w.writeheader()
            for r in sorted(rs,key=lambda r:r['sample_id']):
                original={k:r[k] for k in original_fields};original['split']=split;w.writerow(original)
    manifest_fields=['sample_id','contrast_set_id','source_sample_id','commitment_label','_target_template','_target_core','_context_template','_template_group','_new_split']
    with (destination/'template_split_manifest.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=manifest_fields);w.writeheader()
        for r in sorted(rows,key=lambda r:r['sample_id']):w.writerow({k:r[k] for k in manifest_fields})
    report={'definition':'Group contrast sets, exact raw duplicates, delexicalized target cores and context frames, and any token SequenceMatcher similarity >= 0.8. Contained cores collapse cosmetic wrappers. All grouping decisions fixed before model training. This is operational template separation, not guaranteed semantic independence.','seed':42,'threshold':0.8,'counts':{s:len(rs) for s,rs in partitions.items()},'class_counts':{s:dict(collections.Counter(r['commitment_label'] for r in rs)) for s,rs in partitions.items()},'component_count':len(components),'component_sizes':{g:len(rs) for g,rs in sorted(components.items())},'target_core_count':len(cores),'context_template_count':len({r['_context_template'] for r in rows}),'audit':audits,'csv_sha256':{s:hashlib.sha256((destination/('template_independent_'+s+'.csv')).read_bytes()).hexdigest() for s in partitions},'rows_dropped':0,'input_or_label_changes':0}
    (destination/'template_split_audit.json').write_text(json.dumps(report,indent=2))
    (destination/'near_template_edges.json').write_text(json.dumps(near_edges,indent=2))
    return report


split_audit=build_template_splits('/content/latent_commitment_leakage_free_splits.zip',SPLITDIR)
print('TEMPLATE SPLITS VERIFIED:',json.dumps({k:split_audit[k] for k in ['counts','class_counts','component_count','rows_dropped','input_or_label_changes']},indent=2))
print('Maximum cross-split template similarities:',[(r['split_a'],r['split_b'],r['check'],r['maximum']) for r in split_audit['audit'] if 'maximum' in r])
experiment_data={s:pd.read_csv(SPLITDIR/('template_independent_'+s+'.csv')) for s in ['train','validation','test']}
for df in experiment_data.values():df['labels']=df.commitment_label.map(label2id).astype(int)
experiment_baselines={};experiment_baseline_rows=[]
for column in ['target_utterance','model_input_text']:
    baseline=make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=100000,sublinear_tf=True),LogisticRegression(C=1.0,max_iter=2000,random_state=42))
    baseline.fit(experiment_data['train'][column],experiment_data['train'].labels)
    experiment_baselines[column]=baseline
    experiment_baseline_rows.append({'model':'TF-IDF + Logistic Regression','input':column,'split':'validation',**metrics(experiment_data['validation'].labels,baseline.predict(experiment_data['validation'][column]))})
pd.DataFrame(experiment_baseline_rows).to_csv(ERESULTS/'baseline_comparison.csv',index=False)
print('NEW VALIDATION BASELINES:',json.dumps(experiment_baseline_rows,indent=2))
def experiment_dataset(split):
    df=experiment_data[split]
    ds=Dataset.from_dict({'text':df.model_input_text.tolist(),'labels':df.labels.tolist()})
    return ds.map(lambda b:tokenizer(b['text'],truncation=True,max_length=256),batched=True,remove_columns=['text'])
experiment_train_ds=experiment_dataset('train');experiment_val_ds=experiment_dataset('validation')
print('EXPERIMENT B READY FOR FRESH PRETRAINED MODEL TRAINING')


In [ ]:
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()
# EXPERIMENT B — fresh pretrained weights; same hyperparameters, never reuse Experiment A weights
import gc, time, shutil
assert torch.cuda.is_available()
for name in ['trainer','model','reloaded_model']:
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache();set_seed(42)
class ExperimentProgress(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        (ERESULTS/'trainer_log.json').write_text(json.dumps(state.log_history,indent=2))
        if logs:print('EXPERIMENT B',state.global_step,json.dumps(logs),flush=True)
    def on_save(self,args,state,control,**kwargs):
        print('EXPERIMENT B CHECKPOINT:',state.global_step,'best validation macro-F1:',state.best_metric,flush=True)
experiment_model=AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base',num_labels=4,id2label=id2label,label2id=label2id)
experiment_args=TrainingArguments(output_dir=str(EXPROOT/'checkpoints'),num_train_epochs=3,learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=0.01,warmup_ratio=0.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=2,logging_steps=50,seed=42,data_seed=42,report_to='none',dataloader_num_workers=2,save_safetensors=True,disable_tqdm=True)
experiment_trainer=Trainer(model=experiment_model,args=experiment_args,train_dataset=experiment_train_ds,eval_dataset=experiment_val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8),compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2),ExperimentProgress()])
experiment_start=time.time();experiment_train_output=experiment_trainer.train();experiment_seconds=time.time()-experiment_start
EMODEL=EXPROOT/'best_model';experiment_trainer.save_model(str(EMODEL));tokenizer.save_pretrained(str(EMODEL))
experiment_epoch_logs=list(experiment_trainer.state.log_history)
pd.DataFrame([r for r in experiment_epoch_logs if 'loss' in r]).to_csv(ERESULTS/'training_metrics.csv',index=False)
pd.DataFrame([r for r in experiment_epoch_logs if 'eval_loss' in r]).to_csv(ERESULTS/'validation_metrics.csv',index=False)
experiment_info={'model_initialization':'fresh microsoft/deberta-v3-base; no Experiment A fine-tuned weights','gpu':torch.cuda.get_device_name(0),'training_seconds':experiment_seconds,'epochs_completed':experiment_trainer.state.epoch,'best_checkpoint':experiment_trainer.state.best_model_checkpoint,'best_validation_macro_f1':experiment_trainer.state.best_metric,'global_step':experiment_trainer.state.global_step,'seed':42,'split_csv_sha256':split_audit['csv_sha256'],'same_hyperparameters_as_original':True}
assert np.isfinite(experiment_info['best_validation_macro_f1'])
(ERESULTS/'run_info.json').write_text(json.dumps(experiment_info,indent=2))
print('EXPERIMENT B TRAINING COMPLETED',json.dumps(experiment_info,indent=2))


In [ ]:
# EXPERIMENT B — evaluate new held-out test once after selecting on new validation only
import matplotlib.pyplot as plt, seaborn as sns
assert (EMODEL/'config.json').exists()
experiment_test_cache=ERESULTS/'heldout_predictions.npz'
assert not experiment_test_cache.exists(), 'Experiment B test already evaluated: reuse its saved results, do not evaluate again.'
experiment_test=experiment_trainer.predict(experiment_dataset('test'),metric_key_prefix='test')
np.savez_compressed(experiment_test_cache,logits=experiment_test.predictions,labels=experiment_test.label_ids)
experiment_y=experiment_test.label_ids;experiment_p=experiment_test.predictions.argmax(axis=-1)
experiment_test_metrics=metrics(experiment_y,experiment_p)
pd.DataFrame([experiment_test_metrics]).to_csv(ERESULTS/'test_metrics.csv',index=False)
experiment_report=classification_report(experiment_y,experiment_p,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
pd.DataFrame(experiment_report).T.to_csv(ERESULTS/'classification_report.csv')
experiment_cm=confusion_matrix(experiment_y,experiment_p,labels=range(4))
pd.DataFrame(experiment_cm,index=NAMES,columns=NAMES).to_csv(ERESULTS/'confusion_matrix.csv')
pd.DataFrame({'sample_id':experiment_data['test'].sample_id,'true_label':experiment_y,'predicted_label':experiment_p}).to_csv(ERESULTS/'test_predictions.csv',index=False)
for column,baseline in experiment_baselines.items():
    experiment_baseline_rows.append({'model':'TF-IDF + Logistic Regression','input':column,'split':'test',**metrics(experiment_data['test'].labels,baseline.predict(experiment_data['test'][column]))})
bestval=next(r for r in experiment_epoch_logs if r.get('eval_macro_f1')==experiment_info['best_validation_macro_f1'])
experiment_baseline_rows.append({'model':'DeBERTa-v3-base','input':'model_input_text','split':'validation',**{k:bestval['eval_'+k] for k in experiment_test_metrics}})
experiment_baseline_rows.append({'model':'DeBERTa-v3-base','input':'model_input_text','split':'test',**experiment_test_metrics})
pd.DataFrame(experiment_baseline_rows).to_csv(ERESULTS/'baseline_comparison.csv',index=False)
experiment_comparison=[{'experiment':'Original supplied splits','train_n':7000,'validation_n':1000,'test_n':2000,'test_accuracy':1.0,'test_macro_f1':1.0},{'experiment':'Template-independent v1','train_n':6800,'validation_n':1200,'test_n':2000,'test_accuracy':experiment_test_metrics['accuracy'],'test_macro_f1':experiment_test_metrics['macro_f1']}]
pd.DataFrame(experiment_comparison).to_csv(ERESULTS/'experiment_comparison.csv',index=False)
for key,title,ylabel,name in [('loss','Template-independent training loss','Loss','training_loss.png'),('eval_loss','Template-independent validation loss','Loss','validation_loss.png'),('eval_macro_f1','Template-independent validation macro-F1','Macro-F1','validation_macro_f1.png')]:
    selected=[r for r in experiment_epoch_logs if key in r]
    fig,ax=plt.subplots(figsize=(8,5));ax.plot([r['epoch'] for r in selected],[r[key] for r in selected],marker='o');ax.set(xlabel='Epoch',ylabel=ylabel,title=title);fig.tight_layout();fig.savefig(ERESULTS/name,dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,7));sns.heatmap(experiment_cm,annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES,ax=ax);ax.set(xlabel='Predicted class',ylabel='True class',title='Template-independent held-out test');plt.xticks(rotation=25,ha='right');plt.yticks(rotation=0);fig.tight_layout();fig.savefig(ERESULTS/'confusion_matrix.png',dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,5));ax.bar(NAMES,[experiment_report[n]['f1-score'] for n in NAMES]);ax.set(ylim=(0,1.05),ylabel='F1',title='Template-independent test per-class F1');plt.xticks(rotation=20,ha='right');fig.tight_layout();fig.savefig(ERESULTS/'per_class_f1.png',dpi=160);plt.close(fig)
# Reload the fresh experiment's saved model; verify predictions on validation, not test.
experiment_trainer.model.to('cpu');gc.collect();torch.cuda.empty_cache()
experiment_reloaded=AutoModelForSequenceClassification.from_pretrained(EMODEL).to('cuda').eval()
experiment_tok=AutoTokenizer.from_pretrained(EMODEL,use_fast=False)
probe=experiment_tok(experiment_data['validation'].model_input_text.iloc[0],return_tensors='pt',truncation=True,max_length=256)
with torch.no_grad():
    a=experiment_trainer.model(**probe).logits.cpu();b=experiment_reloaded(**{k:v.to('cuda') for k,v in probe.items()}).logits.cpu()
assert torch.allclose(a,b,atol=2e-4,rtol=2e-4)
experiment_info.update({'test_metrics':experiment_test_metrics,'reload_verified':True,'test_evaluated_once':True})
(ERESULTS/'run_info.json').write_text(json.dumps(experiment_info,indent=2))
print('TEMPLATE-INDEPENDENT TEST RESULTS:',json.dumps(experiment_test_metrics,indent=2))
print('LATENT COMMITMENT:',json.dumps(experiment_report['Latent Commitment'],indent=2))
print('CONFUSION MATRIX:',experiment_cm.tolist())
print('COMPARISON:',json.dumps(experiment_comparison,indent=2))
print('BASELINES:',json.dumps(experiment_baseline_rows,indent=2))


In [ ]:
# EXPERIMENT B — preserve audit, new splits, metrics, charts, and updated notebook
from google.colab import _message, files
import importlib.metadata as metadata
(ERESULTS/'requirements.txt').write_text(''.join(p+'=='+metadata.version(p)+'\n' for p in ['torch','transformers','datasets','accelerate','scikit-learn','pandas','numpy','matplotlib','seaborn','sentencepiece','protobuf','safetensors']))
(ERESULTS/'README.md').write_text(f'''# Template-independent experiment v1
The original experiment is preserved in the preceding notebook cells and its original output directory.
This experiment reconstructs 10000 original rows into 6800 train, 1200 validation, and 2000 test rows.
No text or labels were regenerated or changed; every contrast set remains intact; all rows retained.
Each split is class-balanced. Grouping and allocation were fixed with seed 42 before training.
Definition: remove annotated action/deadline slots, normalize punctuation/case/speaker markers, collapse cosmetic target wrappers via shortest observed contained cores, join exact raw targets/contexts/source IDs, and merge target/context templates with token SequenceMatcher ratio >= 0.8.
A connected group is allocated wholly to one split. See splits/template_split_audit.json and splits/template_split_manifest.csv.
Zero cross-split target cores, context templates, contrast sets, source IDs, exact targets, and exact contexts. No cross-split target-core or context-template similarity reaches 0.8 under the stated rule.
This is operational template separation, not semantic independence. Synonyms and class-defining words can still transfer across different templates. Delexicalization uses annotation metadata only for split grouping; model inputs contain only the original model_input_text.
The experiment starts from fresh microsoft/deberta-v3-base, never the previous fine-tuned model. Same seed, learning rate, batch sizes, gradient accumulation, maximum length, optimizer, warmup, weight decay, FP16, epochs, early stopping and validation selection as the original run.
Training duration: {experiment_seconds:.2f} seconds on {experiment_info['gpu']}.
Selected checkpoint: {experiment_info['best_checkpoint']}; validation macro-F1: {experiment_info['best_validation_macro_f1']}.
New test metrics: {json.dumps(experiment_test_metrics)}.
Latent Commitment: {json.dumps(experiment_report['Latent Commitment'])}.
The test was predicted once after selection; cached logits and sample-level predictions are saved. Both baselines were fitted on new train and evaluated on new validation and test without tuning.
Original test accuracy was 1.0. New test accuracy is {experiment_test_metrics['accuracy']}; delta is {experiment_test_metrics['accuracy']-1.0:.6f}.
A new split changes test composition and training size slightly; this is a template-held-out experiment, not a controlled estimate of the isolated effect of one change. The test is newly held out for the fresh model, but is drawn from the previously inspected dataset.
An independent naturally occurring conversation test set is still needed to establish real-world generalization.
Saved model reload was checked against validation logits. Best weights/tokenizer are in ../best_model in Colab.
''')
print('EXPERIMENT B FINAL SUMMARY:',json.dumps(experiment_info,indent=2))
print('New accuracy equals 1:',experiment_test_metrics['accuracy']==1.0)
# Save the actual notebook with original experiment and this new experiment.
experiment_notebook=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(ERESULTS/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(experiment_notebook,indent=1))
experiment_bundle=EXPROOT/'export';experiment_bundle.mkdir(exist_ok=True)
shutil.copytree(ERESULTS,experiment_bundle/'results',dirs_exist_ok=True)
shutil.copytree(SPLITDIR,experiment_bundle/'splits',dirs_exist_ok=True)
experiment_zip=shutil.make_archive(str(EXPROOT/'latent_commitment_template_independent_results'),'zip',experiment_bundle)
with zipfile.ZipFile(experiment_zip) as z:assert z.testzip() is None
print('NEW RESULTS/SPLITS ZIP VERIFIED:',experiment_zip)
print('NEW BEST MODEL:',EMODEL)
files.download(experiment_zip)


In [ ]:
# Final comparison — reuse saved predictions, no further model evaluation
assert experiment_info['reload_verified'] and experiment_info['test_evaluated_once']
correct=int((experiment_p==experiment_y).sum())
print('TEMPLATE-INDEPENDENT EXPERIMENT COMPLETE')
print('Fresh DeBERTa-v3-base | 3 epochs | Tesla T4 | %.1f seconds' % experiment_seconds)
print('Splits: 6800 train / 1200 validation / 2000 test; no rows dropped.')
print('No cross-split target/context templates or near duplicates at the defined 0.80 threshold.')
print('Selected on validation macro-F1: %.6f (epoch 1)' % experiment_info['best_validation_macro_f1'])
print('Original test accuracy: 100.00%')
print('New test accuracy: %.2f%% (%d / %d correct)' % (100*experiment_test_metrics['accuracy'],correct,len(experiment_y)))
print('New test macro-F1: %.6f' % experiment_test_metrics['macro_f1'])
print('Latent Commitment F1: %.6f' % experiment_report['Latent Commitment']['f1-score'])
print('Saved model reload and export ZIP verified. Synthetic-data limitations still apply.')


In [ ]:
# EXPERIMENT C: fixed-model context intervention, five predeclared seeds.
from pathlib import Path
import re, gc, time, json
AROOT=Path('/content/latent_commitment_context_ablation_v1'); ARES=AROOT/'results'; ARES.mkdir(parents=True,exist_ok=True)
a_test=experiment_data['test'].reset_index(drop=True).copy()
a_y=a_test['commitment_label'].map(label2id).to_numpy()
a_cache=np.load(ERESULTS/'heldout_predictions.npz')
print('Cached prediction keys:',a_cache.files,flush=True)
a_logits=a_cache['logits']; assert np.array_equal(a_cache['labels'],a_y)
a_ref=a_logits.argmax(1)
def a_probs(x):
    z=x-x.max(axis=1,keepdims=True); z=np.exp(z); return z/z.sum(axis=1,keepdims=True)
a_refprob=a_probs(a_logits)
a_pat=re.compile(r'^(\[CONTEXT\] )(.*?)( \[TARGET SPEAKER\].*)$',re.S)
a_parts=[a_pat.match(x) for x in a_test.model_input_text]
assert all(a_parts)
assert all(m.group(2)==c for m,c in zip(a_parts,a_test.conversation_context))
@torch.inference_mode()
def a_infer(texts,model,tok):
    batches=[]
    for start in range(0,len(texts),16):
        batch=tok(texts[start:start+16],padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):out=model(**batch).logits
        batches.append(out.float().cpu().numpy())
    return np.concatenate(batches)
experiment_reloaded.to('cuda').eval()
a_shuffle_rows=[]; a_mappings=[]; a_shuffle_logits={}
for seed in [42,43,44,45,46]:
    rng=np.random.default_rng(seed)
    # Full permutation; reject any same context or contrast-set donor.
    for attempt in range(10000):
        donor=rng.permutation(len(a_test))
        if ((a_test.conversation_context.to_numpy()!=a_test.conversation_context.to_numpy()[donor]) & (a_test.contrast_set_id.to_numpy()!=a_test.contrast_set_id.to_numpy()[donor])).all():break
    else:raise RuntimeError('No valid context derangement')
    texts=[m.group(1)+a_test.conversation_context.iloc[j]+m.group(3) for m,j in zip(a_parts,donor)]
    logits=a_infer(texts,experiment_reloaded,experiment_tok); pred=logits.argmax(1); probs=a_probs(logits)
    a_shuffle_logits[str(seed)]=logits
    row={'seed':seed,**metrics(a_y,pred),'agreement_with_original':float((pred==a_ref).mean()),'prediction_flip_count':int((pred!=a_ref).sum()),'accuracy_delta_pp':float(((pred==a_y).mean()-(a_ref==a_y).mean())*100),'mean_abs_probability_change':float(np.abs(probs-a_refprob).mean()),'original_correct_now_wrong':int(((a_ref==a_y)&(pred!=a_y)).sum()),'original_wrong_now_correct':int(((a_ref!=a_y)&(pred==a_y)).sum())}
    a_shuffle_rows.append(row)
    frame=pd.DataFrame({'seed':seed,'sample_id':a_test.sample_id,'donor_sample_id':a_test.sample_id.to_numpy()[donor],'donor_contrast_set_id':a_test.contrast_set_id.to_numpy()[donor],'true_label':a_y,'original_prediction':a_ref,'shuffled_prediction':pred,'original_true_probability':a_refprob[np.arange(len(a_y)),a_y],'shuffled_true_probability':probs[np.arange(len(a_y)),a_y]})
    a_mappings.append(frame)
    print('SHUFFLE',json.dumps(row),flush=True)
pd.DataFrame(a_shuffle_rows).to_csv(ARES/'context_shuffle_metrics.csv',index=False)
pd.concat(a_mappings).to_csv(ARES/'context_shuffle_predictions_and_donors.csv',index=False)
np.savez_compressed(ARES/'context_shuffle_logits.npz',labels=a_y,original_logits=a_logits,**a_shuffle_logits)
print('CONTEXT SHUFFLING COMPLETE. Original labels retained: perturbation sensitivity, not newly annotated semantic accuracy.',flush=True)


In [ ]:
# EXPERIMENT C: fresh target-only training on identical fixed splits.
def target_dataset(split):
    frame=experiment_data[split]
    enc=tokenizer(frame.target_utterance.tolist(),truncation=True,max_length=256)
    enc['labels']=frame.commitment_label.map(label2id).tolist()
    return Dataset.from_dict(enc)
target_train_ds=target_dataset('train'); target_val_ds=target_dataset('validation')
# TARGET ONLY — fresh pretrained weights; same hyperparameters, never reuse contextual weights
import gc, time, shutil
assert torch.cuda.is_available()
for name in ['experiment_trainer','experiment_model','experiment_reloaded']:
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache();set_seed(42)
class ExperimentProgress(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        (ARES/'trainer_log.json').write_text(json.dumps(state.log_history,indent=2))
        if logs:print('TARGET ONLY',state.global_step,json.dumps(logs),flush=True)
    def on_save(self,args,state,control,**kwargs):
        print('TARGET ONLY CHECKPOINT:',state.global_step,'best validation macro-F1:',state.best_metric,flush=True)
target_model=AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base',num_labels=4,id2label=id2label,label2id=label2id)
target_args=TrainingArguments(output_dir=str(AROOT/'checkpoints'),num_train_epochs=3,learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=0.01,warmup_ratio=0.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=2,logging_steps=50,seed=42,data_seed=42,report_to='none',dataloader_num_workers=2,save_safetensors=True,disable_tqdm=True)
target_trainer=Trainer(model=target_model,args=target_args,train_dataset=target_train_ds,eval_dataset=target_val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8),compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2),ExperimentProgress()])
target_start=time.time();target_train_output=target_trainer.train();target_seconds=time.time()-target_start
TMODEL=AROOT/'best_model';target_trainer.save_model(str(TMODEL));tokenizer.save_pretrained(str(TMODEL))
target_epoch_logs=list(target_trainer.state.log_history)
pd.DataFrame([r for r in target_epoch_logs if 'loss' in r]).to_csv(ARES/'training_metrics.csv',index=False)
pd.DataFrame([r for r in target_epoch_logs if 'eval_loss' in r]).to_csv(ARES/'validation_metrics.csv',index=False)
target_info={'model_initialization':'fresh microsoft/deberta-v3-base; no contextual fine-tuned weights','gpu':torch.cuda.get_device_name(0),'training_seconds':target_seconds,'epochs_completed':target_trainer.state.epoch,'best_checkpoint':target_trainer.state.best_model_checkpoint,'best_validation_macro_f1':target_trainer.state.best_metric,'global_step':target_trainer.state.global_step,'seed':42,'split_csv_sha256':split_audit['csv_sha256'],'same_hyperparameters_as_contextual':True,'input':'target_utterance only'}
assert np.isfinite(target_info['best_validation_macro_f1'])
(ARES/'run_info.json').write_text(json.dumps(target_info,indent=2))
print('TARGET ONLY TRAINING COMPLETED',json.dumps(target_info,indent=2))


In [ ]:
# Evaluate selected target-only checkpoint once; compare against cached contextual test.
target_val_output=target_trainer.predict(target_val_ds)
target_val_metrics=metrics(target_val_output.label_ids,target_val_output.predictions.argmax(1))
target_test_output=target_trainer.predict(target_dataset('test'))
target_logits=target_test_output.predictions; target_pred=target_logits.argmax(1)
assert np.array_equal(target_test_output.label_ids,a_y)
target_metrics=metrics(a_y,target_pred)
np.savez_compressed(ARES/'target_only_logits.npz',logits=target_logits,labels=a_y)
pd.DataFrame([target_metrics]).to_csv(ARES/'target_only_test_metrics.csv',index=False)
target_report=classification_report(a_y,target_pred,labels=list(range(4)),target_names=NAMES,output_dict=True,zero_division=0)
pd.DataFrame(target_report).T.to_csv(ARES/'target_only_classification_report.csv')
pd.DataFrame(confusion_matrix(a_y,target_pred),index=NAMES,columns=NAMES).to_csv(ARES/'target_only_confusion_matrix.csv')
target_predictions=a_test[['sample_id','contrast_set_id','target_utterance','commitment_label']].copy()
target_predictions['target_only_prediction']=[id2label[int(x)] for x in target_pred]
target_predictions['contextual_prediction']=[id2label[int(x)] for x in a_ref]
target_predictions['target_only_correct']=target_pred==a_y
target_predictions.to_csv(ARES/'target_only_test_predictions.csv',index=False)
# Verify serialization on validation, without another test evaluation.
probe_text=experiment_data['validation'].target_utterance.iloc[:16].tolist()
probe_before=a_infer(probe_text,target_model.eval(),tokenizer)
target_trainer.model.to('cpu');gc.collect();torch.cuda.empty_cache()
target_reloaded=AutoModelForSequenceClassification.from_pretrained(TMODEL).to('cuda').eval()
target_reload_tok=AutoTokenizer.from_pretrained(TMODEL,use_fast=False)
probe_after=a_infer(probe_text,target_reloaded,target_reload_tok)
assert np.allclose(probe_before,probe_after,atol=0.002,rtol=0.002)
target_info.update({'test_metrics':target_metrics,'validation_metrics':target_val_metrics,'reload_verified':True,'test_evaluated_once':True,'shuffle_seeds':[42,43,44,45,46]})
(ARES/'run_info.json').write_text(json.dumps(target_info,indent=2))
a_comparison=pd.DataFrame([{'condition':'Contextual model, original context',**metrics(a_y,a_ref)},{'condition':'Fresh target-only model',**target_metrics}]+[{'condition':'Contextual model, shuffled context seed '+str(r['seed']),**{k:r[k] for k in target_metrics}} for r in a_shuffle_rows])
a_comparison.to_csv(ARES/'ablation_comparison.csv',index=False)
a_shuffle_summary=pd.DataFrame(a_shuffle_rows)
print('TARGET-ONLY TEST:',json.dumps(target_metrics),flush=True)
print('TARGET-ONLY VALIDATION:',json.dumps(target_val_metrics),flush=True)
print('SHUFFLE MEANS:',a_shuffle_summary.mean(numeric_only=True).to_json(),flush=True)
print('TARGET-ONLY RELOAD VERIFIED; saved model:',str(TMODEL),flush=True)
print(a_comparison.to_string(index=False),flush=True)


In [ ]:
# Save reproducibility records and actual updated notebook.
from google.colab import _message
import zipfile, shutil
(ARES/'README.md').write_text('''# Context reliance diagnostic
Same template-held-out partitions: 6800 training, 1200 validation, 2000 test examples.
Target-only model is fresh microsoft/deberta-v3-base, trained on target_utterance only with the contextual run's hyperparameters (seed42,3 epochs, validation macro-F1 checkpoint selection). No test-driven tuning.
Context shuffle freezes the previously selected contextual checkpoint. Five predetermined seeds42-46. Each context donor is a permutation of test rows, and must have different raw context and contrast-set ID. Target utterances, target speakers, and original labels stay unchanged. Only the serialized context span is replaced. Donor mappings, per-row predictions, logits, flip counts, probability changes and metrics are exported.
Original contextual predictions are cached from the earlier test, not newly selected. Target-only checkpoint is evaluated once on test after validation selection; model reload verified on validation examples.
Shuffled accuracy is measured against ORIGINAL labels. Mismatched contexts can change meaning or be incoherent; this measures perturbation sensitivity, not correctly annotated performance on changed conversations. Prediction stability does not prove zero context use. A strong target-only result demonstrates that wording suffices for much of THIS dataset, not necessarily the same internal strategy in the contextual model.
Only two connected template groups occur in this test. These are exploratory diagnostics on an already examined synthetic dataset. Independent, natural examples and same-utterance/different-context contrasts with reannotated labels are needed for a context-understanding claim.
Best target-only model and tokenizer are in ../best_model in Colab. Contextual best model remains in /content/latent_commitment_template_independent_v1/best_model.
''')
shutil.copy(SPLITDIR/'template_split_audit.json',ARES/'template_split_audit.json')
shutil.copy(SPLITDIR/'template_split_manifest.csv',ARES/'template_split_manifest.csv')
shutil.copy(ERESULTS/'requirements.txt',ARES/'requirements.txt')
a_notebook=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(ARES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(a_notebook,indent=1))
a_zip=shutil.make_archive(str(AROOT/'latent_commitment_context_ablation_results'),'zip',ARES)
with zipfile.ZipFile(a_zip) as z:assert z.testzip() is None
print('CONTEXT ABLATIONS COMPLETE — saved and ZIP verified')
print('Original context: %.2f%% accuracy' % (100*(a_ref==a_y).mean()))
print('Target-only: %.2f%% accuracy; macro-F1 %.6f' % (100*target_metrics['accuracy'],target_metrics['macro_f1']))
print('Shuffled context: %.2f%% mean accuracy (%.2f–%.2f%%)' % (100*a_shuffle_summary.accuracy.mean(),100*a_shuffle_summary.accuracy.min(),100*a_shuffle_summary.accuracy.max()))
print('Shuffled prediction agreement: %.2f%% mean' % (100*a_shuffle_summary.agreement_with_original.mean()))
print('Export:',a_zip)
print('Target-only model:',TMODEL)


In [ ]:
# Class-level context sensitivity from cached logits; no additional model runs.
a_classrows=[]
for seed,logits in a_shuffle_logits.items():
    pred=logits.argmax(1)
    for k,name in enumerate(NAMES):
        mask=a_y==k
        a_classrows.append({'seed':int(seed),'class':name,'support':int(mask.sum()),'original_recall':float((a_ref[mask]==k).mean()),'shuffled_recall':float((pred[mask]==k).mean()),'prediction_flip_rate':float((pred[mask]!=a_ref[mask]).mean())})
pd.DataFrame(a_classrows).to_csv(ARES/'context_shuffle_per_class.csv',index=False)
print('CONTEXT RELIANCE DIAGNOSTIC COMPLETE')
print('Same fixed template-held-out test: 2000 examples; fresh target-only training on T4.')
print('Contextual / original context: %.2f%% accuracy | macro-F1 %.6f' % (100*(a_ref==a_y).mean(),metrics(a_y,a_ref)['macro_f1']))
print('Target-only / no context: %.2f%% accuracy | macro-F1 %.6f' % (100*target_metrics['accuracy'],target_metrics['macro_f1']))
print('Contextual / shuffled context: %.2f%% mean accuracy | range %.2f–%.2f%%' % (100*a_shuffle_summary.accuracy.mean(),100*a_shuffle_summary.accuracy.min(),100*a_shuffle_summary.accuracy.max()))
print('Prediction agreement after shuffling: %.2f%%; mean accuracy drop: %.2f percentage points' % (100*a_shuffle_summary.agreement_with_original.mean(),-a_shuffle_summary.accuracy_delta_pp.mean()))
print('Mean recall by class after shuffling:')
print(pd.DataFrame(a_classrows).groupby('class')[['original_recall','shuffled_recall','prediction_flip_rate']].mean().to_string())
print('Interpretation: wording alone is highly predictive here; context affects a minority of predictions.')
print('Shuffled inputs retain original labels; this is sensitivity, not reannotated semantic evaluation.')
print('Saved model reload verified. Results, logits, donor mappings, training records exported.')
# Refresh archive to include class analysis and notebook cells.
a_notebook=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(ARES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(a_notebook,indent=1))
a_zip=shutil.make_archive(str(AROOT/'latent_commitment_context_ablation_results'),'zip',ARES)
with zipfile.ZipFile(a_zip) as z:assert z.testzip() is None
print('Final ZIP verified:',a_zip)


In [ ]:
"""Paired context challenge v1; authored and frozen before model inference."""
import json,csv,pathlib,hashlib,collections
pairs=[]
def add(kind,target,c1,l1,c2,l2,r1,r2):
    pairs.append({'pair_id':f'CD{len(pairs)+1:03}','contrast_type':kind,'target':target,'variants':[(c1,l1,r1),(c2,l2,r2)]})
# Factual availability vs implicit uptake of an unresolved request.
nl=[
('My afternoon is clear.','Is your afternoon free, or are you in meetings?','The supplier comparison has no owner. We need someone to finish it this afternoon. Who has room?'),
('I have the checklist open.','Which document is currently on your screen?','We need someone to check the release checklist now. Who is getting started?'),
('The invoices are already on my screen.','What are you looking at while you wait for the call?','Nobody has started checking the disputed invoices. Who can get us moving?'),
('I kept a slot free after lunch.','For the calendar survey, did you leave any personal focus time today?','The new starter still needs a walkthrough after lunch. Is anyone making room for that?'),
('I have the necessary access now.','Has IT finished enabling your account?','The stalled deployment needs investigation, and nobody has picked it up. Who is in a position to start?'),
('The draft is beside me, and I have an hour before my next call.','For the desk inventory, what is on your desk, and when is your next meeting?','The draft needs another reader before the client call. Nobody is reviewing it yet. Who has room?'),
('I already pulled up the error logs.','Which files did you open for the training exercise?','The failed sync is still unassigned. We need someone to investigate the logs now.'),
('I have room in my schedule tomorrow morning.','Which part of tomorrow is free on your personal calendar?','The archive cleanup still has no owner. Who can make a start tomorrow morning?'),
('The previous handover notes are in front of me.','Which reference document did you find in the shared folder?','We still need someone to prepare the next handover. Who is taking a first look?'),
('I set aside the next half hour.','For the wellbeing survey, how much break time did you reserve?','The escalation notes need checking, and no one is on them. Who has time right now?')]
for t,n,l in nl:add('no_vs_latent',t,'[manager]: '+n,0,'[manager]: '+l,1,'Answers a factual question; no unresolved task is accepted.','Responds to an unresolved task with relevant availability or preparation, pragmatically signaling uptake without a direct undertaking.')
# Affirmation of a fact vs direct acceptance of a future action.
ne=[
('Yes.','Did the courier arrive this morning?','Will you send the revised schedule to the team by four?'),
('Yes, that is correct.','Is the office on the third floor?','Just to confirm your offer: you will host the onboarding session tomorrow?'),
("That's right.",'Was the old dashboard retired last week?','To confirm what you offered: you will check the reconciliation file before noon?'),
('Exactly.','Is the contract reference the one printed on the first page?','So the plan you are agreeing to is that you will distribute the agenda before the meeting?'),
('Confirmed.','Has the room booking appeared in the calendar already?','Please confirm that you will personally arrange the equipment pickup on Friday.'),
('Absolutely.','Are you certain that yesterday was a public holiday?','Will you take responsibility for sending the customer update today?'),
('You have that right.','Was the archive originally stored on the older server?','To confirm your proposal: you will review the access list this evening?'),
('Correct.','Is the meeting room number 204?','Your undertaking is to complete the incident summary tomorrow, correct?'),
('Yes, indeed.','Did the visitor sign in at reception?','Will you prepare the stock report for the morning briefing?'),
('That is what we agreed.','Was the room color we chose last month blue?','To confirm our arrangement: you will book the interpreter for Monday?')]
for t,n,e in ne:add('no_vs_explicit',t,'[manager]: '+n,0,'[manager]: '+e,3,'Affirms a past or present fact; no future undertaking.','Direct affirmative answer confirms a proposition in which the speaker undertakes a specific future task; elliptical acceptance counts as explicit.')
# Direct acceptance; condition resides in context and remains unresolved.
eq=[
('Yes, I will.','send the launch note this afternoon','legal clears it first'),
("I'll do that.",'organize the workshop next week','the venue approves the booking'),
('Agreed, I will take care of it.','check the supplier pack tomorrow','procurement releases the files'),
('You can count on me for that.','cover the morning shift','the shift swap is approved'),
('I accept that responsibility.','finish the compliance summary','the auditor sends the missing evidence'),
('Yes, I agree to that arrangement.','run the migration on Saturday','the backup passes verification'),
("That's agreed; I'll handle it.",'send the board slides tonight','finance signs off the numbers'),
('I will take that on.','host the customer demo on Thursday','the test environment is restored'),
('Consider that my undertaking.','prepare the renewal quote','the pricing exception is authorized'),
('I agree to do it.','publish the roster tomorrow','HR approves the staffing plan')]
for t,a,c in eq:add('explicit_vs_qualified',t,f'[manager]: Will you {a}? There is no pending prerequisite for this task.',3,f'[manager]: The proposed arrangement is that you {a} only if {c}. That has not happened yet; otherwise the task is off. Do you accept this conditional arrangement?',2,'Directly accepts a specific future action with no unresolved contingency.','Accepts the conditional arrangement, inheriting its explicit unresolved prerequisite rather than making an unconditional undertaking.')
# Indirect preparation signaling uptake, with and without an unresolved condition.
lq=[
('I have a quiet hour and the review notes ready.','review the audit notes this afternoon','the confidentiality approval arrives'),
('The test plan is open, and I have time reserved.','run the acceptance tests tomorrow','the staging build becomes available'),
('I already have the comparison sheet in front of me.','compare the vendor bids','procurement permits the review'),
('I cleared tomorrow morning and found the old guide.','update the induction guide','the policy team approves the rewrite'),
('The relevant records are open on my screen.','check the retention records','the data owner grants clearance'),
('I have the slides ready to work through and a free evening.','review the presentation tonight','the client authorizes an external review'),
('I put the forms on my desk and left the afternoon open.','process the travel forms','the budget release is approved'),
('The runbook is beside me, and I have a gap after the standup.','check the recovery runbook','the infrastructure team gives permission'),
('I have the shipment list open and some time set aside.','reconcile the shipment list','the warehouse releases the final counts'),
('I have the draft policy in front of me and space in my calendar.','review the remote working policy','the committee authorizes consultation')]
for t,a,c in lq:add('latent_vs_qualified',t,f'[manager]: We still need someone to {a}. The work is ready to begin; who can make a start?',1,f'[manager]: We still need someone to {a}, but only if {c}. That decision is pending; without it there will be no task. Who is preparing to take this on under that arrangement?',2,'Preparation and availability pragmatically signal assumption of the requested task without directly promising it.','Signals uptake of an explicitly conditional proposal; the unresolved context prerequisite qualifies the undertaking.')
rows=[]
for p in pairs:
 for v,(ctx,label,rationale) in enumerate(p['variants']):
  rows.append({'sample_id':p['pair_id']+f'_{v+1}','pair_id':p['pair_id'],'contrast_type':p['contrast_type'],'conversation_context':ctx,'target_speaker':'employee','target_utterance':p['target'],'commitment_label':['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT'][label],'label_id':label,'annotation_rationale':rationale,'annotation_source':'agent-authored before inference','human_review_required':True,'model_input_text':f"[CONTEXT] {ctx} [TARGET SPEAKER] employee [TARGET UTTERANCE] {p['target']}"})
assert len(rows)==80 and len(pairs)==40 and len({p['target'] for p in pairs})==40
assert collections.Counter(r['label_id'] for r in rows)=={0:20,1:20,2:20,3:20}

# Freeze challenge and audit before loading/evaluating either model.
DROOT=Path('/content/latent_commitment_context_dependent_v1'); DRES=DROOT/'results';DRES.mkdir(parents=True,exist_ok=True)
challenge=pd.DataFrame(rows)
challenge_json=json.dumps(rows,indent=2)
assert hashlib.sha256(challenge_json.encode()).hexdigest()=='379e2656eaccd5b177987d1981edc3b9b110dafc660c594565aa73c5eccc29e9'
(DRES/'challenge_v1.json').write_text(challenge_json)
challenge.to_csv(DRES/'challenge_v1.csv',index=False)
for pid,g in challenge.groupby('pair_id'):
    assert len(g)==2 and g.target_utterance.nunique()==1 and g.target_speaker.nunique()==1 and g.label_id.nunique()==2 and g.conversation_context.nunique()==2
all_original=pd.concat(list(experiment_data.values()),ignore_index=True)
challenge_audit={'n_examples':len(challenge),'n_pairs':challenge.pair_id.nunique(),'n_unique_targets':challenge.target_utterance.nunique(),'class_counts':challenge.commitment_label.value_counts().to_dict(),'challenge_json_sha256':hashlib.sha256(challenge_json.encode()).hexdigest(),'exact_full_input_overlap_with_original_dataset':int(challenge.model_input_text.isin(all_original.model_input_text).sum()),'exact_target_overlap_with_original_dataset':int(challenge.target_utterance.isin(all_original.target_utterance).sum()),'target_only_accuracy_ceiling':0.5,'target_only_pair_exact_ceiling':0.0,'annotation_source':'agent-authored; no independent human validation','labels_frozen_before_inference':True,'no_training_or_selection_on_challenge':True}
assert challenge_audit['exact_full_input_overlap_with_original_dataset']==0
(DRES/'challenge_audit.json').write_text(json.dumps(challenge_audit,indent=2))
print('CHALLENGE FROZEN BEFORE INFERENCE',json.dumps(challenge_audit,indent=2),flush=True)


In [ ]:
# EXPERIMENT D — evaluate frozen checkpoints; never train or tune on challenge.
import gc,time
assert torch.cuda.is_available()
d_y=challenge.label_id.to_numpy()
d_paths={'contextual':Path('/content/latent_commitment_template_independent_v1/best_model'),'target_only':Path('/content/latent_commitment_context_ablation_v1/best_model')}
# Free prior inference/training objects, preserving all checkpoints.
for name in ['target_trainer','target_model','target_reloaded','experiment_reloaded']:
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache()
@torch.inference_mode()
def d_infer(texts,mdl,tok):
    pieces=[]
    for start in range(0,len(texts),8):
        batch=tok(texts[start:start+8],padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        pieces.append(mdl(**batch).logits.float().cpu().numpy())
    return np.concatenate(pieces)
d_logits={};d_predictions=challenge.copy();d_summaries=[];d_pairs=[];d_types=[];d_models={}
for kind,path in d_paths.items():
    assert (path/'model.safetensors').exists()
    tok=AutoTokenizer.from_pretrained(path,use_fast=False)
    mdl=AutoModelForSequenceClassification.from_pretrained(path).to('cuda').eval()
    assert {int(k):v for k,v in mdl.config.id2label.items()}==id2label
    if kind=='contextual':texts=challenge.model_input_text.tolist()
    else:texts=challenge.target_utterance.drop_duplicates().tolist()
    lengths=[len(x) for x in tok(texts,add_special_tokens=True,truncation=False)['input_ids']]
    assert max(lengths)<=256,'Challenge truncated: fix evaluation before running.'
    begin=time.time();logits=d_infer(texts,mdl,tok)
    if kind=='target_only':
        unique_map={t:logit for t,logit in zip(texts,logits)}
        logits=np.stack([unique_map[t] for t in challenge.target_utterance])
    pred=logits.argmax(1);probs=a_probs(logits);d_logits[kind]=logits
    d_predictions[kind+'_prediction']=[id2label[int(x)] for x in pred]
    d_predictions[kind+'_correct']=pred==d_y
    d_predictions[kind+'_confidence']=probs.max(1)
    for k,label in id2label.items():d_predictions[kind+'_prob_'+label]=probs[:,k]
    prs=[]
    for pid,g in challenge.groupby('pair_id',sort=False):
        idx=g.index.to_numpy();a,b=idx
        pr={'model':kind,'pair_id':pid,'contrast_type':g.contrast_type.iloc[0],'both_correct':bool((pred[idx]==d_y[idx]).all()),'prediction_changes':bool(pred[a]!=pred[b]),'examples_correct':int((pred[idx]==d_y[idx]).sum()),'probability_total_variation':float(np.abs(probs[a]-probs[b]).sum()/2)}
        prs.append(pr);d_pairs.append(pr)
    summary={'model':kind,**metrics(d_y,pred),'correct_examples':int((pred==d_y).sum()),'n_examples':len(d_y),'pair_exact_accuracy':float(np.mean([r['both_correct'] for r in prs])),'pairs_both_correct':sum(r['both_correct'] for r in prs),'n_pairs':len(prs),'pair_prediction_change_rate':float(np.mean([r['prediction_changes'] for r in prs])),'mean_pair_probability_total_variation':float(np.mean([r['probability_total_variation'] for r in prs]))}
    d_summaries.append(summary)
    for category,g in challenge.groupby('contrast_type',sort=False):
        idx=g.index.to_numpy();pp=[r for r in prs if r['contrast_type']==category]
        d_types.append({'model':kind,'contrast_type':category,'n_examples':len(idx),'accuracy':float((pred[idx]==d_y[idx]).mean()),'pair_exact_accuracy':float(np.mean([r['both_correct'] for r in pp])),'pair_prediction_change_rate':float(np.mean([r['prediction_changes'] for r in pp]))})
    report_d=classification_report(d_y,pred,labels=range(4),target_names=NAMES,zero_division=0,output_dict=True)
    pd.DataFrame(report_d).T.to_csv(DRES/(kind+'_classification_report.csv'))
    pd.DataFrame(confusion_matrix(d_y,pred,labels=range(4)),index=NAMES,columns=NAMES).to_csv(DRES/(kind+'_confusion_matrix.csv'))
    d_models[kind]={'checkpoint_directory':str(path),'weights_sha256':hashlib.sha256((path/'model.safetensors').read_bytes()).hexdigest(),'config_sha256':hashlib.sha256((path/'config.json').read_bytes()).hexdigest(),'max_input_tokens':max(lengths),'truncated':0,'inference_seconds':time.time()-begin,'precision':'float32 eval; no gradients'}
    print('CHALLENGE RESULT',json.dumps(summary),flush=True)
    del mdl;gc.collect();torch.cuda.empty_cache()
assert next(r for r in d_summaries if r['model']=='target_only')['pair_prediction_change_rate']==0
assert next(r for r in d_summaries if r['model']=='target_only')['accuracy']<=0.5
pd.DataFrame(d_summaries).to_csv(DRES/'model_comparison.csv',index=False)
pd.DataFrame(d_types).to_csv(DRES/'contrast_type_results.csv',index=False)
pd.DataFrame(d_pairs).to_csv(DRES/'pair_results.csv',index=False)
d_predictions.to_csv(DRES/'challenge_predictions.csv',index=False)
np.savez_compressed(DRES/'challenge_logits.npz',labels=d_y,**d_logits)
(DRES/'model_provenance.json').write_text(json.dumps(d_models,indent=2))
print('CHALLENGE COMPLETE: full set retained; labels unchanged; both frozen models evaluated.',flush=True)
print(pd.DataFrame(d_types).to_string(index=False),flush=True)


In [ ]:
# Exports and interpretation; all results use the original frozen challenge labels.
from google.colab import _message
import shutil,zipfile
import matplotlib.pyplot as plt,seaborn as sns
d_comparison=pd.DataFrame(d_summaries)
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,key,title in zip(axes,['accuracy','pair_exact_accuracy'],['Example accuracy','Both contexts correct per pair']):
    ax.bar(d_comparison.model,d_comparison[key],color=['#2878b5','#e68a2e']);ax.set_ylim(0,1);ax.set_title(title);ax.set_ylabel('Proportion')
    for i,v in enumerate(d_comparison[key]):ax.text(i,min(v+.035,.96),f'{v:.1%}',ha='center')
fig.tight_layout();fig.savefig(DRES/'challenge_comparison.png',dpi=170);plt.close(fig)
for kind,logits in d_logits.items():
    fig,ax=plt.subplots(figsize=(8,6));sns.heatmap(confusion_matrix(d_y,logits.argmax(1),labels=range(4)),annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES,ax=ax)
    ax.set(xlabel='Predicted label',ylabel='Challenge label',title=kind+' — paired context challenge');plt.xticks(rotation=25,ha='right');fig.tight_layout();fig.savefig(DRES/(kind+'_confusion_matrix.png'),dpi=150);plt.close(fig)
# Transparent scope: conditional-scope labels are a separate, harder policy-dependent subset.
d_subsets=[]
for kind,logits in d_logits.items():
    pred=logits.argmax(1)
    for subset,mask in [('factual_vs_uptake',challenge.contrast_type.isin(['no_vs_latent','no_vs_explicit']).to_numpy()),('inherited_condition',challenge.contrast_type.isin(['explicit_vs_qualified','latent_vs_qualified']).to_numpy())]:
        d_subsets.append({'model':kind,'subset':subset,'n_examples':int(mask.sum()),**metrics(d_y[mask],pred[mask])})
pd.DataFrame(d_subsets).to_csv(DRES/'subset_results.csv',index=False)
(DRES/'ANNOTATION_GUIDE.md').write_text('''# Paired context challenge v1 — provisional labeling guide
Unit: the current employee turn interpreted in the preceding manager context. Classify the employee undertaking, not the manager's request. Context can supply the action, scope, and an unresolved condition.
No Commitment: factual response/confirmation with no acceptance of a future task.
Latent Commitment: relevant preparation or availability offered in response to an unresolved task, pragmatically signaling uptake without directly undertaking the action. This follows the supplied dataset's broad pragmatic definition; preparation alone in a factual context is No Commitment.
Explicit Commitment: direct acceptance of a specific future task. An affirmative answer to a future undertaking question counts as explicit even when the action is elided in the target.
Qualified Commitment: acceptance/uptake of a proposal whose explicit condition remains unresolved. A direct affirmative answer inherits that condition; it does not override it. This interpretation is tested separately in the inherited_condition subset, since a target-only annotation policy would differ.
Every row includes its rationale. All labels are agent-authored and were frozen with a SHA256 before either frozen checkpoint was evaluated. No independent human annotation or inter-annotator agreement has been obtained. Indirect uptake is defeasible and may attract disagreement; obtain blind human review before calling these gold-standard labels. Do not relabel based on which model is correct.
''')
(DRES/'README.md').write_text('''# Context-dependent paired diagnostic, v1
80 examples in 40 pairs; 40 distinct target utterances; 20 examples per class. Exactly identical target text and target-speaker metadata within each pair; different context and different intended label. Four contrast families, ten pairs each: no_vs_latent, no_vs_explicit, explicit_vs_qualified, latent_vs_qualified. These families reuse construction patterns and are not 40 independent domains.
This is a newly authored synthetic challenge, not an independent natural-data benchmark. No exact target or complete input overlaps any of the original 10000 rows. The label freeze, original-data overlap audit, per-row rationales, conditional-scope policy, model checkpoint paths and SHA256 hashes are exported. No training, threshold tuning, checkpoint selection, post-hoc row removal, or label revision occurred on this challenge.
Both frozen models are the template-held-out contextual checkpoint and its target-only ablation, each previously selected on validation. Contextual input is the original [CONTEXT] / [TARGET SPEAKER] / [TARGET UTTERANCE] serialization; target-only input is raw target_utterance. No annotation fields, IDs, or label names enter either model. Models run in float32 eval mode. No inputs are truncated.
Target-only is inferred once per unique target and its exact logits reused across the two contexts. A deterministic target-only model can get at most one example per pair correct (50% overall) and cannot get both correct (0% pair exact); this is a structural ceiling, not random chance performance. Uniform four-class chance is 25% example accuracy. Contextual prediction-change rate measures sensitivity, not correctness; pair exact accuracy requires both labels correct.
The factual_vs_uptake and inherited_condition subsets are separately reported. The latter tests whether a qualification in the manager's proposal scopes over the employee acceptance. See ANNOTATION_GUIDE.md for explicit policy and limitations.
This set was designed after earlier shortcut diagnostics and is exploratory. Scores describe these authored cases and their provisional labels; they do not estimate workplace population accuracy or prove all context reasoning fails. Independent blind human labeling and broader paraphrases are needed for stronger claims.
Contents: challenge CSV/JSON, rationales, model comparison, per-class reports, confusion matrices/charts, pair outcomes, predictions/probabilities, logits, overlap audit, checkpoint hashes, subset results, and updated same-notebook snapshot.
''')
print('CONTEXT-DEPENDENT CHALLENGE COMPLETE: 80 examples / 40 identical-target pairs')
for r in d_summaries:
    print('%s: %.2f%% accuracy (%d/80); macro-F1 %.4f; both contexts correct %d/40; predictions change %.1f%% of pairs' % (r['model'],100*r['accuracy'],r['correct_examples'],r['macro_f1'],r['pairs_both_correct'],100*r['pair_prediction_change_rate']))
print('Target-only structural ceiling: 50% examples, 0% complete pairs.')
print('No exact target/input overlap with original dataset. Labels frozen before inference.')
print('Labels are agent-authored, provisional; conditional-scope subset reported separately.')
print('Subset results:',pd.DataFrame(d_subsets).to_string(index=False))
shutil.copy(ERESULTS/'requirements.txt',DRES/'requirements.txt')
d_nb=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(DRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(d_nb,indent=1))
d_zip=shutil.make_archive(str(DROOT/'latent_commitment_context_dependent_results'),'zip',DRES)
with zipfile.ZipFile(d_zip) as z:assert z.testzip() is None
print('ZIP VERIFIED:',d_zip)


In [ ]:
"""Context training v1: fixed authored split families, no performance-based allocation."""
import csv,json,re,hashlib,itertools,difflib,random,collections
from pathlib import Path
LABELS_C=['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT']
REPLIES={
 'train':[
 ('Yes, I confirm your statement.','I have an hour open and the reference files ready.','I am not agreeing to own this work.'),
 ('That matches what I understand.','My next block is free, and the material is in front of me.','Please assign this to somebody else; I am declining.'),
 ('You have my agreement on that.','I already brought up the relevant pages and cleared a slot.','I am unable to accept this assignment.'),
 ('I agree with what you just said.','There is room on my calendar, and I have the documents here.','This is not an undertaking I can give you.'),
 ('That is an accurate account.','The working files are open beside me, and I have time.','I will not take responsibility for that task.'),
 ('I can confirm that is right.','I kept a gap in the day and collected the background notes.','I must turn down the proposed responsibility.'),
 ('Yes, you described it correctly.','I have the source material handy and a clear stretch ahead.','I cannot be the person accountable for this.'),
 ('I assent to that proposition.','My schedule has space, and the necessary records are at hand.','I am saying no to taking that on.')],
 'validation':[
 ('Your understanding matches mine.','I reserved a work period and located the supporting packet.','I decline the role you are offering.'),
 ('That gets a yes from me.','I left a window in the morning and retrieved the paperwork.','You need a different owner for this; I refuse.'),
 ('We are in agreement there.','The background is on my desk, with an empty slot ahead.','I am not willing to undertake the assignment.'),
 ('Affirmative; that is settled.','I made space between calls and found the reference copy.','That responsibility cannot be mine.')],
 'test':[
 ('You have my confirmation.','The briefing pack is within reach and my diary has an opening.','My answer is no; I will not be its owner.'),
 ('I endorse that statement.','I blocked out a working session and assembled the relevant evidence.','Count me out of carrying this through.'),
 ('Those words reflect my position.','I am sitting with the supporting documents during an unbooked hour.','Find another volunteer; I am passing on this.'),
 ('Yes, we understand each other.','I rearranged my appointments and gathered the source pages nearby.','I reject the invitation to be responsible for this.')]
}
# Each tuple: future action, unresolved authorization/prerequisite, task domain.
SCENARIOS={
 'train':[
 ('review the invoice ledger','finance releases the source entries','finance'),('reconcile the expense claims','the receipts become accessible','finance'),('prepare the payment summary','the controller approves the figures','finance'),('check the supplier balances','the bank file is delivered','finance'),
 ('update the access register','security grants permission','security'),('inspect the firewall checklist','the test account is activated','security'),('compile the incident timeline','the investigation lead releases the logs','security'),('verify the backup inventory','the storage scan finishes','security'),
 ('organize the recruitment agenda','the panel confirms attendance','people'),('revise the induction notes','HR approves the new policy','people'),('prepare the training roster','the department releases the names','people'),('check the leave spreadsheet','the team leads approve the entries','people'),
 ('compare the delivery quotes','procurement opens the tender files','purchasing'),('summarize the vendor proposals','the buyer authorizes the assessment','purchasing'),('review the order schedule','the warehouse confirms capacity','purchasing'),('check the stock adjustment','the inventory count is signed','purchasing'),
 ('edit the product announcement','marketing approves the draft','marketing'),('assemble the campaign brief','the sponsor clears the budget','marketing'),('review the customer newsletter','the editor grants clearance','marketing'),('prepare the launch checklist','the product team confirms the date','marketing'),
 ('check the deployment notes','engineering releases the build','engineering'),('revise the testing guide','the maintainer approves the changes','engineering'),('summarize the defect reports','the tracker becomes available','engineering'),('review the integration plan','the partner supplies the interface','engineering'),
 ('prepare the service overview','support releases the case list','support'),('check the escalation queue','the shift supervisor authorizes access','support'),('revise the troubleshooting notes','the specialist confirms the fix','support'),('summarize the outage tickets','the monitoring export arrives','support'),
 ('review the facilities rota','the building manager approves staffing','facilities'),('prepare the equipment log','the technician completes inspection','facilities'),('check the office supplies list','the department approves its request','facilities'),('revise the maintenance schedule','the contractor confirms the visit','facilities'),
 ('summarize the workshop feedback','the organizer releases the responses','events'),('prepare the seminar handout','the presenter approves the content','events'),('check the visitor schedule','reception confirms the appointments','events'),('review the travel bookings','the coordinator clears the itinerary','events'),
 ('revise the document index','the archivist releases the catalogue','records'),('check the records transfer','the custodian authorizes the move','records'),('prepare the retention table','the policy owner signs the revision','records'),('summarize the meeting minutes','the chair approves the record','records')],
 'validation':[
 ('map the clinical equipment locations','the hospital administrator grants clearance','hospital_operations'),('compile the laboratory calibration list','the laboratory head approves the access','laboratory'),('review the museum loan schedule','the curator releases the loan records','museum'),('prepare the library digitization plan','the librarian approves scanning','library'),('check the kitchen allergen guide','the catering lead confirms ingredients','catering'),('summarize the fleet inspection sheets','the transport manager authorizes release','fleet'),('organize the theatre rehearsal timetable','the director confirms availability','theatre'),('revise the farm delivery manifest','the farm operator approves dispatch','agriculture'),('prepare the nursery watering chart','the horticulturist signs the plan','horticulture'),('check the sports venue reservations','the venue officer approves the bookings','sports'),('summarize the repair depot backlog','the depot manager releases the job cards','repair'),('review the radio studio allocations','the producer confirms the recording slots','broadcasting')],
 'test':[
 ('assemble the harbour mooring register','the harbourmaster authorizes the inspection','maritime'),('check the solar array inspection notes','the site engineer permits entry','renewable_energy'),('review the railway platform signage','the station controller clears the work','railway'),('prepare the geological sample inventory','the field lead releases the specimens','geology'),('summarize the observatory instrument bookings','the astronomer approves instrument use','astronomy'),('revise the water treatment sampling plan','the plant chemist validates the protocol','water'),('compile the textile loom service records','the mill supervisor permits the audit','textiles'),('organize the wildlife survey route','the ranger issues the field permit','conservation'),('review the bridge inspection photographs','the civil engineer releases the images','civil_engineering'),('check the bakery production sequence','the production chief confirms the batch order','bakery'),('prepare the recycling depot handover','the depot controller approves the transfer','recycling'),('summarize the wind tunnel experiment schedule','the researcher authorizes the test window','aerodynamics')]
}
# Context templates are allocated wholly by split. All contain placeholders, never label names.
TEMPLATES={
 'train':[
 ('We need someone to {a} {t}. Can we agree that you will be the person doing it?', 'The proposal is that you {a} {t}, but only if {c}. The prerequisite is still pending; without it the work will not go ahead. Do you accept that arrangement?', 'For the records check, the material needed to {a} is in the shared folder. Is that statement accurate?', 'For the availability record, what time and reference material do you currently have? We are gathering facts, not assigning the work to {a}.'),
 ('Can I put you down to {a} {t}? The work is ready to start.', 'Can I put you down to {a} {t}, contingent on {c}? That event has not occurred, so the arrangement depends on it.', 'The office record says we have a request to {a}. Are those facts correct?', 'We are documenting capacity before deciding who will {a}. Describe your available time and material; no owner is being chosen in this conversation.'),
 ('Will you personally {a} {t}? We have everything required.', 'Would you take on the plan to {a} {t} provided {c}? That permission is not yet available, and the plan is off otherwise.', 'Someone asked us to {a}. I am only checking that this is what the note says. Do you agree that the note says that?', 'For a factual desk and calendar check, tell me what is ready and when you are free. This question does not ask you to {a}.'),
 ('I propose that you {a} {t}. Do you agree to be responsible?', 'I propose that you {a} {t} subject to {c}. The decision is unresolved, so that restriction remains part of the proposal. Do you agree?', 'The background files for the request to {a} exist in the archive. Do you confirm that description?', 'Before any assignment to {a}, describe the time and documents you have. I am recording resources only.'),
 ('We have no owner to {a} {t}. Are you accepting the job?', 'We need an owner to {a} {t} if {c}, and that has not been decided. Are you accepting this dependent arrangement?', 'The planning list includes an item to {a}. Am I reporting that correctly?', 'I am doing a schedule survey, not choosing someone to {a}. What preparation and space do you currently have?'),
 ('Please confirm the agreement that you will {a} {t}. Nothing is waiting on authorization.', 'Please confirm the agreement that you will {a} {t} only after {c}. Authorization remains outstanding and the job depends on it.', 'I read that there is a request to {a}. Is my reading of the document correct, without discussing who will do it?', 'For my factual notes, describe your current readiness. This is an inventory question, not a request to {a}.'),
 ('Who can take responsibility to {a} {t}? Is that something you are taking on?', 'Who will take responsibility to {a} {t} on the condition that {c}? We do not have that clearance yet. Is that conditional plan acceptable?', 'The reference card mentions a plan to {a}. Is that an accurate description of the card?', 'I need your calendar status and what is on your desk for the resource census; nobody is being assigned to {a} here.'),
 ('Can we settle on you as the owner to {a} {t}? All prerequisites have already been satisfied.', 'Can we settle on you to {a} {t} in the event that {c}? The event is uncertain and has not occurred, so there is no work without it.', 'Does the task record correctly state that there was a request to {a}? This is a check of the record only.', 'For the capacity spreadsheet, report your available resources, rather than accepting a task to {a}.')],
 'validation':[
 ('Your proposed role would be to {a} {t}. Shall we treat that as your undertaking?', 'Your proposed role would be to {a} {t} once {c}. We are awaiting that prerequisite, and the offer is limited by it. Does that describe what you accept?', 'Do you endorse this factual statement: our archive contains documents relevant to the proposal to {a}?', 'Give me a snapshot of your work area and diary. We will choose who should {a} in another meeting, not in this one.'),
 ('Are we agreed that you are the one who will {a} {t}? The necessary approval is in place.', 'Are we agreed that you would {a} {t} dependent on {c}? Nobody has obtained that prerequisite yet.', 'Our request log contains a line about whether to {a}. Is my account of the log true?', 'I want a factual description of your current capacity before we talk about responsibility to {a}.'),
 ('I would like you to {a} {t}. Are you committing to that?', 'I would like you to {a} {t} if and only if {c}. We still lack that decision; is the constrained proposal what you are accepting?', 'The information sheet refers to a possible need to {a}. Are you confirming the sheet contains that information?', 'Tell me what material you have to hand and how busy you are. This is for a resource audit; it does not appoint you to {a}.'),
 ('May I record your acceptance of the responsibility to {a} {t}? The task is authorized.', 'May I record your acceptance to {a} {t} as long as {c}? That authorization is outstanding; it limits the arrangement.', 'Please verify the truth of the statement that our notes discuss an item to {a}.', 'Describe your preparation and spare time for the scheduling survey. We have deliberately left ownership to {a} undecided today.')],
 'test':[
 ('Shall your name be entered as accountable for the work to {a} {t}? We can proceed immediately.', 'Shall your name be entered to {a} {t} with the proviso that {c}? The proviso is unsatisfied; the arrangement lapses without it.', 'In a fact check, I said that our documents mention a proposal to {a}. Was my statement true?', 'This interview concerns the present contents of your desk and diary. Allocation of the work to {a} is outside this interview. What is your current situation?'),
 ('Do you give your agreement to personally {a} {t}? There are no outstanding permissions.', 'Do you give your agreement to {a} {t} contingent upon {c}? That requirement has not been met and cannot be waived.', 'Does your view concur with the historical fact that the request to {a} was entered in the system?', 'We are surveying current readiness only. The choice of who will {a} will be made later by someone else. Describe what you already have available.'),
 ('I am asking for your undertaking to {a} {t}. Are we of one mind about you doing the work?', 'I am asking whether you accept the limited undertaking to {a} {t}, pending {c}. That is unresolved, and there is no authorization to proceed without it.', 'The question is whether my description of the paperwork about the request to {a} is accurate. Do you support that description?', 'For an observational study, describe your free time and nearby reference materials. This interview offers no assignment to {a}.'),
 ('Would you assent to being responsible to {a} {t}? You can start without waiting for anything.', 'Would you assent to the arrangement to {a} {t} in case {c}? That contingency is still open; failure to satisfy it cancels the task.', 'Please affirm or reject this account of the facts: the task list discusses whether to {a}.', 'We need a descriptive account of your calendar and papers, with responsibility to {a} reserved for a future discussion.')]
}
TIMES={'train':['this morning','this afternoon','tomorrow','before the next briefing','by the end of the week','during the next shift','before Thursday','after the team meeting'],'validation':['before the monthly review','during the upcoming work session','by next Wednesday','over the next two days'],'test':['ahead of the next inspection','during the following duty period','before the project checkpoint','by the next planning call']}
def build_context_training(destination):
 out=Path(destination);out.mkdir(parents=True,exist_ok=True);allrows={};manifest=[]
 for split in ['train','validation','test']:
  rows=[]
  for si,(action,condition,domain) in enumerate(SCENARIOS[split]):
   for fi,(direct,indirect,refusal) in enumerate(REPLIES[split]):
    u,c,fd,fi_ctx=TEMPLATES[split][fi];fmt={'a':action,'c':condition,'t':TIMES[split][fi]}
    contexts={'U':u.format(**fmt),'C':c.format(**fmt),'FD':fd.format(**fmt),'FI':fi_ctx.format(**fmt)}
    gid=f'{split}_S{si:02}_F{fi:02}'
    specs=[('direct','FD',0,direct),('direct','U',3,direct),('direct','C',2,direct),('indirect','FI',0,indirect),('indirect','U',1,indirect),('indirect','C',2,indirect),('refusal','U',0,refusal),('refusal','C',0,refusal)]
    for j,(replykind,ctxkind,label,target) in enumerate(specs):
     ctx='[manager]: '+contexts[ctxkind]
     rationale={0:('Explicit refusal of responsibility.' if replykind=='refusal' else 'A factual confirmation/readiness report, with no task acceptance requested.'),1:'Readiness/preparation answers an ownership request and pragmatically signals uptake without a direct undertaking.',2:'The employee accepts or signals uptake of a proposal whose unresolved prerequisite remains in scope.',3:'A direct affirmative confirmation accepts the contextual proposition that the employee will perform the task.'}[label]
     rows.append({'sample_id':gid+f'_{j}','scenario_id':f'{split}_S{si:02}','family_id':f'{split}_F{fi:02}','contrast_group_id':gid,'target_group_id':gid+'_'+replykind,'reply_kind':replykind,'context_kind':ctxkind,'domain':domain,'split':split,'conversation_context':ctx,'target_speaker':'employee','target_utterance':target,'model_input_text':f'[CONTEXT] {ctx} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target}','commitment_label':LABELS_C[label],'labels':label,'annotation_rationale':rationale,'human_review_required':True})
  random.Random(42).shuffle(rows);allrows[split]=rows
  with (out/(split+'.csv')).open('w',newline='') as f:
   w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
  manifest.extend({k:r[k] for k in ['sample_id','scenario_id','family_id','contrast_group_id','target_group_id','split']} for r in rows)
 audit={'seed':42,'split_method':'authored disjoint reply/context families and scenario domains; fixed before training','counts':{},'cross_split_overlap':{},'csv_sha256':{},'annotation':'agent-authored provisional; no human validation','earlier_challenge_used_for_training':False}
 def norm(x):return ' '.join(re.findall(r'[a-z]+',x.lower()))
 for split,rows in allrows.items():
  assert len({r['model_input_text'] for r in rows})==len(rows)
  groups=collections.defaultdict(list)
  for r in rows:groups[r['target_group_id']].append(r)
  for g in groups.values():
   assert len({r['target_utterance'] for r in g})==1
   if g[0]['reply_kind']!='refusal':assert len(g)==3 and len({r['labels'] for r in g})==3
  def ceiling(field):
   grouped=collections.defaultdict(collections.Counter)
   for r in rows:grouped[r[field]][r['labels']]+=1
   return sum(max(c.values()) for c in grouped.values())/len(rows)
  audit['counts'][split]={'rows':len(rows),'class_counts':dict(collections.Counter(r['commitment_label'] for r in rows)),'families':len(REPLIES[split]),'scenarios':len(SCENARIOS[split]),'context_dependent_triplets':sum(g[0]['reply_kind']!='refusal' for g in groups.values()),'refusal_control_groups':sum(g[0]['reply_kind']=='refusal' for g in groups.values()),'target_only_empirical_ceiling':ceiling('target_utterance'),'context_only_empirical_ceiling':ceiling('conversation_context')}
  audit['csv_sha256'][split]=hashlib.sha256((out/(split+'.csv')).read_bytes()).hexdigest()
 for a,b in itertools.combinations(allrows,2):
  check={}
  for col in ['model_input_text','conversation_context','target_utterance','scenario_id','family_id','contrast_group_id','target_group_id','domain']:
   check[col]=len({norm(r[col]) for r in allrows[a]}&{norm(r[col]) for r in allrows[b]})
   assert check[col]==0,(a,b,col,check[col])
  for name,items in [('target_templates',REPLIES),('context_templates',TEMPLATES)]:
   aa=[norm(t) for p in items[a] for t in p];bb=[norm(t) for p in items[b] for t in p]
   check['max_'+name+'_token_similarity']=max(difflib.SequenceMatcher(None,x.split(),y.split(),autojunk=False).ratio() for x in aa for y in bb)
  audit['cross_split_overlap'][a+'_'+b]=check
 with (out/'split_manifest.csv').open('w',newline='') as f:
  w=csv.DictWriter(f,fieldnames=list(manifest[0]));w.writeheader();w.writerows(manifest)
 (out/'split_audit.json').write_text(json.dumps(audit,indent=2))
 return allrows,audit

# EXPERIMENT E: independent splits frozen before model training.
CROOT=Path('/content/latent_commitment_context_training_v1'); CRES=CROOT/'results'; CSPLIT=CROOT/'splits';CRES.mkdir(parents=True,exist_ok=True)
c_rows,c_audit=build_context_training(CSPLIT)
c_expected={'train':'837f88ef956e09fe6868e2537535207bab89bbeba346623b811acd1e89a3b9aa','validation':'f42dd19b1bb6d957f17556dda49d5ba9d2cc0aa31413277e8fea61c1335a8337','test':'b5ae9fc9ff1de819f71f32ee28f01b553414dbf26aa0a921ccf5a7c4fc1acbb7'}
assert c_audit['csv_sha256']==c_expected
c_data={s:pd.DataFrame(rs) for s,rs in c_rows.items()}
previous_challenge=pd.read_csv('/content/latent_commitment_context_dependent_v1/results/challenge_v1.csv')
c_audit['previous_challenge_overlap']={s:{col:int(df[col].isin(previous_challenge[col]).sum()) for col in ['model_input_text','target_utterance']} for s,df in c_data.items()}
assert all(v==0 for d in c_audit['previous_challenge_overlap'].values() for v in d.values())
(CSPLIT/'split_audit.json').write_text(json.dumps(c_audit,indent=2))
def c_dataset(split):
    frame=c_data[split];enc=tokenizer(frame.model_input_text.tolist(),truncation=True,max_length=256);enc['labels']=frame.labels.tolist();return Dataset.from_dict(enc)
c_trunc={s:{'max_tokens':max(map(len,tokenizer(df.model_input_text.tolist(),truncation=False)['input_ids'])),'truncated':sum(len(x)>256 for x in tokenizer(df.model_input_text.tolist(),truncation=False)['input_ids'])} for s,df in c_data.items()}
assert all(v['truncated']==0 for v in c_trunc.values())
(CRES/'tokenization_audit.json').write_text(json.dumps(c_trunc,indent=2))
c_train_ds=c_dataset('train');c_val_ds=c_dataset('validation')
# Baselines fit train only; validation results reported before final held-out evaluation.
c_baselines={};c_baseline_rows=[]
for column in ['target_utterance','conversation_context','model_input_text']:
    baseline=make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=1),LogisticRegression(max_iter=2000,random_state=42))
    baseline.fit(c_data['train'][column],c_data['train'].labels);c_baselines[column]=baseline
    c_baseline_rows.append({'model':'TF-IDF + LR','input':column,'split':'validation',**metrics(c_data['validation'].labels,baseline.predict(c_data['validation'][column]))})
print('CONTEXT TRAINING DATA FROZEN; ZERO CROSS-SPLIT EXACT OVERLAPS',json.dumps(c_audit,indent=2),flush=True)
print('VALIDATION BASELINES:',pd.DataFrame(c_baseline_rows).to_string(index=False),flush=True)


In [ ]:
# CONTEXT TRAINING — fresh pretrained weights; same hyperparameters, never reuse previous weights
import gc, time, shutil
assert torch.cuda.is_available()
for name in ['target_trainer','target_model','target_reloaded','experiment_trainer','experiment_model','experiment_reloaded']:
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache();set_seed(42)
class ExperimentProgress(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        (CRES/'trainer_log.json').write_text(json.dumps(state.log_history,indent=2))
        if logs:print('CONTEXT TRAINING',state.global_step,json.dumps(logs),flush=True)
    def on_save(self,args,state,control,**kwargs):
        print('CONTEXT TRAINING CHECKPOINT:',state.global_step,'best validation macro-F1:',state.best_metric,flush=True)
c_model=AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base',num_labels=4,id2label=id2label,label2id=label2id)
c_args=TrainingArguments(output_dir=str(CROOT/'checkpoints'),num_train_epochs=3,learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=0.01,warmup_ratio=0.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=2,logging_steps=50,seed=42,data_seed=42,report_to='none',dataloader_num_workers=2,save_safetensors=True,disable_tqdm=True)
c_trainer=Trainer(model=c_model,args=c_args,train_dataset=c_train_ds,eval_dataset=c_val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8),compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2),ExperimentProgress()])
c_start=time.time();c_train_output=c_trainer.train();c_seconds=time.time()-c_start
CMODEL=CROOT/'best_model';c_trainer.save_model(str(CMODEL));tokenizer.save_pretrained(str(CMODEL))
c_epoch_logs=list(c_trainer.state.log_history)
pd.DataFrame([r for r in c_epoch_logs if 'loss' in r]).to_csv(CRES/'training_metrics.csv',index=False)
pd.DataFrame([r for r in c_epoch_logs if 'eval_loss' in r]).to_csv(CRES/'validation_metrics.csv',index=False)
c_info={'model_initialization':'fresh microsoft/deberta-v3-base; no previous fine-tuned weights','gpu':torch.cuda.get_device_name(0),'training_seconds':c_seconds,'epochs_completed':c_trainer.state.epoch,'best_checkpoint':c_trainer.state.best_model_checkpoint,'best_validation_macro_f1':c_trainer.state.best_metric,'global_step':c_trainer.state.global_step,'seed':42,'split_csv_sha256':c_audit['csv_sha256'],'max_length':256,'context_training_v1':True,'test_used_for_selection':False}
assert np.isfinite(c_info['best_validation_macro_f1'])
(CRES/'run_info.json').write_text(json.dumps(c_info,indent=2))
print('CONTEXT TRAINING TRAINING COMPLETED',json.dumps(c_info,indent=2))


In [ ]:
# EXPERIMENT E — one test pass after validation-only selection.
assert not (CRES/'heldout_predictions.npz').exists(),'Reuse saved predictions; do not retest after selection.'
c_out=c_trainer.predict(c_dataset('test'),metric_key_prefix='test')
c_logits=c_out.predictions;c_y=c_out.label_ids;c_pred=c_logits.argmax(1)
np.savez_compressed(CRES/'heldout_predictions.npz',logits=c_logits,labels=c_y)
c_frame=c_data['test'].reset_index(drop=True).copy()
assert np.array_equal(c_y,c_frame.labels.to_numpy())
def c_pair_metrics(frame,pred,tag):
    pairrows=[];triplets=[]
    for gid,g in frame.groupby('target_group_id',sort=False):
        idx=g.index.to_numpy()
        if g.reply_kind.iloc[0]=='refusal':continue
        assert len(idx)==3 and g.target_utterance.nunique()==1 and g.labels.nunique()==3
        triplets.append({'model':tag,'target_group_id':gid,'family_id':g.family_id.iloc[0],'scenario_id':g.scenario_id.iloc[0],'reply_kind':g.reply_kind.iloc[0],'all_three_correct':bool((pred[idx]==frame.labels.to_numpy()[idx]).all()),'three_different_predictions':len(set(pred[idx]))==3})
        for a,b in itertools.combinations(idx,2):
            pairrows.append({'model':tag,'pair_id':frame.sample_id.iloc[a]+'__'+frame.sample_id.iloc[b],'target_group_id':gid,'reply_kind':g.reply_kind.iloc[0],'label_pair':'_'.join(map(str,sorted([int(frame.labels.iloc[a]),int(frame.labels.iloc[b])]))),'both_correct':bool(pred[a]==frame.labels.iloc[a] and pred[b]==frame.labels.iloc[b]),'prediction_changes':bool(pred[a]!=pred[b])})
    summary={'model':tag,**metrics(frame.labels,pred),'correct_examples':int((pred==frame.labels.to_numpy()).sum()),'n_examples':len(frame),'n_context_dependent_pairs':len(pairrows),'pair_both_correct':sum(r['both_correct'] for r in pairrows),'pair_exact_accuracy':float(np.mean([r['both_correct'] for r in pairrows])),'n_triplets':len(triplets),'triplets_all_correct':sum(r['all_three_correct'] for r in triplets),'triplet_exact_accuracy':float(np.mean([r['all_three_correct'] for r in triplets])),'pair_prediction_change_rate':float(np.mean([r['prediction_changes'] for r in pairrows]))}
    return summary,pairrows,triplets
c_test_metrics,c_pairrows,c_tripletrows=c_pair_metrics(c_frame,c_pred,'Context-trained DeBERTa')
c_report=classification_report(c_y,c_pred,labels=range(4),target_names=NAMES,zero_division=0,output_dict=True)
pd.DataFrame(c_report).T.to_csv(CRES/'classification_report.csv')
c_cm=confusion_matrix(c_y,c_pred,labels=range(4));pd.DataFrame(c_cm,index=NAMES,columns=NAMES).to_csv(CRES/'confusion_matrix.csv')
pd.DataFrame([c_test_metrics]).to_csv(CRES/'test_metrics.csv',index=False)
c_predictions=c_frame.copy();c_predictions['predicted_label']=[id2label[int(x)] for x in c_pred];c_predictions['correct']=c_pred==c_y
c_prob=a_probs(c_logits)
for k,label in id2label.items():c_predictions['prob_'+label]=c_prob[:,k]
c_predictions.to_csv(CRES/'test_predictions.csv',index=False)
c_comparisons=[c_test_metrics]
for column,baseline in c_baselines.items():
    pred=baseline.predict(c_frame[column]);row,pr,tr=c_pair_metrics(c_frame,pred,'TF-IDF '+column)
    c_comparisons.append(row);c_pairrows.extend(pr);c_tripletrows.extend(tr)
    c_baseline_rows.append({'model':'TF-IDF + LR','input':column,'split':'test',**metrics(c_y,pred)})
# Serialization verified using validation only, no second test pass.
c_probe=c_data['validation'].model_input_text.iloc[:16].tolist()
c_before=d_infer(c_probe,c_model.eval(),tokenizer)
c_trainer.model.to('cpu');gc.collect();torch.cuda.empty_cache()
c_reloaded=AutoModelForSequenceClassification.from_pretrained(CMODEL).to('cuda').eval()
c_tok=AutoTokenizer.from_pretrained(CMODEL,use_fast=False)
c_after=d_infer(c_probe,c_reloaded,c_tok)
assert np.allclose(c_before,c_after,atol=0.002,rtol=0.002)
c_reloaded.to('cpu');del c_reloaded;gc.collect();torch.cuda.empty_cache()
# Previous contextual model frozen before this dataset existed, for same-test comparison.
c_previous=AutoModelForSequenceClassification.from_pretrained('/content/latent_commitment_template_independent_v1/best_model').to('cuda').eval()
c_previous_tok=AutoTokenizer.from_pretrained('/content/latent_commitment_template_independent_v1/best_model',use_fast=False)
c_old_logits=d_infer(c_frame.model_input_text.tolist(),c_previous,c_previous_tok);np.savez_compressed(CRES/'previous_contextual_logits.npz',logits=c_old_logits,labels=c_y)
c_old_metrics,pr,tr=c_pair_metrics(c_frame,c_old_logits.argmax(1),'Previous contextual DeBERTa');c_comparisons.append(c_old_metrics);c_pairrows.extend(pr);c_tripletrows.extend(tr)
del c_previous;gc.collect();torch.cuda.empty_cache()
pd.DataFrame(c_comparisons).to_csv(CRES/'same_test_model_comparison.csv',index=False)
pd.DataFrame(c_baseline_rows).to_csv(CRES/'baseline_comparison.csv',index=False)
pd.DataFrame(c_pairrows).to_csv(CRES/'pair_results.csv',index=False)
pd.DataFrame(c_tripletrows).to_csv(CRES/'triplet_results.csv',index=False)
c_family_rows=[]
for col in ['family_id','domain','reply_kind','context_kind']:
    for key,g in c_predictions.groupby(col):c_family_rows.append({'grouping':col,'group':key,'n':len(g),'accuracy':float(g.correct.mean())})
pd.DataFrame(c_family_rows).to_csv(CRES/'subgroup_results.csv',index=False)
c_info.update({'test_metrics':c_test_metrics,'test_predicted_once_after_selection':True,'reload_verified_on_validation':True,'selected_weights_sha256':hashlib.sha256((CMODEL/'model.safetensors').read_bytes()).hexdigest()})
(CRES/'run_info.json').write_text(json.dumps(c_info,indent=2))
print('CONTEXT-TRAINED HELD-OUT TEST:',json.dumps(c_test_metrics,indent=2),flush=True)
print('PREVIOUS MODEL ON SAME NEW TEST:',json.dumps(c_old_metrics,indent=2),flush=True)
print('LATENT COMMITMENT:',json.dumps(c_report['Latent Commitment']),flush=True)
print('SAVED MODEL RELOAD VERIFIED. No test-driven tuning or data changes.',flush=True)


In [ ]:
# EXPERIMENT E — preserve splits, checkpoint, metrics, and notebook.
from google.colab import _message
import zipfile,shutil,matplotlib.pyplot as plt,seaborn as sns
for key,title,name in [('loss','Training loss','training_loss.png'),('eval_loss','Validation loss','validation_loss.png'),('eval_macro_f1','Validation macro-F1','validation_macro_f1.png')]:
    logs_k=[r for r in c_epoch_logs if key in r]
    fig,ax=plt.subplots(figsize=(7,4));ax.plot([r['epoch'] for r in logs_k],[r[key] for r in logs_k],marker='o');ax.set(xlabel='Training epoch',ylabel=key,title=title+' — context training');fig.tight_layout();fig.savefig(CRES/name,dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6));sns.heatmap(c_cm,annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES,ax=ax);ax.set(xlabel='Predicted',ylabel='True',title='Unseen context-dependent test');plt.xticks(rotation=25,ha='right');fig.tight_layout();fig.savefig(CRES/'confusion_matrix.png',dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4));ax.bar(NAMES,[c_report[n]['f1-score'] for n in NAMES]);ax.set_ylim(0,1);ax.set_ylabel('F1');ax.set_title('Unseen context-dependent test: per-class F1');plt.xticks(rotation=20,ha='right');fig.tight_layout();fig.savefig(CRES/'per_class_f1.png',dpi=160);plt.close(fig)
(CRES/'README.md').write_text(f'''# Context-dependent training experiment v1
New, separately stored synthetic dataset; original experiments and data remain intact.
Train 2560 rows: 40 scenarios from ten domains, eight authored dialogue/reply families. Validation 384 rows: twelve separate domains/scenarios, four different families. Test 384 rows: twelve further domains/scenarios, four further families. Exact inputs, contexts, targets, family IDs, scenario IDs and domains do not cross partitions. Every same-target group remains wholly in one partition. Cross-split target-template token SequenceMatcher ratio is at most0.60 and context-template ratio at most0.50. These are operational audits, not a guarantee of semantic independence or absence of every possible artifact.
All data/labels and allocation were fixed and hashed before training. Splits were not selected for performance; no example filtering or label editing followed test inference. The earlier80-row challenge has zero exact target/input overlap and was not used for training or selection. It is not presented as a fresh final test after its earlier analysis.
Each scenario/family has8 rows: one direct affirmative and one indirect preparation/availability reply each repeated under factual, unconditional-task, and unresolved-condition contexts (six rows), plus an explicit refusal under the latter two contexts (two controls). Direct labels are No/Explicit/Qualified; indirect labels are No/Latent/Qualified; refusals are No. The same task context consequently has different labels for different replies, preventing a context-only solution. Individual targets do not uniquely determine labels either.
Class distribution:50% No,25% Qualified,12.5% Latent,12.5% Explicit, identical in each split. Majority accuracy50%. Empirical deterministic target-only accuracy ceiling50%; context-only ceiling62.5%. Target-only ceiling on the six context-dependent rows per group is one-third. Macro-F1 and per-class metrics accompany accuracy to reveal imbalance.
Test has96 context-dependent triplets (48 direct and48 indirect), giving288 different-label same-target pairs. It also has48 two-context refusal control groups. Primary pair metrics exclude refusal controls. Pair outcomes are correlated within triplets;96triplets are not96independent utterance families. There are only four held-out reply/context families and twelve test domains. Domain/family breakdowns are included.
Label policy: factual agreement/preparation without uptake is No Commitment. Relevant preparation offered in answer to an ownership request counts as Latent Commitment under the supplied dataset's broad pragmatic definition. Direct affirmative confirmation of a future undertaking counts as Explicit Commitment, including ellipsis. An unresolved condition in the preceding proposal remains in scope for either kind of uptake, giving Qualified Commitment. Explicit refusal stays No regardless of request. These labels are agent-authored/provisional and not independently human-validated. Indirect uptake and inherited conditions may attract annotator disagreement. Obtain blind human review before treating this as a gold benchmark.
A fresh microsoft/deberta-v3-base was fine-tuned, not any earlier fine-tuned checkpoint. Seed42;3epochs;learning rate2e-5;AdamW;batch8;gradient accumulation2;evalbatch16;weightdecay0.01;warmup0.1;FP16;maxlength256;dynamicpadding;earlystoppingpatience2. Best checkpoint selected on validation macro-F1 only. Test predicted once after selection. No target,rationale,IDs,label metadata beyond original context/target serialization enters input. No input truncation. Saved weights/tokenizer reload verified on validation logits.
Training:{c_seconds:.2f}seconds on {c_info['gpu']}; selected checkpoint:{c_info['best_checkpoint']}; validation macro-F1:{c_info['best_validation_macro_f1']}.
New test:{json.dumps(c_test_metrics)}.
Previous contextual model evaluated on exactly the same new test:{json.dumps(c_old_metrics)}.
TF-IDF+LR target-only,context-only,full-input baselines fit only new train with fixed defaults; validation/test reported without tuning.
This is an exploratory demonstration that context-dependent supervision can change behavior across held-out authored templates and tasks. New contexts still contain systematic linguistic cues and share an annotation/generation mechanism. It is not evidence of robust general workplace understanding, and a high score would not eliminate shortcuts. Independent naturally occurring dialogues, adversarial paraphrases, and human annotation remain necessary.
Best model is in ../best_model and the separate model ZIP. Results ZIP includes dataset CSVs, generator source, split/audit hashes, manifest, cached predictions, pair/triplet results, same-test comparison, subgroup metrics, charts, requirements and updated notebook.
''')
shutil.copy(ERESULTS/'requirements.txt',CRES/'requirements.txt')
(CMODEL/'label_mappings.json').write_text(json.dumps({'label2id':label2id,'id2label':id2label,'display_names':NAMES},indent=2))
print('CONTEXT-DEPENDENT TRAINING COMPLETE')
print('Train2560 / validation384 / test384; disjoint reply/context families, scenarios and domains.')
print('No shared exact inputs, contexts or targets. All contrast groups remain intact.')
print('Selected using validation only: macro-F1 %.6f' % c_info['best_validation_macro_f1'])
print('New held-out test: %.2f%% accuracy (%d/384); macro-F1 %.6f' % (100*c_test_metrics['accuracy'],c_test_metrics['correct_examples'],c_test_metrics['macro_f1']))
print('Both labels correct: %d/288 pairs; all three correct: %d/96 triplets' % (c_test_metrics['pair_both_correct'],c_test_metrics['triplets_all_correct']))
print('Previous model on same test: %.2f%% accuracy; %d/96 triplets correct' % (100*c_old_metrics['accuracy'],c_old_metrics['triplets_all_correct']))
print('Latent Commitment F1: %.6f' % c_report['Latent Commitment']['f1-score'])
print('Model reload verified. Synthetic labels and limited family diversity remain limitations.')
c_nb=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(CRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(c_nb,indent=1))
c_builder=''.join(c_nb['cells'][20]['source']).split('# EXPERIMENT E: independent splits')[0]
(CSPLIT/'build_context_training.py').write_text(c_builder)
c_bundle=CROOT/'export';c_bundle.mkdir(exist_ok=True)
shutil.copytree(CRES,c_bundle/'results',dirs_exist_ok=True);shutil.copytree(CSPLIT,c_bundle/'splits',dirs_exist_ok=True)
c_results_zip=shutil.make_archive(str(CROOT/'latent_commitment_context_training_results'),'zip',c_bundle)
c_model_zip=shutil.make_archive(str(CROOT/'latent_commitment_context_trained_model'),'zip',CMODEL)
for archive in [c_results_zip,c_model_zip]:
    with zipfile.ZipFile(archive) as z:assert z.testzip() is None
print('RESULTS ZIP VERIFIED:',c_results_zip)
print('MODEL ZIP VERIFIED:',c_model_zip)


In [ ]:
# Final subgroup summary from cached predictions only; no further training or inference.
c_subset_rows=[]
for tag,pred in [('Context-trained DeBERTa',c_pred),('Previous contextual DeBERTa',c_old_logits.argmax(1))]:
    for subset,mask in [('context_dependent',c_frame.reply_kind.ne('refusal').to_numpy()),('refusal_controls',c_frame.reply_kind.eq('refusal').to_numpy())]:
        c_subset_rows.append({'model':tag,'subset':subset,'n':int(mask.sum()),'correct':int((pred[mask]==c_y[mask]).sum()),'accuracy':float((pred[mask]==c_y[mask]).mean())})
pd.DataFrame(c_subset_rows).to_csv(CRES/'context_dependent_vs_control_results.csv',index=False)
print('FINAL CONTEXT-DEPENDENT TRAINING RESULTS — cached predictions')
print(pd.DataFrame(c_subset_rows).to_string(index=False))
print('All held-out rows: new84.90% (326/384), previous52.86% (203/384).')
print('All-context triplets: new55/96, previous1/96; pair exact: new189/288, previous89/288.')
print('Macro-F1: new0.8051, previous0.5312. Latent F1:0.7848.')
print('No shared exact inputs/contexts/targets or assigned families/scenarios/domains across splits.')
print('Best checkpoint chosen on validation only; test predicted once. Saved model reload verified.')
print('Training229.1sec on Tesla T4. Synthetic provisional labels; only four test wording families.')
print('Model ZIP verified and results ZIP refreshed; no test-driven tuning.')
p=CRES/'README.md'
p.write_text(p.read_text()+'\nContext-dependent versus refusal-control subset results: '+json.dumps(c_subset_rows)+'\n')
c_nb=_message.blocking_request('get_ipynb',timeout_sec=30)['ipynb']
(CRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(c_nb,indent=1))
shutil.copytree(CRES,c_bundle/'results',dirs_exist_ok=True)
c_results_zip=shutil.make_archive(str(CROOT/'latent_commitment_context_training_results'),'zip',c_bundle)
with zipfile.ZipFile(c_results_zip) as z:assert z.testzip() is None
print('FINAL RESULTS ZIP VERIFIED:',c_results_zip)


In [ ]:
"""Context training v1: fixed authored split families, no performance-based allocation."""
import csv,json,re,hashlib,itertools,difflib,random,collections
from pathlib import Path
LABELS_C=['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT']
REPLIES={
 'train':[
 ('Yes, I confirm your statement.','I have an hour open and the reference files ready.','I am not agreeing to own this work.'),
 ('That matches what I understand.','My next block is free, and the material is in front of me.','Please assign this to somebody else; I am declining.'),
 ('You have my agreement on that.','I already brought up the relevant pages and cleared a slot.','I am unable to accept this assignment.'),
 ('I agree with what you just said.','There is room on my calendar, and I have the documents here.','This is not an undertaking I can give you.'),
 ('That is an accurate account.','The working files are open beside me, and I have time.','I will not take responsibility for that task.'),
 ('I can confirm that is right.','I kept a gap in the day and collected the background notes.','I must turn down the proposed responsibility.'),
 ('Yes, you described it correctly.','I have the source material handy and a clear stretch ahead.','I cannot be the person accountable for this.'),
 ('I assent to that proposition.','My schedule has space, and the necessary records are at hand.','I am saying no to taking that on.')],
 'validation':[
 ('Your understanding matches mine.','I reserved a work period and located the supporting packet.','I decline the role you are offering.'),
 ('That gets a yes from me.','I left a window in the morning and retrieved the paperwork.','You need a different owner for this; I refuse.'),
 ('We are in agreement there.','The background is on my desk, with an empty slot ahead.','I am not willing to undertake the assignment.'),
 ('Affirmative; that is settled.','I made space between calls and found the reference copy.','That responsibility cannot be mine.')],
 'test':[
 ('You have my confirmation.','The briefing pack is within reach and my diary has an opening.','My answer is no; I will not be its owner.'),
 ('I endorse that statement.','I blocked out a working session and assembled the relevant evidence.','Count me out of carrying this through.'),
 ('Those words reflect my position.','I am sitting with the supporting documents during an unbooked hour.','Find another volunteer; I am passing on this.'),
 ('Yes, we understand each other.','I rearranged my appointments and gathered the source pages nearby.','I reject the invitation to be responsible for this.')]
}
# Each tuple: future action, unresolved authorization/prerequisite, task domain.
SCENARIOS={
 'train':[
 ('review the invoice ledger','finance releases the source entries','finance'),('reconcile the expense claims','the receipts become accessible','finance'),('prepare the payment summary','the controller approves the figures','finance'),('check the supplier balances','the bank file is delivered','finance'),
 ('update the access register','security grants permission','security'),('inspect the firewall checklist','the test account is activated','security'),('compile the incident timeline','the investigation lead releases the logs','security'),('verify the backup inventory','the storage scan finishes','security'),
 ('organize the recruitment agenda','the panel confirms attendance','people'),('revise the induction notes','HR approves the new policy','people'),('prepare the training roster','the department releases the names','people'),('check the leave spreadsheet','the team leads approve the entries','people'),
 ('compare the delivery quotes','procurement opens the tender files','purchasing'),('summarize the vendor proposals','the buyer authorizes the assessment','purchasing'),('review the order schedule','the warehouse confirms capacity','purchasing'),('check the stock adjustment','the inventory count is signed','purchasing'),
 ('edit the product announcement','marketing approves the draft','marketing'),('assemble the campaign brief','the sponsor clears the budget','marketing'),('review the customer newsletter','the editor grants clearance','marketing'),('prepare the launch checklist','the product team confirms the date','marketing'),
 ('check the deployment notes','engineering releases the build','engineering'),('revise the testing guide','the maintainer approves the changes','engineering'),('summarize the defect reports','the tracker becomes available','engineering'),('review the integration plan','the partner supplies the interface','engineering'),
 ('prepare the service overview','support releases the case list','support'),('check the escalation queue','the shift supervisor authorizes access','support'),('revise the troubleshooting notes','the specialist confirms the fix','support'),('summarize the outage tickets','the monitoring export arrives','support'),
 ('review the facilities rota','the building manager approves staffing','facilities'),('prepare the equipment log','the technician completes inspection','facilities'),('check the office supplies list','the department approves its request','facilities'),('revise the maintenance schedule','the contractor confirms the visit','facilities'),
 ('summarize the workshop feedback','the organizer releases the responses','events'),('prepare the seminar handout','the presenter approves the content','events'),('check the visitor schedule','reception confirms the appointments','events'),('review the travel bookings','the coordinator clears the itinerary','events'),
 ('revise the document index','the archivist releases the catalogue','records'),('check the records transfer','the custodian authorizes the move','records'),('prepare the retention table','the policy owner signs the revision','records'),('summarize the meeting minutes','the chair approves the record','records')],
 'validation':[
 ('map the clinical equipment locations','the hospital administrator grants clearance','hospital_operations'),('compile the laboratory calibration list','the laboratory head approves the access','laboratory'),('review the museum loan schedule','the curator releases the loan records','museum'),('prepare the library digitization plan','the librarian approves scanning','library'),('check the kitchen allergen guide','the catering lead confirms ingredients','catering'),('summarize the fleet inspection sheets','the transport manager authorizes release','fleet'),('organize the theatre rehearsal timetable','the director confirms availability','theatre'),('revise the farm delivery manifest','the farm operator approves dispatch','agriculture'),('prepare the nursery watering chart','the horticulturist signs the plan','horticulture'),('check the sports venue reservations','the venue officer approves the bookings','sports'),('summarize the repair depot backlog','the depot manager releases the job cards','repair'),('review the radio studio allocations','the producer confirms the recording slots','broadcasting')],
 'test':[
 ('assemble the harbour mooring register','the harbourmaster authorizes the inspection','maritime'),('check the solar array inspection notes','the site engineer permits entry','renewable_energy'),('review the railway platform signage','the station controller clears the work','railway'),('prepare the geological sample inventory','the field lead releases the specimens','geology'),('summarize the observatory instrument bookings','the astronomer approves instrument use','astronomy'),('revise the water treatment sampling plan','the plant chemist validates the protocol','water'),('compile the textile loom service records','the mill supervisor permits the audit','textiles'),('organize the wildlife survey route','the ranger issues the field permit','conservation'),('review the bridge inspection photographs','the civil engineer releases the images','civil_engineering'),('check the bakery production sequence','the production chief confirms the batch order','bakery'),('prepare the recycling depot handover','the depot controller approves the transfer','recycling'),('summarize the wind tunnel experiment schedule','the researcher authorizes the test window','aerodynamics')]
}
# Context templates are allocated wholly by split. All contain placeholders, never label names.
TEMPLATES={
 'train':[
 ('We need someone to {a} {t}. Can we agree that you will be the person doing it?', 'The proposal is that you {a} {t}, but only if {c}. The prerequisite is still pending; without it the work will not go ahead. Do you accept that arrangement?', 'For the records check, the material needed to {a} is in the shared folder. Is that statement accurate?', 'For the availability record, what time and reference material do you currently have? We are gathering facts, not assigning the work to {a}.'),
 ('Can I put you down to {a} {t}? The work is ready to start.', 'Can I put you down to {a} {t}, contingent on {c}? That event has not occurred, so the arrangement depends on it.', 'The office record says we have a request to {a}. Are those facts correct?', 'We are documenting capacity before deciding who will {a}. Describe your available time and material; no owner is being chosen in this conversation.'),
 ('Will you personally {a} {t}? We have everything required.', 'Would you take on the plan to {a} {t} provided {c}? That permission is not yet available, and the plan is off otherwise.', 'Someone asked us to {a}. I am only checking that this is what the note says. Do you agree that the note says that?', 'For a factual desk and calendar check, tell me what is ready and when you are free. This question does not ask you to {a}.'),
 ('I propose that you {a} {t}. Do you agree to be responsible?', 'I propose that you {a} {t} subject to {c}. The decision is unresolved, so that restriction remains part of the proposal. Do you agree?', 'The background files for the request to {a} exist in the archive. Do you confirm that description?', 'Before any assignment to {a}, describe the time and documents you have. I am recording resources only.'),
 ('We have no owner to {a} {t}. Are you accepting the job?', 'We need an owner to {a} {t} if {c}, and that has not been decided. Are you accepting this dependent arrangement?', 'The planning list includes an item to {a}. Am I reporting that correctly?', 'I am doing a schedule survey, not choosing someone to {a}. What preparation and space do you currently have?'),
 ('Please confirm the agreement that you will {a} {t}. Nothing is waiting on authorization.', 'Please confirm the agreement that you will {a} {t} only after {c}. Authorization remains outstanding and the job depends on it.', 'I read that there is a request to {a}. Is my reading of the document correct, without discussing who will do it?', 'For my factual notes, describe your current readiness. This is an inventory question, not a request to {a}.'),
 ('Who can take responsibility to {a} {t}? Is that something you are taking on?', 'Who will take responsibility to {a} {t} on the condition that {c}? We do not have that clearance yet. Is that conditional plan acceptable?', 'The reference card mentions a plan to {a}. Is that an accurate description of the card?', 'I need your calendar status and what is on your desk for the resource census; nobody is being assigned to {a} here.'),
 ('Can we settle on you as the owner to {a} {t}? All prerequisites have already been satisfied.', 'Can we settle on you to {a} {t} in the event that {c}? The event is uncertain and has not occurred, so there is no work without it.', 'Does the task record correctly state that there was a request to {a}? This is a check of the record only.', 'For the capacity spreadsheet, report your available resources, rather than accepting a task to {a}.')],
 'validation':[
 ('Your proposed role would be to {a} {t}. Shall we treat that as your undertaking?', 'Your proposed role would be to {a} {t} once {c}. We are awaiting that prerequisite, and the offer is limited by it. Does that describe what you accept?', 'Do you endorse this factual statement: our archive contains documents relevant to the proposal to {a}?', 'Give me a snapshot of your work area and diary. We will choose who should {a} in another meeting, not in this one.'),
 ('Are we agreed that you are the one who will {a} {t}? The necessary approval is in place.', 'Are we agreed that you would {a} {t} dependent on {c}? Nobody has obtained that prerequisite yet.', 'Our request log contains a line about whether to {a}. Is my account of the log true?', 'I want a factual description of your current capacity before we talk about responsibility to {a}.'),
 ('I would like you to {a} {t}. Are you committing to that?', 'I would like you to {a} {t} if and only if {c}. We still lack that decision; is the constrained proposal what you are accepting?', 'The information sheet refers to a possible need to {a}. Are you confirming the sheet contains that information?', 'Tell me what material you have to hand and how busy you are. This is for a resource audit; it does not appoint you to {a}.'),
 ('May I record your acceptance of the responsibility to {a} {t}? The task is authorized.', 'May I record your acceptance to {a} {t} as long as {c}? That authorization is outstanding; it limits the arrangement.', 'Please verify the truth of the statement that our notes discuss an item to {a}.', 'Describe your preparation and spare time for the scheduling survey. We have deliberately left ownership to {a} undecided today.')],
 'test':[
 ('Shall your name be entered as accountable for the work to {a} {t}? We can proceed immediately.', 'Shall your name be entered to {a} {t} with the proviso that {c}? The proviso is unsatisfied; the arrangement lapses without it.', 'In a fact check, I said that our documents mention a proposal to {a}. Was my statement true?', 'This interview concerns the present contents of your desk and diary. Allocation of the work to {a} is outside this interview. What is your current situation?'),
 ('Do you give your agreement to personally {a} {t}? There are no outstanding permissions.', 'Do you give your agreement to {a} {t} contingent upon {c}? That requirement has not been met and cannot be waived.', 'Does your view concur with the historical fact that the request to {a} was entered in the system?', 'We are surveying current readiness only. The choice of who will {a} will be made later by someone else. Describe what you already have available.'),
 ('I am asking for your undertaking to {a} {t}. Are we of one mind about you doing the work?', 'I am asking whether you accept the limited undertaking to {a} {t}, pending {c}. That is unresolved, and there is no authorization to proceed without it.', 'The question is whether my description of the paperwork about the request to {a} is accurate. Do you support that description?', 'For an observational study, describe your free time and nearby reference materials. This interview offers no assignment to {a}.'),
 ('Would you assent to being responsible to {a} {t}? You can start without waiting for anything.', 'Would you assent to the arrangement to {a} {t} in case {c}? That contingency is still open; failure to satisfy it cancels the task.', 'Please affirm or reject this account of the facts: the task list discusses whether to {a}.', 'We need a descriptive account of your calendar and papers, with responsibility to {a} reserved for a future discussion.')]
}
TIMES={'train':['this morning','this afternoon','tomorrow','before the next briefing','by the end of the week','during the next shift','before Thursday','after the team meeting'],'validation':['before the monthly review','during the upcoming work session','by next Wednesday','over the next two days'],'test':['ahead of the next inspection','during the following duty period','before the project checkpoint','by the next planning call']}
def build_context_training(destination):
 out=Path(destination);out.mkdir(parents=True,exist_ok=True);allrows={};manifest=[]
 for split in ['train','validation','test']:
  rows=[]
  for si,(action,condition,domain) in enumerate(SCENARIOS[split]):
   for fi,(direct,indirect,refusal) in enumerate(REPLIES[split]):
    u,c,fd,fi_ctx=TEMPLATES[split][fi];fmt={'a':action,'c':condition,'t':TIMES[split][fi]}
    contexts={'U':u.format(**fmt),'C':c.format(**fmt),'FD':fd.format(**fmt),'FI':fi_ctx.format(**fmt)}
    gid=f'{split}_S{si:02}_F{fi:02}'
    specs=[('direct','FD',0,direct),('direct','U',3,direct),('direct','C',2,direct),('indirect','FI',0,indirect),('indirect','U',1,indirect),('indirect','C',2,indirect),('refusal','U',0,refusal),('refusal','C',0,refusal)]
    for j,(replykind,ctxkind,label,target) in enumerate(specs):
     ctx='[manager]: '+contexts[ctxkind]
     rationale={0:('Explicit refusal of responsibility.' if replykind=='refusal' else 'A factual confirmation/readiness report, with no task acceptance requested.'),1:'Readiness/preparation answers an ownership request and pragmatically signals uptake without a direct undertaking.',2:'The employee accepts or signals uptake of a proposal whose unresolved prerequisite remains in scope.',3:'A direct affirmative confirmation accepts the contextual proposition that the employee will perform the task.'}[label]
     rows.append({'sample_id':gid+f'_{j}','scenario_id':f'{split}_S{si:02}','family_id':f'{split}_F{fi:02}','contrast_group_id':gid,'target_group_id':gid+'_'+replykind,'reply_kind':replykind,'context_kind':ctxkind,'domain':domain,'split':split,'conversation_context':ctx,'target_speaker':'employee','target_utterance':target,'model_input_text':f'[CONTEXT] {ctx} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target}','commitment_label':LABELS_C[label],'labels':label,'annotation_rationale':rationale,'human_review_required':True})
  random.Random(42).shuffle(rows);allrows[split]=rows
  with (out/(split+'.csv')).open('w',newline='') as f:
   w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
  manifest.extend({k:r[k] for k in ['sample_id','scenario_id','family_id','contrast_group_id','target_group_id','split']} for r in rows)
 audit={'seed':42,'split_method':'authored disjoint reply/context families and scenario domains; fixed before training','counts':{},'cross_split_overlap':{},'csv_sha256':{},'annotation':'agent-authored provisional; no human validation','earlier_challenge_used_for_training':False}
 def norm(x):return ' '.join(re.findall(r'[a-z]+',x.lower()))
 for split,rows in allrows.items():
  assert len({r['model_input_text'] for r in rows})==len(rows)
  groups=collections.defaultdict(list)
  for r in rows:groups[r['target_group_id']].append(r)
  for g in groups.values():
   assert len({r['target_utterance'] for r in g})==1
   if g[0]['reply_kind']!='refusal':assert len(g)==3 and len({r['labels'] for r in g})==3
  def ceiling(field):
   grouped=collections.defaultdict(collections.Counter)
   for r in rows:grouped[r[field]][r['labels']]+=1
   return sum(max(c.values()) for c in grouped.values())/len(rows)
  audit['counts'][split]={'rows':len(rows),'class_counts':dict(collections.Counter(r['commitment_label'] for r in rows)),'families':len(REPLIES[split]),'scenarios':len(SCENARIOS[split]),'context_dependent_triplets':sum(g[0]['reply_kind']!='refusal' for g in groups.values()),'refusal_control_groups':sum(g[0]['reply_kind']=='refusal' for g in groups.values()),'target_only_empirical_ceiling':ceiling('target_utterance'),'context_only_empirical_ceiling':ceiling('conversation_context')}
  audit['csv_sha256'][split]=hashlib.sha256((out/(split+'.csv')).read_bytes()).hexdigest()
 for a,b in itertools.combinations(allrows,2):
  check={}
  for col in ['model_input_text','conversation_context','target_utterance','scenario_id','family_id','contrast_group_id','target_group_id','domain']:
   check[col]=len({norm(r[col]) for r in allrows[a]}&{norm(r[col]) for r in allrows[b]})
   assert check[col]==0,(a,b,col,check[col])
  for name,items in [('target_templates',REPLIES),('context_templates',TEMPLATES)]:
   aa=[norm(t) for p in items[a] for t in p];bb=[norm(t) for p in items[b] for t in p]
   check['max_'+name+'_token_similarity']=max(difflib.SequenceMatcher(None,x.split(),y.split(),autojunk=False).ratio() for x in aa for y in bb)
  audit['cross_split_overlap'][a+'_'+b]=check
 with (out/'split_manifest.csv').open('w',newline='') as f:
  w=csv.DictWriter(f,fieldnames=list(manifest[0]));w.writeheader();w.writerows(manifest)
 (out/'split_audit.json').write_text(json.dumps(audit,indent=2))
 return allrows,audit

"""Balanced context-dependent dataset v2. Fixed authoring before training/testing."""
import csv,json,re,hashlib,random,itertools,collections,difflib
from pathlib import Path
# Load only the previous generator's definitions, not its train/evaluate routines.
# In Colab, REPLIES/SCENARIOS/TEMPLATES are already present from experiment E.
NEW_DIRECT=[
 ('Yes, I am agreeing to the proposal you stated.','affirmative'),('We have a deal on those terms.','affirmative'),('That proposal has my acceptance.','affirmative'),('You may record my agreement.','affirmative'),('I am saying yes to what you proposed.','affirmative'),('That accurately expresses what I agree to.','affirmative'),('I concur with that account.','affirmative'),('It is as you have just described.','affirmative'),
 ("I'll take this responsibility and carry it through.",'undertaking'),('I will make sure it gets done.','undertaking'),('I promise to handle the work you described.','undertaking'),('I am taking ownership of that task.','undertaking'),("I'll be the one who does it.",'undertaking'),('You can rely on me to finish that work.','undertaking'),('I commit to carrying out the requested work.','undertaking'),('I will deliver what you have asked for.','undertaking')]
NEW_INDIRECT=[
 'I have the working notes beside me and a stretch of time available.',
 'My calls are out of the way, and the source documents are already open.',
 'I brought the background papers over and kept an hour aside.',
 'The task material is on my screen, and I moved my other meeting.',
 'I gathered the supporting records and have a quiet period now.',
 'The reference packet is at my desk and my afternoon has room.',
 'I made a space in my workload and pulled together the relevant information.',
 'I already found the source pages, with time left before my next appointment.',
 'The first page is marked up in front of me, and I left the rest of the day clear.',
 'I have begun looking through the material and reserved time to continue.',
 'My initial notes are already on the page, and there is a gap between calls.',
 'I pulled out the documents and started following the outstanding points.',
 'I am at the desk with the records, and the next hour is unbooked.',
 'The preliminary checks are open here and I set aside a work period.',
 'I have been tracing the relevant details, with time put by to continue.',
 'The material is laid out for me and I shifted my other work out of the way.']
NEW_REFUSAL=[
 'I am declining to be responsible for this work.','I cannot give you that undertaking.',
 'That is not something I will own.','My response is a refusal to take the job.',
 'I am not accepting the task you have proposed.','Please leave my name off this assignment.',
 'I will not be accountable for completing this.','I am turning down the request to do that.',
 'No, I am not volunteering for this responsibility.','I need to decline; I will not do the work.',
 'I do not agree to take charge of it.','I am saying no to the assignment.',
 'This responsibility needs to go to someone other than me.','I am not the owner of this and will not become its owner.',
 'I cannot accept your proposal that I do it.','Do not count on me to carry out that task.']
V2_REPLY={'train':[(d,i,r,'affirmative') for d,i,r in REPLIES['train']]+[(d,NEW_INDIRECT[k],NEW_REFUSAL[k],mode) for k,(d,mode) in enumerate(NEW_DIRECT)],
'validation':[
 ('Your proposal is acceptable to me.','I cleared a work slot and placed the background folder nearby.','I must decline the responsibility you are suggesting.','affirmative'),
 ('I acknowledge and accept those terms.','The relevant documents are spread out and I have an interval free.','I am not going to undertake the task on offer.','affirmative'),
 ('I undertake to complete the task you described.','I have made a first pass through the records and kept time to go on.','I am withdrawing myself from consideration for that work.','undertaking'),
 ('I will see that the requested work is finished.','My preparation is on the desk and I have put other work aside.','Please do not treat me as agreeing to this responsibility.','undertaking')],
 'test':[
 ('I give my assent to your proposal.','I have a working copy ready and rearranged the next part of my day.','I am refusing the role you have put forward.','affirmative'),
 ('Those terms are ones I accept.','I am sitting with the collected evidence and have a free work period.','I will not assume the obligation to do this.','affirmative'),
 ('We can regard that proposal as agreed.','The preparatory pages are here and my upcoming appointments have been moved.','The work needs a different person; I am declining to own it.','affirmative'),
 ('I shall take charge of getting that done.','I have the opening checks on the screen and kept a session vacant.','You do not have my acceptance of the assignment.','undertaking'),
 ('I promise to see the assignment through to completion.','The background material has my first annotations and I made time for the next step.','I am not offering to carry this out for you.','undertaking'),
 ('I will personally carry out the work you requested.','I assembled the relevant packet and set my other duties aside for a while.','I decline any responsibility for performing that work.','undertaking')]
}
# New validation/test domains/actions have never appeared in prior model evaluation.
V2_SCENARIOS={'train':SCENARIOS['train'][::2],
 'validation':[
 ('prepare the aquarium feed inventory','the aquarist approves the revised feeding amounts','aquarium'),('check the bookbinding repair register','the conservator releases the damaged volumes','bookbinding'),('compile the planetarium demonstration rota','the programme director confirms the equipment','planetarium'),('review the ceramic kiln schedule','the workshop lead certifies the firing plan','ceramics'),('organize the language centre assessment packs','the coordinator clears the assessment material','language_centre'),('revise the cable car servicing checklist','the operations engineer approves the inspection','cable_car'),('summarize the cinema projection handover','the chief projectionist releases the logs','cinema'),('check the glassworks dispatch sequence','the furnace manager confirms the safe packing order','glassworks')],
 'test':[
 ('compile the seed bank accession register','the collections officer permits use of the accession records','seed_bank'),('review the puppet theatre touring checklist','the touring manager confirms the transport arrangements','puppet_theatre'),('prepare the robotics lab equipment rotation','the lab coordinator authorizes the robot allocation','robotics'),('check the quarry survey handover','the survey supervisor signs the site clearance','quarry'),('summarize the ice rink maintenance record','the rink technician approves the servicing details','ice_rink'),('organize the weaving class material packs','the tutor confirms the thread allocations','weaving_education'),('revise the community garden tool schedule','the garden steward approves the shared equipment plan','community_garden'),('review the sound archive restoration queue','the archive engineer releases the audio masters','sound_archive')]
}
# Eight train context families, four validation, six test. A profile has:
# U1 ordinary request, U2 another unqualified request, U3 fulfilled prerequisite,
# U4 prerequisite removed, C1 pending condition, C2 multi-turn pending condition,
# F affirmative factual confirmation, F indirect resource report, F undertaking quotation.
# Same U/C contexts take direct, indirect and refusal replies with different labels.
V2_CONTEXTS={
 'train':[
 ('Can we agree that you will {a} {t}?','I need you to {a} {t}. Are you taking responsibility?',
 'Earlier I said you could {a} {t} only if {c}. That has now happened, and the condition is satisfied. Will you take the work on?',
 'We considered making the plan to {a} dependent on {c}, but that restriction has been removed. The request now is for you to {a} {t}. Do you accept?',
 'Will you {a} {t} only if {c}? That decision is pending and the task will not proceed without it.',
 '[employee]: Is the work to {a} approved? [manager]: Not yet. You would {a} {t} provided {c}, which is still unresolved. Do you accept that arrangement?',
 'I stated that a request to {a} exists in the document. Is that account correct?',
 'For the capacity survey, describe your available time and the material on your desk. This is a factual check, not an assignment to {a}.',
 'We are rehearsing a line for a training recording, not assigning any work to {a}. Read the sample employee promise aloud; the recording creates no actual obligation.'),
 ('Are you agreeing to own the work to {a} {t}?','May I record your agreement that you will {a} {t}?',
 'The earlier condition was that you would {a} if {c}. We have confirmation that it did occur. We now need you to {a} {t}. Are you agreeing?',
 'Approval is no longer a requirement: we dropped the proposal that {c} must happen first. Will you {a} {t} without that restriction?',
 'The offer is that you {a} {t} on condition that {c}. That has not occurred. Does that restricted offer have your agreement?',
 '[employee]: Can I begin the task to {a}? [manager]: We are still waiting. You may undertake it {t} contingent on {c}; otherwise there is no task. Are you accepting those terms?',
 'Our log records a proposal to {a}. Am I describing the log truthfully?',
 'I am asking for a description of your diary and reference material, not asking you to {a}. What is already available?',
 'For the staff workshop, quote a sample acceptance response from the script. We are only practicing speech, with no real assignment to {a}.'),
 ('Will you personally {a} {t}?','The work to {a} needs an owner {t}. Is that you?',
 'We previously had to wait for {c} before anyone could {a}. That prerequisite has been met. Can we count on you to {a} {t}?',
 'The requirement that {c} has been waived completely. I am asking you to {a} {t}, with nothing left to wait for. Do you agree?',
 'Can you accept responsibility to {a} {t} provided {c}? We are still awaiting that event.',
 '[manager]: We need someone to {a}. [employee]: Is the permission settled? [manager]: No. The plan is that you do it {t} if {c}, and that remains open. Do you agree to that plan?',
 'Is it factually right that our notes mention a need to {a}?',
 'For a calendar and desk inventory, tell me what is open and when you are free. We are not choosing the person to {a} in this exchange.',
 'In this pronunciation exercise, say the fictional employee undertaking. Nobody is actually being asked to {a}; it is a line from an example dialogue.'),
 ('Please confirm you will {a} {t}.','Would you accept the assignment to {a} {t}?',
 'The condition for the work to {a} was that {c}. It is now fulfilled, and the work is ready. Do you agree to {a} {t}?',
 'Our first plan tied the work to {c}, but we deleted that dependency. Are you now agreeing to {a} {t} as an ordinary assignment?',
 'Please confirm the conditional plan that you {a} {t} if {c}. We do not have that clearance yet.',
 '[employee]: Do we have clearance to {a}? [manager]: It is outstanding. I propose that you {a} {t} subject to {c}. Are those terms acceptable?',
 'I read that the task list contains an item to {a}. Do you agree that my reading is accurate?',
 'Before ownership of the task to {a} is discussed, report the resources you have at hand. I am collecting facts only.',
 'This is a simulated role-play. Deliver the employee acceptance line for the imaginary task to {a}; the exercise does not give you that real responsibility.'),
 ('Do you accept the proposal that you {a} {t}?','Shall I put your name against the work to {a} {t}?',
 'We were waiting to {a} until {c}. The wait is over: that event has taken place. Will you {a} {t}?',
 'We decided not to require {c} after all. Can I record your undertaking to {a} {t}, with that prerequisite removed?',
 'Shall I put you down to {a} {t} in the event that {c}? The prerequisite remains unsatisfied.',
 '[manager]: There is a proposal to {a}. [employee]: Is it definite? [manager]: It will happen {t} only if {c}, which is uncertain. Will you accept the proposal on that basis?',
 'The planning document refers to a request to {a}. Does your understanding agree with that factual statement?',
 'This is a readiness census rather than a request to {a}. Describe your current preparation and free time.',
 'Read a promise as a quoted example for our communications class. It concerns an imaginary task to {a}, and reading it is not accepting that task.'),
 ('Can you confirm your undertaking to {a} {t}?','Are you the person agreeing to {a} {t}?',
 'If {c}, we said the work to {a} could go ahead. It did happen; the requirement is complete. Are you taking the job {t}?',
 'The proposal originally depended on {c}; management has now cancelled that dependency. Are you agreeing to {a} {t}?',
 'Are you accepting the undertaking to {a} {t} dependent on {c}? We are waiting for that confirmation.',
 '[employee]: Is {c} confirmed? [manager]: No. Your proposed responsibility to {a} {t} would apply only after that happens. Are you taking on that limited arrangement?',
 'For a fact check, I said a request to {a} was registered. Is that description one you agree with?',
 'We are recording your existing work setup and calendar. No agreement to {a} is being sought in this question; describe what is there.',
 'During the microphone test, speak an example of an employee undertaking. The line is only scripted audio, not your actual acceptance of work to {a}.'),
 ('May we settle on you doing the task to {a} {t}?','I propose you take charge and {a} {t}. Do you agree?',
 'The earlier restriction said we could {a} only once {c}. That event is now verified. Will you handle the job {t}?',
 'There is no longer any dependency on {c}. The earlier condition was withdrawn. Will you {a} {t}?',
 'I propose you {a} {t} as long as {c}. That remains pending; there will be no task if it fails.',
 '[manager]: Could you {a}? [employee]: Is approval available? [manager]: It is not. The arrangement would be for you to do it {t} once {c}. Does that describe what you accept?',
 'Please confirm whether our folder contains information about a proposal to {a}.',
 'For the workload notes, describe the time and materials currently available. We are not assigning you the work to {a}.',
 'Repeat a fictional promise for the acting exercise. This is not a live discussion of who will {a}; the words belong to the sample character.'),
 ('Are we agreed that you will {a} {t}?','Will you accept responsibility for the request to {a} {t}?',
 'The request to {a} was subject to {c}. That has been completed, so the restriction no longer blocks it. Do you accept the work {t}?',
 'We removed the rule that {c} must come first. Are you agreeing to {a} {t} without that qualification?',
 'Would you agree to {a} {t} only after {c}? That prerequisite is not yet in place.',
 '[employee]: Is the plan to {a} approved? [manager]: It is a conditional proposal: you would do it {t} if {c}. Nothing has been decided about that yet. Is that proposal acceptable?',
 'I am checking the record, not allocating the task: does the record correctly say we have a request to {a}?',
 'For a factual resource report, tell me your current situation with time and documents. The decision about who will {a} will happen elsewhere.',
 'Please voice the sample employee promise in the rehearsal script. It has no effect on real responsibility to {a}; we only need the line for practice.')],
 'validation':[
 ('Is your answer an acceptance of the role to {a} {t}?','Can the plan name you as the person who will {a} {t}?',
 'The trigger we had specified, namely {c}, has already occurred. The job to {a} can now proceed {t}; will you accept it?',
 'We abandoned the trigger involving {c}. Do you agree to {a} {t} now that no trigger limits the request?',
 'Would you accept the role to {a} {t}, pending {c}? The trigger has not occurred.',
 '[employee]: Is everything settled for the plan to {a}? [manager]: We still await the trigger: {c}. The role would apply {t} only on that basis. Do you accept?',
 'Do you support this account of the records: they discuss a proposal to {a}?',
 'In our descriptive survey, give your present diary and document status. Today we are not offering you the job to {a}.',
 'For a narrated example, say the character acceptance line. This narration does not establish an obligation for you to {a}.'),
 ('Would you endorse the proposal that you will {a} {t}?','Do you consent to being the owner who will {a} {t}?',
 'We were prohibited from starting until {c}; that barrier has been cleared. Will you {a} {t}?',
 'A prior proposal required {c}. The requirement has been struck out. Would you endorse the current request to {a} {t}?',
 'Would you endorse taking the work to {a} {t} contingent upon {c}? That barrier remains uncleared.',
 '[manager]: We propose that you {a}. [employee]: Is the barrier cleared? [manager]: No, it still depends on {c}. The proposed work {t} stays conditional. Do you endorse it?',
 'I am verifying a historical record about whether a request to {a} was made. Do you concur with that record?',
 'Describe what you already have available for the schedule review. We are documenting capacity, with ownership to {a} reserved for a different meeting.',
 'We are recording an instructional quotation rather than allocating work. Speak the example undertaking; it is not a real agreement that you will {a}.'),
 ('Will you take the offered role and {a} {t}?','Is it settled that you are volunteering to {a} {t}?',
 'There was an outstanding requirement that {c}. That requirement has been fulfilled. Will you volunteer to {a} {t}?',
 'The restriction involving {c} was discarded. Is it settled that you will {a} {t} under the revised unrestricted plan?',
 'The offered role to {a} {t} applies exclusively if {c}. We have no resolution on that. Will you accept the offer?',
 '[employee]: Are we able to {a} yet? [manager]: That is unresolved because {c} has not happened. The role would apply {t} subject to it. Are you volunteering on those terms?',
 'The paper states that an item to {a} is under discussion. Is that factual account acceptable?',
 'We are making observations about your readiness, not asking you to {a}. Describe your existing timetable and materials.',
 'This turn is part of a fictional scene. Read the acceptance sentence from the scene; it creates no live commitment to {a}.'),
 ('Can your acceptance to {a} {t} be recorded?','Are you undertaking the work to {a} {t}?',
 'The approval represented by {c} has been obtained. There is no remaining contingency before the work to {a}. Will you do it {t}?',
 'We rescinded the approval requirement concerning {c}. Will you accept the assignment to {a} {t} as it now stands?',
 'Can we record your acceptance to {a} {t} contingent on {c}, which remains outstanding?',
 '[employee]: Do we have a final decision to {a}? [manager]: Not a final one: we await {c}. You would take it on {t} only under that restriction. Do you accept?',
 'Our notes describe a potential request to {a}. Are you affirming that this is what the notes say?',
 'For a descriptive interview, report your capacity and available documents. This exchange does not choose you to {a}.',
 'The assignment to {a} is imaginary in this script. Read the character promise for the demonstration, without accepting any real duty.')],
 'test':[
 ('Will your agreement put you in charge of the task to {a} {t}?','Do you accept being accountable for carrying out the work to {a} {t}?',
 'We had tied the work to {c}. That is now a confirmed fact, so the prerequisite is discharged. Are you accepting responsibility to {a} {t}?',
 'The dependency on {c} was formally dropped. There is no longer any such restriction. Are you accepting responsibility to {a} {t}?',
 'Will you accept the contingent responsibility to {a} {t} on the proviso that {c}? The proviso is still unmet.',
 '[employee]: Has the decision to {a} been made? [manager]: Not conclusively. Your role {t} would arise only upon {c}, which is undecided. Is that the arrangement you accept?',
 'My statement was that the recorded facts include a proposal to {a}. Is that a statement you would affirm?',
 'This is an observational account of your present resources. Tell me about your diary and papers; assigning someone to {a} is not the purpose of this conversation.',
 'A fictional colleague has an acceptance line in this voice recording. Perform that line as the character; it does not assign you any real work to {a}.'),
 ('Do we have your assent to doing the work to {a} {t}?','Will you be responsible for making sure we {a} {t}?',
 'The prerequisite, {c}, has been fully accomplished. We are past the earlier condition. Do you assent to the actual job to {a} {t}?',
 'We rescinded the stipulation involving {c}. Do we have your assent to {a} {t} under the request as revised?',
 'Do we have your assent to {a} {t} should {c}? That prerequisite remains unfulfilled.',
 '[manager]: We need a person to {a}. [employee]: Has the prerequisite been fulfilled? [manager]: No. Your undertaking {t} would depend on {c}. Are you assenting to that conditional proposal?',
 'Can you affirm the historical description that a proposal to {a} was put in our notes?',
 'Describe what your current preparation and unoccupied time look like. This is solely an observation exercise; the choice of who will {a} is deferred.',
 'For the speech exercise, utter the fictional worker promise as quoted dialogue. The exercise conveys no actual responsibility to {a}.'),
 ('Would you agree that you are going to {a} {t}?','Can I take your answer as agreeing to the assignment to {a} {t}?',
 'The contingency about {c} is now settled positively. With it satisfied, will you accept the request to {a} {t}?',
 'The earlier plan linked your task to {c}. That link has been cancelled outright. Will you agree to {a} {t} on the new unrestricted basis?',
 'Would you agree to the task to {a} {t} solely in the eventuality that {c}? We do not yet know whether that will happen.',
 '[employee]: Is the task to {a} definite? [manager]: No; the eventuality that {c} remains open. It is a limited proposal for you to act {t} only if it happens. Are you agreeing to that?',
 'Does your position support this factual assertion: the archive refers to a proposal to {a}?',
 'For an account of present circumstances, tell me about your time and reference material. We are not seeking an undertaking to {a} from you in this interview.',
 'Take the fictional role for this rehearsal and deliver the sample promise. The scene is invented, with no actual assignment to {a}.'),
 ('Are you giving your undertaking that you will {a} {t}?','Shall the responsibility to {a} {t} be yours?',
 'We have received definite confirmation that {c}. The previously conditional work is therefore authorized. Are you undertaking to {a} {t}?',
 'The proposal no longer includes the restriction that {c}. It was withdrawn, leaving the assignment unrestricted. Are you undertaking to {a} {t}?',
 'Are you giving an undertaking to {a} {t} subject to the unresolved matter that {c}? Without it, there is no assignment.',
 '[employee]: Have we resolved authorization to {a}? [manager]: Authorization awaits {c}; no resolution has been reached. Are you accepting the restricted role for {t}?',
 'We are checking a statement about paperwork, not an undertaking: the paperwork lists a request to {a}. Is my statement supported by the facts?',
 'Describe the documents at hand and time available for a record of your current situation. Any request to {a} belongs to a subsequent conversation.',
 'Please render the character undertaking in this staged reading. These are fictional words and do not commit you to the actual work to {a}.'),
 ('Do you promise to be the one who will {a} {t}?','Will you own the requested outcome and {a} {t}?',
 'It was necessary that {c} before this job could start. We now have that confirmation. Do you accept the job to {a} {t}?',
 'There was once a necessity for {c}, but we have abolished it. Are you agreeing to {a} {t} without waiting for that event?',
 'Do you accept the job to {a} {t} conditional upon {c}? The necessary event has not taken place.',
 '[manager]: The request is for you to {a}. [employee]: Can it proceed? [manager]: Only upon {c}, and we have not reached that point. Will you accept that dependent assignment for {t}?',
 'Is the factual proposition that our records contain an item to {a} something you affirm?',
 'We are describing existing capacity, not allocating an outcome. Tell me what is ready and what time you have; nobody is being appointed to {a} here.',
 'The example speaker in this training script promises to do the job. Voice the speaker line without taking on the real task to {a}.'),
 ('Are you agreeing that the responsibility to {a} {t} rests with you?','Do you undertake to {a} {t} as requested?',
 'The gate requiring {c} has been passed successfully. The request is now active. Will you take responsibility to {a} {t}?',
 'We removed the gate associated with {c}; it does not apply to the work anymore. Will you take responsibility to {a} {t}?',
 'Are you agreeing to the gated arrangement to {a} {t}, applicable only if {c}? That gate is still closed.',
 '[employee]: Are we past the gate for the work to {a}? [manager]: No. It remains tied to {c}, which has not occurred. Are you accepting the limited undertaking for {t}?',
 'I am asking you to support or reject a factual account of the task register. It says a proposal to {a} was entered. Do you support that account?',
 'Report your current calendar and working papers as factual information. The responsibility to {a} is outside the scope of this descriptive question.',
 'We are demonstrating a hypothetical employee response. Say the scripted undertaking as that employee; there is no real transfer of responsibility to {a}.')]
}
V2_TIME={'train':['before the team update','over the next working day','by the next status meeting','during the coming afternoon','before Friday lunch','during the following morning','ahead of the handover','by the end of the current shift'], 'validation':['before the next coordination session','over the coming work interval','ahead of the weekly checkpoint','during the scheduled preparation window'],'test':['before the next operational briefing','during the upcoming duty window','ahead of the following review session','within the next scheduled work period','before the next allocation meeting','during the forthcoming project slot']}
def build_balanced_v2(destination):
 out=Path(destination);out.mkdir(parents=True,exist_ok=True);parts={};manifest=[]
 for split in ['train','validation','test']:
  rows=[]
  for si,(action,condition,domain) in enumerate(V2_SCENARIOS[split]):
   for ri,(direct,indirect,refusal,mode) in enumerate(V2_REPLY[split]):
    ci=ri%len(V2_CONTEXTS[split]);templates=V2_CONTEXTS[split][ci];fmt={'a':action,'c':condition,'t':V2_TIME[split][ci]}
    contexts={key:template.format(**fmt) for key,template in zip(['U1','U2','U3','U4','C1','C2','FD','FI','FQ'],templates)}
    gid=f'B2_{split}_S{si:02}_R{ri:02}'
    specs=[]
    for kind,target,fact,unconditional_label in [('direct',direct,'FQ' if mode=='undertaking' else 'FD',3),('indirect',indirect,'FI',1)]:
     specs.append((kind,target,fact,0))
     specs.extend((kind,target,u,unconditional_label) for u in ['U1','U2','U3','U4'])
     specs.extend((kind,target,c,2) for c in ['C1','C2'])
    specs.extend(('refusal',refusal,c,0) for c in ['U1','C1'])
    for j,(kind,target,ctxkey,label) in enumerate(specs):
     context='[manager]: '+contexts[ctxkey]
     rationale=('The employee explicitly declines responsibility.' if kind=='refusal' else 'The reply is a factual confirmation/report or scripted quotation, not a live acceptance.' if label==0 else 'Readiness or preparation offered in answer to an ownership request signals uptake without directly undertaking the work.' if label==1 else 'Acceptance or pragmatic uptake inherits the still-unresolved condition of the proposal.' if label==2 else 'The employee directly affirms or undertakes the future task; any earlier prerequisite is fulfilled or withdrawn.')
     rows.append({'sample_id':gid+f'_{j:02}','scenario_id':f'B2_{split}_S{si:02}','reply_family_id':f'B2_{split}_R{ri:02}','context_family_id':f'B2_{split}_C{ci:02}','contrast_group_id':gid,'target_group_id':gid+'_'+kind,'reply_kind':kind,'direct_mode':mode,'context_kind':ctxkey,'domain':domain,'split':split,'conversation_context':context,'target_speaker':'employee','target_utterance':target,'model_input_text':f'[CONTEXT] {context} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target}','commitment_label':LABELS_C[label],'labels':label,'annotation_rationale':rationale,'human_review_required':True})
  random.Random(2026).shuffle(rows);parts[split]=rows
  with (out/(split+'.csv')).open('w',newline='') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
  manifest.extend({k:r[k] for k in ['sample_id','scenario_id','reply_family_id','context_family_id','contrast_group_id','target_group_id','split']} for r in rows)
 audit={'version':'balanced_context_v2','allocation':'preassigned reply/context/scenario families; no random row split','seed':2026,'counts':{},'overlap':{},'csv_sha256':{},'annotation':'agent-authored provisional; requires human review'}
 def norm(t):return ' '.join(re.findall(r'[a-z]+',t.lower()))
 for split,rows in parts.items():
  counts=collections.Counter(r['labels'] for r in rows);assert len(set(counts.values()))==1
  assert len({r['model_input_text'] for r in rows})==len(rows)
  def ceiling(col):
   cs=collections.defaultdict(collections.Counter)
   for r in rows:cs[r[col]][r['labels']]+=1
   return sum(max(c.values()) for c in cs.values())/len(rows)
  groups=collections.defaultdict(list)
  for r in rows:groups[r['target_group_id']].append(r)
  for g in groups.values():
   assert len({r['target_utterance'] for r in g})==1
   if g[0]['reply_kind']!='refusal':assert len(g)==7 and len({r['labels'] for r in g})==3
  audit['counts'][split]={'rows':len(rows),'class_counts':dict(counts),'reply_families':len(V2_REPLY[split]),'context_families':len(V2_CONTEXTS[split]),'domains_scenarios':len(V2_SCENARIOS[split]),'dependent_groups':sum(g[0]['reply_kind']!='refusal' for g in groups.values()),'target_only_ceiling':ceiling('target_utterance'),'context_only_ceiling':ceiling('conversation_context')}
  audit['csv_sha256'][split]=hashlib.sha256((out/(split+'.csv')).read_bytes()).hexdigest()
 for a,b in itertools.combinations(parts,2):
  overlaps={}
  for col in ['model_input_text','conversation_context','target_utterance','scenario_id','reply_family_id','context_family_id','contrast_group_id','target_group_id','domain']:
   overlaps[col]=len({norm(r[col]) for r in parts[a]}&{norm(r[col]) for r in parts[b]});assert overlaps[col]==0,(a,b,col)
  for kind,source in [('target',V2_REPLY),('context',V2_CONTEXTS)]:
   aa=[norm(t) for tpl in source[a] for t in (tpl[:3] if kind=='target' else tpl)];bb=[norm(t) for tpl in source[b] for t in (tpl[:3] if kind=='target' else tpl)]
   overlaps['max_'+kind+'_template_similarity']=max(difflib.SequenceMatcher(None,x.split(),y.split(),autojunk=False).ratio() for x in aa for y in bb)
  audit['overlap'][a+'_'+b]=overlaps
 with (out/'split_manifest.csv').open('w',newline='') as f:w=csv.DictWriter(f,fieldnames=list(manifest[0]));w.writeheader();w.writerows(manifest)
 (out/'split_audit.json').write_text(json.dumps(audit,indent=2))
 return parts,audit

# EXPERIMENT F: fixed balanced v2 partitions, frozen before training.
BROOT=Path('/content/latent_commitment_balanced_v2'); BRES=BROOT/'results'; BSPLIT=BROOT/'splits';BRES.mkdir(parents=True,exist_ok=True)
b_rows,b_audit=build_balanced_v2(BSPLIT)
b_expected={'train': '6a9681330c8f15a6abb5155ab6819d719ed450fbb03b7147cf618a8540fec9b4', 'validation': 'b5cc2b89696d189942dbc67154101af2dcbc2d22ba16ee29dff97958db2a85c6', 'test': '329e8d33eda138dbb1fe67cee929859f05c16035f1400fd9772cfd166fd2b729'}
assert b_audit['csv_sha256']==b_expected
b_data={s:pd.DataFrame(rs) for s,rs in b_rows.items()}
# Fresh final test has no exact targets, contexts or inputs from any previous experiment's data.
prior_frames=list(experiment_data.values())+list(c_data.values())+[challenge]
prior_all=pd.concat(prior_frames,ignore_index=True)
b_audit['fresh_test_overlap_with_prior_experiments']={col:int(b_data['test'][col].isin(prior_all[col]).sum()) for col in ['model_input_text','conversation_context','target_utterance']}
assert all(v==0 for v in b_audit['fresh_test_overlap_with_prior_experiments'].values())
for values in b_audit['overlap'].values():
    assert values['max_target_template_similarity']<0.8 and values['max_context_template_similarity']<0.8
(BSPLIT/'split_audit.json').write_text(json.dumps(b_audit,indent=2))
def b_dataset(split):
    frame=b_data[split];enc=tokenizer(frame.model_input_text.tolist(),truncation=True,max_length=256);enc['labels']=frame.labels.tolist();return Dataset.from_dict(enc)
b_token_audit={}
for split,df in b_data.items():
    lengths=list(map(len,tokenizer(df.model_input_text.tolist(),truncation=False)['input_ids']))
    b_token_audit[split]={'rows':len(lengths),'max_tokens':max(lengths),'truncated':sum(x>256 for x in lengths)}
    assert max(lengths)<=256
(BRES/'tokenization_audit.json').write_text(json.dumps(b_token_audit,indent=2))
b_train_ds=b_dataset('train');b_val_ds=b_dataset('validation')
b_baselines={};b_baseline_rows=[]
for col in ['target_utterance','conversation_context','model_input_text']:
    baseline=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=2000,random_state=2026))
    baseline.fit(b_data['train'][col],b_data['train'].labels);b_baselines[col]=baseline
    b_baseline_rows.append({'model':'TF-IDF + LR','input':col,'split':'validation',**metrics(b_data['validation'].labels,baseline.predict(b_data['validation'][col]))})
print('BALANCED V2 DATA FROZEN',json.dumps(b_audit,indent=2),flush=True)
print('VALIDATION BASELINES',pd.DataFrame(b_baseline_rows).to_string(index=False),flush=True)


In [ ]:
# BALANCED V2 — fresh pretrained weights; same hyperparameters, never reuse previous weights
import gc, time, shutil
assert torch.cuda.is_available()
for name in ['c_trainer','c_model','c_reloaded','target_trainer','target_model','target_reloaded','experiment_trainer','experiment_model','experiment_reloaded']:
    if name in globals():del globals()[name]
gc.collect();torch.cuda.empty_cache();set_seed(2026)
class ExperimentProgress(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        (BRES/'trainer_log.json').write_text(json.dumps(state.log_history,indent=2))
        if logs:print('BALANCED V2',state.global_step,json.dumps(logs),flush=True)
    def on_save(self,args,state,control,**kwargs):
        print('BALANCED V2 CHECKPOINT:',state.global_step,'best validation macro-F1:',state.best_metric,flush=True)
b_model=AutoModelForSequenceClassification.from_pretrained('microsoft/deberta-v3-base',num_labels=4,id2label=id2label,label2id=label2id)
b_args=TrainingArguments(output_dir=str(BROOT/'checkpoints'),num_train_epochs=3,learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=0.01,warmup_ratio=0.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=2,logging_steps=50,seed=2026,data_seed=2026,report_to='none',dataloader_num_workers=2,save_safetensors=True,disable_tqdm=True)
b_trainer=Trainer(model=b_model,args=b_args,train_dataset=b_train_ds,eval_dataset=b_val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer,pad_to_multiple_of=8),compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2),ExperimentProgress()])
b_start=time.time();b_train_output=b_trainer.train();b_seconds=time.time()-b_start
BMODEL=BROOT/'best_model';b_trainer.save_model(str(BMODEL));tokenizer.save_pretrained(str(BMODEL))
b_epoch_logs=list(b_trainer.state.log_history)
pd.DataFrame([r for r in b_epoch_logs if 'loss' in r]).to_csv(BRES/'training_metrics.csv',index=False)
pd.DataFrame([r for r in b_epoch_logs if 'eval_loss' in r]).to_csv(BRES/'validation_metrics.csv',index=False)
b_info={'model_initialization':'fresh microsoft/deberta-v3-base; no previous fine-tuned weights','gpu':torch.cuda.get_device_name(0),'training_seconds':b_seconds,'epochs_completed':b_trainer.state.epoch,'best_checkpoint':b_trainer.state.best_model_checkpoint,'best_validation_macro_f1':b_trainer.state.best_metric,'global_step':b_trainer.state.global_step,'seed':2026,'split_csv_sha256':b_audit['csv_sha256'],'max_length':256,'balanced_context_v2':True,'test_used_for_selection':False}
assert np.isfinite(b_info['best_validation_macro_f1'])
(BRES/'run_info.json').write_text(json.dumps(b_info,indent=2))
print('BALANCED V2 TRAINING COMPLETED',json.dumps(b_info,indent=2))


In [ ]:
# EXPERIMENT F — one final test pass after validation-only checkpoint selection.
assert not (BRES/'heldout_predictions.npz').exists(),'Use cached test predictions; do not reselect based on test.'
b_test_out=b_trainer.predict(b_dataset('test'),metric_key_prefix='test')
b_logits=b_test_out.predictions;b_y=b_test_out.label_ids;b_pred=b_logits.argmax(1)
np.savez_compressed(BRES/'heldout_predictions.npz',logits=b_logits,labels=b_y)
b_frame=b_data['test'].reset_index(drop=True).copy();assert np.array_equal(b_y,b_frame.labels.to_numpy())
def b_group_metrics(frame,pred,tag):
    pairs=[];groups=[];triplets=[]
    for gid,g in frame.groupby('target_group_id',sort=False):
        if g.reply_kind.iloc[0]=='refusal':continue
        idx=g.index.to_numpy();assert len(idx)==7 and g.target_utterance.nunique()==1 and g.labels.nunique()==3
        groups.append({'model':tag,'target_group_id':gid,'all_seven_correct':bool((pred[idx]==frame.labels.to_numpy()[idx]).all()),'reply_kind':g.reply_kind.iloc[0]})
        representative=g[g.context_kind.isin(['FD','FI','FQ','U1','C1'])].index.to_numpy();assert len(representative)==3
        triplets.append({'model':tag,'target_group_id':gid,'all_three_correct':bool((pred[representative]==frame.labels.to_numpy()[representative]).all())})
        for a,b in itertools.combinations(idx,2):
            if frame.labels.iloc[a]==frame.labels.iloc[b]:continue
            pairs.append({'model':tag,'pair_id':frame.sample_id.iloc[a]+'__'+frame.sample_id.iloc[b],'target_group_id':gid,'both_correct':bool(pred[a]==frame.labels.iloc[a] and pred[b]==frame.labels.iloc[b]),'prediction_changes':bool(pred[a]!=pred[b])})
    mask=frame.reply_kind.ne('refusal').to_numpy()
    summary={'model':tag,**metrics(frame.labels,pred),'correct_examples':int((pred==frame.labels.to_numpy()).sum()),'n_examples':len(frame),'context_dependent_accuracy':float((pred[mask]==frame.labels.to_numpy()[mask]).mean()),'n_context_dependent_examples':int(mask.sum()),'n_pairs':len(pairs),'pairs_both_correct':sum(p['both_correct'] for p in pairs),'pair_exact_accuracy':float(np.mean([p['both_correct'] for p in pairs])),'pair_prediction_change_rate':float(np.mean([p['prediction_changes'] for p in pairs])),'n_groups':len(groups),'groups_all_seven_correct':sum(p['all_seven_correct'] for p in groups),'all_seven_accuracy':float(np.mean([p['all_seven_correct'] for p in groups])),'representative_triplets_all_correct':sum(p['all_three_correct'] for p in triplets),'representative_triplet_accuracy':float(np.mean([p['all_three_correct'] for p in triplets]))}
    return summary,pairs,groups,triplets
b_test_metrics,b_pairs,b_groups,b_triplets=b_group_metrics(b_frame,b_pred,'Balanced v2 DeBERTa')
b_report=classification_report(b_y,b_pred,labels=range(4),target_names=NAMES,zero_division=0,output_dict=True)
pd.DataFrame(b_report).T.to_csv(BRES/'classification_report.csv')
b_cm=confusion_matrix(b_y,b_pred,labels=range(4));pd.DataFrame(b_cm,index=NAMES,columns=NAMES).to_csv(BRES/'confusion_matrix.csv')
pd.DataFrame([b_test_metrics]).to_csv(BRES/'test_metrics.csv',index=False)
b_predictions=b_frame.copy();b_predictions['predicted_label']=[id2label[int(x)] for x in b_pred];b_predictions['correct']=b_pred==b_y
b_prob=a_probs(b_logits)
for k,label in id2label.items():b_predictions['prob_'+label]=b_prob[:,k]
b_predictions.to_csv(BRES/'test_predictions.csv',index=False)
b_comparisons=[b_test_metrics]
for col,baseline in b_baselines.items():
    pred=baseline.predict(b_frame[col]);summary,prs,grs,trs=b_group_metrics(b_frame,pred,'TF-IDF '+col);b_comparisons.append(summary);b_pairs.extend(prs);b_groups.extend(grs);b_triplets.extend(trs)
    b_baseline_rows.append({'model':'TF-IDF + LR','input':col,'split':'test',**metrics(b_y,pred)})
# Verify saved model on validation only, before same-test model comparison.
b_probe=b_data['validation'].model_input_text.iloc[:16].tolist();b_before=d_infer(b_probe,b_model.eval(),tokenizer)
b_trainer.model.to('cpu');gc.collect();torch.cuda.empty_cache()
b_reloaded=AutoModelForSequenceClassification.from_pretrained(BMODEL).to('cuda').eval();b_tok=AutoTokenizer.from_pretrained(BMODEL,use_fast=False)
b_after=d_infer(b_probe,b_reloaded,b_tok);assert np.allclose(b_before,b_after,atol=0.002,rtol=0.002)
b_reloaded.to('cpu');del b_reloaded;gc.collect();torch.cuda.empty_cache()
# The previous84.90% model is frozen; evaluate it on this fresh test for a fair comparison.
b_old=AutoModelForSequenceClassification.from_pretrained('/content/latent_commitment_context_training_v1/best_model').to('cuda').eval()
b_old_tok=AutoTokenizer.from_pretrained('/content/latent_commitment_context_training_v1/best_model',use_fast=False)
b_old_logits=d_infer(b_frame.model_input_text.tolist(),b_old,b_old_tok);b_old_pred=b_old_logits.argmax(1)
np.savez_compressed(BRES/'previous_model_logits.npz',logits=b_old_logits,labels=b_y)
b_old_metrics,prs,grs,trs=b_group_metrics(b_frame,b_old_pred,'Previous84.90% DeBERTa');b_comparisons.append(b_old_metrics);b_pairs.extend(prs);b_groups.extend(grs);b_triplets.extend(trs)
b_old_report=classification_report(b_y,b_old_pred,labels=range(4),target_names=NAMES,zero_division=0,output_dict=True)
pd.DataFrame(b_old_report).T.to_csv(BRES/'previous_model_classification_report.csv')
del b_old;gc.collect();torch.cuda.empty_cache()
pd.DataFrame(b_comparisons).to_csv(BRES/'same_test_comparison.csv',index=False)
pd.DataFrame(b_baseline_rows).to_csv(BRES/'baseline_comparison.csv',index=False)
pd.DataFrame(b_pairs).to_csv(BRES/'pair_results.csv',index=False);pd.DataFrame(b_groups).to_csv(BRES/'all_context_group_results.csv',index=False);pd.DataFrame(b_triplets).to_csv(BRES/'representative_triplet_results.csv',index=False)
b_subgroups=[]
for col in ['reply_family_id','context_family_id','domain','reply_kind','context_kind','direct_mode']:
    for key,g in b_predictions.groupby(col):b_subgroups.append({'grouping':col,'group':key,'n':len(g),'accuracy':float(g.correct.mean())})
pd.DataFrame(b_subgroups).to_csv(BRES/'subgroup_results.csv',index=False)
b_info.update({'test_metrics':b_test_metrics,'test_predicted_once_after_selection':True,'reload_verified_on_validation':True,'selected_weights_sha256':hashlib.sha256((BMODEL/'model.safetensors').read_bytes()).hexdigest(),'previous_model_same_test_metrics':b_old_metrics})
(BRES/'run_info.json').write_text(json.dumps(b_info,indent=2))
print('BALANCED V2 HELD-OUT TEST:',json.dumps(b_test_metrics,indent=2),flush=True)
print('PREVIOUS MODEL ON SAME FRESH TEST:',json.dumps(b_old_metrics,indent=2),flush=True)
print('NEW PER CLASS:',json.dumps({n:b_report[n] for n in NAMES},indent=2),flush=True)
print('RELOAD VERIFIED. No changes or tuning after test evaluation.',flush=True)


In [ ]:
# Preserve the selected model, balanced dataset, comparisons and project inference helper.
from google.colab import _message
import zipfile,shutil,matplotlib.pyplot as plt,seaborn as sns
for key,title,name in [('loss','Training loss','training_loss.png'),('eval_loss','Validation loss','validation_loss.png'),('eval_macro_f1','Validation macro-F1','validation_macro_f1.png')]:
    selected=[r for r in b_epoch_logs if key in r]
    fig,ax=plt.subplots(figsize=(7,4));ax.plot([r['epoch'] for r in selected],[r[key] for r in selected],marker='o');ax.set(xlabel='Epoch',ylabel=key,title=title+' — balanced v2');fig.tight_layout();fig.savefig(BRES/name,dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6));sns.heatmap(b_cm,annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES,ax=ax);ax.set(xlabel='Predicted',ylabel='True',title='Balanced v2 — fresh held-out test');plt.xticks(rotation=25,ha='right');fig.tight_layout();fig.savefig(BRES/'confusion_matrix.png',dpi=160);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4));ax.bar(NAMES,[b_report[n]['f1-score'] for n in NAMES]);ax.set_ylim(0,1);ax.set_ylabel('F1');ax.set_title('Balanced v2 — per-class F1');plt.xticks(rotation=20,ha='right');fig.tight_layout();fig.savefig(BRES/'per_class_f1.png',dpi=160);plt.close(fig)
b_predictions[~b_predictions.correct].to_csv(BRES/'misclassified_examples.csv',index=False)
(BMODEL/'label_mappings.json').write_text(json.dumps({'label2id':label2id,'id2label':id2label,'display_names':NAMES},indent=2))
(BMODEL/'project_predict.py').write_text('"""Load the exported balanced-v2 model for a project demonstration."""\nfrom pathlib import Path\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\n\nclass CommitmentPredictor:\n    def __init__(self, model_directory, device=None):\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.tokenizer = AutoTokenizer.from_pretrained(Path(model_directory), use_fast=False)\n        self.model = AutoModelForSequenceClassification.from_pretrained(Path(model_directory)).to(self.device).eval()\n\n    @torch.inference_mode()\n    def predict(self, context, target_utterance, target_speaker=\'employee\'):\n        text = f\'[CONTEXT] {context} [TARGET SPEAKER] {target_speaker} [TARGET UTTERANCE] {target_utterance}\'\n        tokens = self.tokenizer(text, truncation=False, return_tensors=\'pt\')\n        if tokens[\'input_ids\'].shape[1] > 256:\n            raise ValueError(\'Input exceeds the 256-token training limit; shorten the conversation context.\')\n        probabilities = self.model(**{k:v.to(self.device) for k,v in tokens.items()}).logits.softmax(-1)[0].cpu().tolist()\n        category = max(range(len(probabilities)), key=probabilities.__getitem__)\n        return {\'category\':category, \'label\':self.model.config.id2label[category], \'confidence\':probabilities[category], \'probabilities\':{self.model.config.id2label[i]:p for i,p in enumerate(probabilities)}}\n')
(BRES/'project_predict.py').write_text('"""Load the exported balanced-v2 model for a project demonstration."""\nfrom pathlib import Path\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\n\nclass CommitmentPredictor:\n    def __init__(self, model_directory, device=None):\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.tokenizer = AutoTokenizer.from_pretrained(Path(model_directory), use_fast=False)\n        self.model = AutoModelForSequenceClassification.from_pretrained(Path(model_directory)).to(self.device).eval()\n\n    @torch.inference_mode()\n    def predict(self, context, target_utterance, target_speaker=\'employee\'):\n        text = f\'[CONTEXT] {context} [TARGET SPEAKER] {target_speaker} [TARGET UTTERANCE] {target_utterance}\'\n        tokens = self.tokenizer(text, truncation=False, return_tensors=\'pt\')\n        if tokens[\'input_ids\'].shape[1] > 256:\n            raise ValueError(\'Input exceeds the 256-token training limit; shorten the conversation context.\')\n        probabilities = self.model(**{k:v.to(self.device) for k,v in tokens.items()}).logits.softmax(-1)[0].cpu().tolist()\n        category = max(range(len(probabilities)), key=probabilities.__getitem__)\n        return {\'category\':category, \'label\':self.model.config.id2label[category], \'confidence\':probabilities[category], \'probabilities\':{self.model.config.id2label[i]:p for i,p in enumerate(probabilities)}}\n')
(BRES/'README.md').write_text(f'''# Balanced context-dependent project model v2
Goal: improve commitment detection by balancing the labels and adding more varied examples. Previous datasets/models/results remain unchanged in earlier notebook cells.
7680 training examples (1920 per class);512 validation (128 per class);768 fresh test (192 per class). Training uses24 reply families,8 context families,20 scenarios across ten workplace domains. Validation uses4 reply/context families and8 separate domains. Test uses6 new reply/context families and8 new domains, including seed-bank operations, robotics, sound archives and other previously unused scenarios.
Every scenario/reply family contributes16 balanced examples. Direct and indirect replies each occur verbatim in seven contexts: one factual report/quotation (No),four unconditional requests (Explicit for direct,Latent for indirect),two unresolved conditional requests (Qualified). The same refusal occurs under an unconditional and conditional request (No). Hence each group has four rows per class. Same contexts receive different labels for different replies. This balances actual dataset rows, not just loss weights or duplicate oversampling.
New variations include explicit promises, affirmative ellipsis, preparatory/in-progress language, multi-turn prerequisite discussions, fulfilled prerequisites, withdrawn prerequisites, unresolved prerequisites, and quoted/scripted speech versus live undertakings. A direct promise in an explicitly fictional rehearsal context is No; it is not treated as a real undertaking. Pragmatic availability offered in reply to an ownership request follows the broad Latent policy of the supplied project dataset. Qualified acceptance inherits an unresolved condition from context. These are agent-authored provisional labels; human review has not been performed.
Leakage controls: splits are assigned by reply families,context families and domain/scenario before training; no random row split. Same-target groups stay intact. Zero cross-split exact inputs,contexts,targets,assigned families,scenarios or domains. Test has zero exact target/context/input overlap with all prior experimental datasets. Cross-split target-template token SequenceMatcher similarity <=0.625 and context-template similarity <=0.7778 (below0.80). Between train and final test the maxima are0.5556 and0.6667. CSV hashes,manifest and reproducible generator are included. No model performance informed split allocation or labels; no test-based filtering or tuning occurred. These checks do not guarantee absence of every semantic similarity or dataset artifact.
Balanced majority/chance accuracy25%. Empirical deterministic target-only and context-only ceilings are each62.5% because identical inputs occur with conflicting labels. Measured TF-IDF+LR baselines are included. Full-input learning is needed for high accuracy, although synthetic construction cues remain.
Fresh microsoft/deberta-v3-base, not any previous fine-tuned checkpoint. Three epochs,seed2026,lr2e-5,batch8,accumulation2,evalbatch16,AdamW,weightdecay0.01,warmup0.1,FP16,maxlength256,dynamicpadding,earlystoppingpatience2. Best checkpoint selected only on validation macro-F1. Test predicted once after selection. Model input contains only serialized conversation,target speaker and target utterance; no labels,rationales,IDs or split fields. No inputs truncated. Reload compared against validation logits.
Training seconds:{b_seconds:.2f};GPU:{b_info['gpu']};selected checkpoint:{b_info['best_checkpoint']};validation macro-F1:{b_info['best_validation_macro_f1']}.
New model on fresh test:{json.dumps(b_test_metrics)}.
Previous84.90% model evaluated on the same fresh test:{json.dumps(b_old_metrics)}.
The old84.90% figure was on a different test, so use same_test_comparison.csv for fair comparison. New-vs-old changes involve balancing,variety,training size and seed; they do not isolate one factor's causal effect.
Test has96 context-dependent groups,each with seven contexts,plus96 refusal-control rows. There are14 different-label pairs per seven-context group,1344 pairs total. Pair metrics exclude same-label pairs and refusal controls. The representative triplet picks factual,U1,C1 for each group,for comparison of three different labels. Complete seven-context accuracy is stricter. Rows/pairs are correlated: there are six held-out reply/context families and eight scenario domains,not768 independent language constructions. Group/domain/condition breakdowns and error rows are saved.
This is a useful project demonstration evaluation under a synthetic data distribution. It is not a claim of production certification or natural workplace accuracy. Scores may benefit from shared annotation logic and systematic context wording. In particular,explicit factual-vs-task and rehearsal descriptions make some negative examples clearer than real conversations. Independent human-reviewed examples should be used to assess deployment.
The separate model ZIP includes weights,tokenizer,label mappings and project_predict.py. To use: unzip it into a model directory; import CommitmentPredictor from project_predict; create CommitmentPredictor(model_directory); call predict(context,target_utterance). It returns label,confidence and all four probabilities. It rejects inputs above256tokens rather than silently removing context. Confidence is an uncalibrated softmax score.
''')
shutil.copy(ERESULTS/'requirements.txt',BRES/'requirements.txt')
print('BALANCED PROJECT MODEL V2 — TRAINING AND FRESH TEST COMPLETE')
print('Train7680 / validation512 / test768; all four classes equally represented.')
print('New model: %.2f%% accuracy (%d/768); macro-F1 %.6f' % (100*b_test_metrics['accuracy'],b_test_metrics['correct_examples'],b_test_metrics['macro_f1']))
print('Previous model on SAME fresh test: %.2f%% accuracy; macro-F1 %.6f' % (100*b_old_metrics['accuracy'],b_old_metrics['macro_f1']))
print('Context-dependent rows: new%.2f%%; previous%.2f%%' % (100*b_test_metrics['context_dependent_accuracy'],100*b_old_metrics['context_dependent_accuracy']))
print('Latent recall %.2f%% / F1 %.6f; Explicit recall %.2f%% / F1 %.6f' % (100*b_report['Latent Commitment']['recall'],b_report['Latent Commitment']['f1-score'],100*b_report['Explicit Commitment']['recall'],b_report['Explicit Commitment']['f1-score']))
print('All seven contexts correct: %d/96 groups. Representative triplets: %d/96.' % (b_test_metrics['groups_all_seven_correct'],b_test_metrics['representative_triplets_all_correct']))
print('Zero exact split overlap; fresh test absent from prior datasets. Validation selection only.')
print('Saved model reload verified. Synthetic labels; six held-out test wording families.')
try:
    b_payload=_message.blocking_request('get_ipynb',timeout_sec=10)
except Exception as exc:
    b_payload=None
if isinstance(b_payload,dict) and 'ipynb' in b_payload:
    b_nb=b_payload['ipynb'];(BRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(b_nb,indent=1))
else:
    print('Notebook API unavailable: download the actual notebook through File > Download; all model/data exports continue.',flush=True)

(BSPLIT/'build_balanced_context_v2.py').write_text('"""Context training v1: fixed authored split families, no performance-based allocation."""\nimport csv,json,re,hashlib,itertools,difflib,random,collections\nfrom pathlib import Path\nLABELS_C=[\'NO_COMMITMENT\',\'LATENT_COMMITMENT\',\'QUALIFIED_COMMITMENT\',\'EXPLICIT_COMMITMENT\']\nREPLIES={\n \'train\':[\n (\'Yes, I confirm your statement.\',\'I have an hour open and the reference files ready.\',\'I am not agreeing to own this work.\'),\n (\'That matches what I understand.\',\'My next block is free, and the material is in front of me.\',\'Please assign this to somebody else; I am declining.\'),\n (\'You have my agreement on that.\',\'I already brought up the relevant pages and cleared a slot.\',\'I am unable to accept this assignment.\'),\n (\'I agree with what you just said.\',\'There is room on my calendar, and I have the documents here.\',\'This is not an undertaking I can give you.\'),\n (\'That is an accurate account.\',\'The working files are open beside me, and I have time.\',\'I will not take responsibility for that task.\'),\n (\'I can confirm that is right.\',\'I kept a gap in the day and collected the background notes.\',\'I must turn down the proposed responsibility.\'),\n (\'Yes, you described it correctly.\',\'I have the source material handy and a clear stretch ahead.\',\'I cannot be the person accountable for this.\'),\n (\'I assent to that proposition.\',\'My schedule has space, and the necessary records are at hand.\',\'I am saying no to taking that on.\')],\n \'validation\':[\n (\'Your understanding matches mine.\',\'I reserved a work period and located the supporting packet.\',\'I decline the role you are offering.\'),\n (\'That gets a yes from me.\',\'I left a window in the morning and retrieved the paperwork.\',\'You need a different owner for this; I refuse.\'),\n (\'We are in agreement there.\',\'The background is on my desk, with an empty slot ahead.\',\'I am not willing to undertake the assignment.\'),\n (\'Affirmative; that is settled.\',\'I made space between calls and found the reference copy.\',\'That responsibility cannot be mine.\')],\n \'test\':[\n (\'You have my confirmation.\',\'The briefing pack is within reach and my diary has an opening.\',\'My answer is no; I will not be its owner.\'),\n (\'I endorse that statement.\',\'I blocked out a working session and assembled the relevant evidence.\',\'Count me out of carrying this through.\'),\n (\'Those words reflect my position.\',\'I am sitting with the supporting documents during an unbooked hour.\',\'Find another volunteer; I am passing on this.\'),\n (\'Yes, we understand each other.\',\'I rearranged my appointments and gathered the source pages nearby.\',\'I reject the invitation to be responsible for this.\')]\n}\n# Each tuple: future action, unresolved authorization/prerequisite, task domain.\nSCENARIOS={\n \'train\':[\n (\'review the invoice ledger\',\'finance releases the source entries\',\'finance\'),(\'reconcile the expense claims\',\'the receipts become accessible\',\'finance\'),(\'prepare the payment summary\',\'the controller approves the figures\',\'finance\'),(\'check the supplier balances\',\'the bank file is delivered\',\'finance\'),\n (\'update the access register\',\'security grants permission\',\'security\'),(\'inspect the firewall checklist\',\'the test account is activated\',\'security\'),(\'compile the incident timeline\',\'the investigation lead releases the logs\',\'security\'),(\'verify the backup inventory\',\'the storage scan finishes\',\'security\'),\n (\'organize the recruitment agenda\',\'the panel confirms attendance\',\'people\'),(\'revise the induction notes\',\'HR approves the new policy\',\'people\'),(\'prepare the training roster\',\'the department releases the names\',\'people\'),(\'check the leave spreadsheet\',\'the team leads approve the entries\',\'people\'),\n (\'compare the delivery quotes\',\'procurement opens the tender files\',\'purchasing\'),(\'summarize the vendor proposals\',\'the buyer authorizes the assessment\',\'purchasing\'),(\'review the order schedule\',\'the warehouse confirms capacity\',\'purchasing\'),(\'check the stock adjustment\',\'the inventory count is signed\',\'purchasing\'),\n (\'edit the product announcement\',\'marketing approves the draft\',\'marketing\'),(\'assemble the campaign brief\',\'the sponsor clears the budget\',\'marketing\'),(\'review the customer newsletter\',\'the editor grants clearance\',\'marketing\'),(\'prepare the launch checklist\',\'the product team confirms the date\',\'marketing\'),\n (\'check the deployment notes\',\'engineering releases the build\',\'engineering\'),(\'revise the testing guide\',\'the maintainer approves the changes\',\'engineering\'),(\'summarize the defect reports\',\'the tracker becomes available\',\'engineering\'),(\'review the integration plan\',\'the partner supplies the interface\',\'engineering\'),\n (\'prepare the service overview\',\'support releases the case list\',\'support\'),(\'check the escalation queue\',\'the shift supervisor authorizes access\',\'support\'),(\'revise the troubleshooting notes\',\'the specialist confirms the fix\',\'support\'),(\'summarize the outage tickets\',\'the monitoring export arrives\',\'support\'),\n (\'review the facilities rota\',\'the building manager approves staffing\',\'facilities\'),(\'prepare the equipment log\',\'the technician completes inspection\',\'facilities\'),(\'check the office supplies list\',\'the department approves its request\',\'facilities\'),(\'revise the maintenance schedule\',\'the contractor confirms the visit\',\'facilities\'),\n (\'summarize the workshop feedback\',\'the organizer releases the responses\',\'events\'),(\'prepare the seminar handout\',\'the presenter approves the content\',\'events\'),(\'check the visitor schedule\',\'reception confirms the appointments\',\'events\'),(\'review the travel bookings\',\'the coordinator clears the itinerary\',\'events\'),\n (\'revise the document index\',\'the archivist releases the catalogue\',\'records\'),(\'check the records transfer\',\'the custodian authorizes the move\',\'records\'),(\'prepare the retention table\',\'the policy owner signs the revision\',\'records\'),(\'summarize the meeting minutes\',\'the chair approves the record\',\'records\')],\n \'validation\':[\n (\'map the clinical equipment locations\',\'the hospital administrator grants clearance\',\'hospital_operations\'),(\'compile the laboratory calibration list\',\'the laboratory head approves the access\',\'laboratory\'),(\'review the museum loan schedule\',\'the curator releases the loan records\',\'museum\'),(\'prepare the library digitization plan\',\'the librarian approves scanning\',\'library\'),(\'check the kitchen allergen guide\',\'the catering lead confirms ingredients\',\'catering\'),(\'summarize the fleet inspection sheets\',\'the transport manager authorizes release\',\'fleet\'),(\'organize the theatre rehearsal timetable\',\'the director confirms availability\',\'theatre\'),(\'revise the farm delivery manifest\',\'the farm operator approves dispatch\',\'agriculture\'),(\'prepare the nursery watering chart\',\'the horticulturist signs the plan\',\'horticulture\'),(\'check the sports venue reservations\',\'the venue officer approves the bookings\',\'sports\'),(\'summarize the repair depot backlog\',\'the depot manager releases the job cards\',\'repair\'),(\'review the radio studio allocations\',\'the producer confirms the recording slots\',\'broadcasting\')],\n \'test\':[\n (\'assemble the harbour mooring register\',\'the harbourmaster authorizes the inspection\',\'maritime\'),(\'check the solar array inspection notes\',\'the site engineer permits entry\',\'renewable_energy\'),(\'review the railway platform signage\',\'the station controller clears the work\',\'railway\'),(\'prepare the geological sample inventory\',\'the field lead releases the specimens\',\'geology\'),(\'summarize the observatory instrument bookings\',\'the astronomer approves instrument use\',\'astronomy\'),(\'revise the water treatment sampling plan\',\'the plant chemist validates the protocol\',\'water\'),(\'compile the textile loom service records\',\'the mill supervisor permits the audit\',\'textiles\'),(\'organize the wildlife survey route\',\'the ranger issues the field permit\',\'conservation\'),(\'review the bridge inspection photographs\',\'the civil engineer releases the images\',\'civil_engineering\'),(\'check the bakery production sequence\',\'the production chief confirms the batch order\',\'bakery\'),(\'prepare the recycling depot handover\',\'the depot controller approves the transfer\',\'recycling\'),(\'summarize the wind tunnel experiment schedule\',\'the researcher authorizes the test window\',\'aerodynamics\')]\n}\n# Context templates are allocated wholly by split. All contain placeholders, never label names.\nTEMPLATES={\n \'train\':[\n (\'We need someone to {a} {t}. Can we agree that you will be the person doing it?\', \'The proposal is that you {a} {t}, but only if {c}. The prerequisite is still pending; without it the work will not go ahead. Do you accept that arrangement?\', \'For the records check, the material needed to {a} is in the shared folder. Is that statement accurate?\', \'For the availability record, what time and reference material do you currently have? We are gathering facts, not assigning the work to {a}.\'),\n (\'Can I put you down to {a} {t}? The work is ready to start.\', \'Can I put you down to {a} {t}, contingent on {c}? That event has not occurred, so the arrangement depends on it.\', \'The office record says we have a request to {a}. Are those facts correct?\', \'We are documenting capacity before deciding who will {a}. Describe your available time and material; no owner is being chosen in this conversation.\'),\n (\'Will you personally {a} {t}? We have everything required.\', \'Would you take on the plan to {a} {t} provided {c}? That permission is not yet available, and the plan is off otherwise.\', \'Someone asked us to {a}. I am only checking that this is what the note says. Do you agree that the note says that?\', \'For a factual desk and calendar check, tell me what is ready and when you are free. This question does not ask you to {a}.\'),\n (\'I propose that you {a} {t}. Do you agree to be responsible?\', \'I propose that you {a} {t} subject to {c}. The decision is unresolved, so that restriction remains part of the proposal. Do you agree?\', \'The background files for the request to {a} exist in the archive. Do you confirm that description?\', \'Before any assignment to {a}, describe the time and documents you have. I am recording resources only.\'),\n (\'We have no owner to {a} {t}. Are you accepting the job?\', \'We need an owner to {a} {t} if {c}, and that has not been decided. Are you accepting this dependent arrangement?\', \'The planning list includes an item to {a}. Am I reporting that correctly?\', \'I am doing a schedule survey, not choosing someone to {a}. What preparation and space do you currently have?\'),\n (\'Please confirm the agreement that you will {a} {t}. Nothing is waiting on authorization.\', \'Please confirm the agreement that you will {a} {t} only after {c}. Authorization remains outstanding and the job depends on it.\', \'I read that there is a request to {a}. Is my reading of the document correct, without discussing who will do it?\', \'For my factual notes, describe your current readiness. This is an inventory question, not a request to {a}.\'),\n (\'Who can take responsibility to {a} {t}? Is that something you are taking on?\', \'Who will take responsibility to {a} {t} on the condition that {c}? We do not have that clearance yet. Is that conditional plan acceptable?\', \'The reference card mentions a plan to {a}. Is that an accurate description of the card?\', \'I need your calendar status and what is on your desk for the resource census; nobody is being assigned to {a} here.\'),\n (\'Can we settle on you as the owner to {a} {t}? All prerequisites have already been satisfied.\', \'Can we settle on you to {a} {t} in the event that {c}? The event is uncertain and has not occurred, so there is no work without it.\', \'Does the task record correctly state that there was a request to {a}? This is a check of the record only.\', \'For the capacity spreadsheet, report your available resources, rather than accepting a task to {a}.\')],\n \'validation\':[\n (\'Your proposed role would be to {a} {t}. Shall we treat that as your undertaking?\', \'Your proposed role would be to {a} {t} once {c}. We are awaiting that prerequisite, and the offer is limited by it. Does that describe what you accept?\', \'Do you endorse this factual statement: our archive contains documents relevant to the proposal to {a}?\', \'Give me a snapshot of your work area and diary. We will choose who should {a} in another meeting, not in this one.\'),\n (\'Are we agreed that you are the one who will {a} {t}? The necessary approval is in place.\', \'Are we agreed that you would {a} {t} dependent on {c}? Nobody has obtained that prerequisite yet.\', \'Our request log contains a line about whether to {a}. Is my account of the log true?\', \'I want a factual description of your current capacity before we talk about responsibility to {a}.\'),\n (\'I would like you to {a} {t}. Are you committing to that?\', \'I would like you to {a} {t} if and only if {c}. We still lack that decision; is the constrained proposal what you are accepting?\', \'The information sheet refers to a possible need to {a}. Are you confirming the sheet contains that information?\', \'Tell me what material you have to hand and how busy you are. This is for a resource audit; it does not appoint you to {a}.\'),\n (\'May I record your acceptance of the responsibility to {a} {t}? The task is authorized.\', \'May I record your acceptance to {a} {t} as long as {c}? That authorization is outstanding; it limits the arrangement.\', \'Please verify the truth of the statement that our notes discuss an item to {a}.\', \'Describe your preparation and spare time for the scheduling survey. We have deliberately left ownership to {a} undecided today.\')],\n \'test\':[\n (\'Shall your name be entered as accountable for the work to {a} {t}? We can proceed immediately.\', \'Shall your name be entered to {a} {t} with the proviso that {c}? The proviso is unsatisfied; the arrangement lapses without it.\', \'In a fact check, I said that our documents mention a proposal to {a}. Was my statement true?\', \'This interview concerns the present contents of your desk and diary. Allocation of the work to {a} is outside this interview. What is your current situation?\'),\n (\'Do you give your agreement to personally {a} {t}? There are no outstanding permissions.\', \'Do you give your agreement to {a} {t} contingent upon {c}? That requirement has not been met and cannot be waived.\', \'Does your view concur with the historical fact that the request to {a} was entered in the system?\', \'We are surveying current readiness only. The choice of who will {a} will be made later by someone else. Describe what you already have available.\'),\n (\'I am asking for your undertaking to {a} {t}. Are we of one mind about you doing the work?\', \'I am asking whether you accept the limited undertaking to {a} {t}, pending {c}. That is unresolved, and there is no authorization to proceed without it.\', \'The question is whether my description of the paperwork about the request to {a} is accurate. Do you support that description?\', \'For an observational study, describe your free time and nearby reference materials. This interview offers no assignment to {a}.\'),\n (\'Would you assent to being responsible to {a} {t}? You can start without waiting for anything.\', \'Would you assent to the arrangement to {a} {t} in case {c}? That contingency is still open; failure to satisfy it cancels the task.\', \'Please affirm or reject this account of the facts: the task list discusses whether to {a}.\', \'We need a descriptive account of your calendar and papers, with responsibility to {a} reserved for a future discussion.\')]\n}\nTIMES={\'train\':[\'this morning\',\'this afternoon\',\'tomorrow\',\'before the next briefing\',\'by the end of the week\',\'during the next shift\',\'before Thursday\',\'after the team meeting\'],\'validation\':[\'before the monthly review\',\'during the upcoming work session\',\'by next Wednesday\',\'over the next two days\'],\'test\':[\'ahead of the next inspection\',\'during the following duty period\',\'before the project checkpoint\',\'by the next planning call\']}\ndef build_context_training(destination):\n out=Path(destination);out.mkdir(parents=True,exist_ok=True);allrows={};manifest=[]\n for split in [\'train\',\'validation\',\'test\']:\n  rows=[]\n  for si,(action,condition,domain) in enumerate(SCENARIOS[split]):\n   for fi,(direct,indirect,refusal) in enumerate(REPLIES[split]):\n    u,c,fd,fi_ctx=TEMPLATES[split][fi];fmt={\'a\':action,\'c\':condition,\'t\':TIMES[split][fi]}\n    contexts={\'U\':u.format(**fmt),\'C\':c.format(**fmt),\'FD\':fd.format(**fmt),\'FI\':fi_ctx.format(**fmt)}\n    gid=f\'{split}_S{si:02}_F{fi:02}\'\n    specs=[(\'direct\',\'FD\',0,direct),(\'direct\',\'U\',3,direct),(\'direct\',\'C\',2,direct),(\'indirect\',\'FI\',0,indirect),(\'indirect\',\'U\',1,indirect),(\'indirect\',\'C\',2,indirect),(\'refusal\',\'U\',0,refusal),(\'refusal\',\'C\',0,refusal)]\n    for j,(replykind,ctxkind,label,target) in enumerate(specs):\n     ctx=\'[manager]: \'+contexts[ctxkind]\n     rationale={0:(\'Explicit refusal of responsibility.\' if replykind==\'refusal\' else \'A factual confirmation/readiness report, with no task acceptance requested.\'),1:\'Readiness/preparation answers an ownership request and pragmatically signals uptake without a direct undertaking.\',2:\'The employee accepts or signals uptake of a proposal whose unresolved prerequisite remains in scope.\',3:\'A direct affirmative confirmation accepts the contextual proposition that the employee will perform the task.\'}[label]\n     rows.append({\'sample_id\':gid+f\'_{j}\',\'scenario_id\':f\'{split}_S{si:02}\',\'family_id\':f\'{split}_F{fi:02}\',\'contrast_group_id\':gid,\'target_group_id\':gid+\'_\'+replykind,\'reply_kind\':replykind,\'context_kind\':ctxkind,\'domain\':domain,\'split\':split,\'conversation_context\':ctx,\'target_speaker\':\'employee\',\'target_utterance\':target,\'model_input_text\':f\'[CONTEXT] {ctx} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target}\',\'commitment_label\':LABELS_C[label],\'labels\':label,\'annotation_rationale\':rationale,\'human_review_required\':True})\n  random.Random(42).shuffle(rows);allrows[split]=rows\n  with (out/(split+\'.csv\')).open(\'w\',newline=\'\') as f:\n   w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n  manifest.extend({k:r[k] for k in [\'sample_id\',\'scenario_id\',\'family_id\',\'contrast_group_id\',\'target_group_id\',\'split\']} for r in rows)\n audit={\'seed\':42,\'split_method\':\'authored disjoint reply/context families and scenario domains; fixed before training\',\'counts\':{},\'cross_split_overlap\':{},\'csv_sha256\':{},\'annotation\':\'agent-authored provisional; no human validation\',\'earlier_challenge_used_for_training\':False}\n def norm(x):return \' \'.join(re.findall(r\'[a-z]+\',x.lower()))\n for split,rows in allrows.items():\n  assert len({r[\'model_input_text\'] for r in rows})==len(rows)\n  groups=collections.defaultdict(list)\n  for r in rows:groups[r[\'target_group_id\']].append(r)\n  for g in groups.values():\n   assert len({r[\'target_utterance\'] for r in g})==1\n   if g[0][\'reply_kind\']!=\'refusal\':assert len(g)==3 and len({r[\'labels\'] for r in g})==3\n  def ceiling(field):\n   grouped=collections.defaultdict(collections.Counter)\n   for r in rows:grouped[r[field]][r[\'labels\']]+=1\n   return sum(max(c.values()) for c in grouped.values())/len(rows)\n  audit[\'counts\'][split]={\'rows\':len(rows),\'class_counts\':dict(collections.Counter(r[\'commitment_label\'] for r in rows)),\'families\':len(REPLIES[split]),\'scenarios\':len(SCENARIOS[split]),\'context_dependent_triplets\':sum(g[0][\'reply_kind\']!=\'refusal\' for g in groups.values()),\'refusal_control_groups\':sum(g[0][\'reply_kind\']==\'refusal\' for g in groups.values()),\'target_only_empirical_ceiling\':ceiling(\'target_utterance\'),\'context_only_empirical_ceiling\':ceiling(\'conversation_context\')}\n  audit[\'csv_sha256\'][split]=hashlib.sha256((out/(split+\'.csv\')).read_bytes()).hexdigest()\n for a,b in itertools.combinations(allrows,2):\n  check={}\n  for col in [\'model_input_text\',\'conversation_context\',\'target_utterance\',\'scenario_id\',\'family_id\',\'contrast_group_id\',\'target_group_id\',\'domain\']:\n   check[col]=len({norm(r[col]) for r in allrows[a]}&{norm(r[col]) for r in allrows[b]})\n   assert check[col]==0,(a,b,col,check[col])\n  for name,items in [(\'target_templates\',REPLIES),(\'context_templates\',TEMPLATES)]:\n   aa=[norm(t) for p in items[a] for t in p];bb=[norm(t) for p in items[b] for t in p]\n   check[\'max_\'+name+\'_token_similarity\']=max(difflib.SequenceMatcher(None,x.split(),y.split(),autojunk=False).ratio() for x in aa for y in bb)\n  audit[\'cross_split_overlap\'][a+\'_\'+b]=check\n with (out/\'split_manifest.csv\').open(\'w\',newline=\'\') as f:\n  w=csv.DictWriter(f,fieldnames=list(manifest[0]));w.writeheader();w.writerows(manifest)\n (out/\'split_audit.json\').write_text(json.dumps(audit,indent=2))\n return allrows,audit\n\n"""Balanced context-dependent dataset v2. Fixed authoring before training/testing."""\nimport csv,json,re,hashlib,random,itertools,collections,difflib\nfrom pathlib import Path\n# Load only the previous generator\'s definitions, not its train/evaluate routines.\n# In Colab, REPLIES/SCENARIOS/TEMPLATES are already present from experiment E.\nNEW_DIRECT=[\n (\'Yes, I am agreeing to the proposal you stated.\',\'affirmative\'),(\'We have a deal on those terms.\',\'affirmative\'),(\'That proposal has my acceptance.\',\'affirmative\'),(\'You may record my agreement.\',\'affirmative\'),(\'I am saying yes to what you proposed.\',\'affirmative\'),(\'That accurately expresses what I agree to.\',\'affirmative\'),(\'I concur with that account.\',\'affirmative\'),(\'It is as you have just described.\',\'affirmative\'),\n ("I\'ll take this responsibility and carry it through.",\'undertaking\'),(\'I will make sure it gets done.\',\'undertaking\'),(\'I promise to handle the work you described.\',\'undertaking\'),(\'I am taking ownership of that task.\',\'undertaking\'),("I\'ll be the one who does it.",\'undertaking\'),(\'You can rely on me to finish that work.\',\'undertaking\'),(\'I commit to carrying out the requested work.\',\'undertaking\'),(\'I will deliver what you have asked for.\',\'undertaking\')]\nNEW_INDIRECT=[\n \'I have the working notes beside me and a stretch of time available.\',\n \'My calls are out of the way, and the source documents are already open.\',\n \'I brought the background papers over and kept an hour aside.\',\n \'The task material is on my screen, and I moved my other meeting.\',\n \'I gathered the supporting records and have a quiet period now.\',\n \'The reference packet is at my desk and my afternoon has room.\',\n \'I made a space in my workload and pulled together the relevant information.\',\n \'I already found the source pages, with time left before my next appointment.\',\n \'The first page is marked up in front of me, and I left the rest of the day clear.\',\n \'I have begun looking through the material and reserved time to continue.\',\n \'My initial notes are already on the page, and there is a gap between calls.\',\n \'I pulled out the documents and started following the outstanding points.\',\n \'I am at the desk with the records, and the next hour is unbooked.\',\n \'The preliminary checks are open here and I set aside a work period.\',\n \'I have been tracing the relevant details, with time put by to continue.\',\n \'The material is laid out for me and I shifted my other work out of the way.\']\nNEW_REFUSAL=[\n \'I am declining to be responsible for this work.\',\'I cannot give you that undertaking.\',\n \'That is not something I will own.\',\'My response is a refusal to take the job.\',\n \'I am not accepting the task you have proposed.\',\'Please leave my name off this assignment.\',\n \'I will not be accountable for completing this.\',\'I am turning down the request to do that.\',\n \'No, I am not volunteering for this responsibility.\',\'I need to decline; I will not do the work.\',\n \'I do not agree to take charge of it.\',\'I am saying no to the assignment.\',\n \'This responsibility needs to go to someone other than me.\',\'I am not the owner of this and will not become its owner.\',\n \'I cannot accept your proposal that I do it.\',\'Do not count on me to carry out that task.\']\nV2_REPLY={\'train\':[(d,i,r,\'affirmative\') for d,i,r in REPLIES[\'train\']]+[(d,NEW_INDIRECT[k],NEW_REFUSAL[k],mode) for k,(d,mode) in enumerate(NEW_DIRECT)],\n\'validation\':[\n (\'Your proposal is acceptable to me.\',\'I cleared a work slot and placed the background folder nearby.\',\'I must decline the responsibility you are suggesting.\',\'affirmative\'),\n (\'I acknowledge and accept those terms.\',\'The relevant documents are spread out and I have an interval free.\',\'I am not going to undertake the task on offer.\',\'affirmative\'),\n (\'I undertake to complete the task you described.\',\'I have made a first pass through the records and kept time to go on.\',\'I am withdrawing myself from consideration for that work.\',\'undertaking\'),\n (\'I will see that the requested work is finished.\',\'My preparation is on the desk and I have put other work aside.\',\'Please do not treat me as agreeing to this responsibility.\',\'undertaking\')],\n \'test\':[\n (\'I give my assent to your proposal.\',\'I have a working copy ready and rearranged the next part of my day.\',\'I am refusing the role you have put forward.\',\'affirmative\'),\n (\'Those terms are ones I accept.\',\'I am sitting with the collected evidence and have a free work period.\',\'I will not assume the obligation to do this.\',\'affirmative\'),\n (\'We can regard that proposal as agreed.\',\'The preparatory pages are here and my upcoming appointments have been moved.\',\'The work needs a different person; I am declining to own it.\',\'affirmative\'),\n (\'I shall take charge of getting that done.\',\'I have the opening checks on the screen and kept a session vacant.\',\'You do not have my acceptance of the assignment.\',\'undertaking\'),\n (\'I promise to see the assignment through to completion.\',\'The background material has my first annotations and I made time for the next step.\',\'I am not offering to carry this out for you.\',\'undertaking\'),\n (\'I will personally carry out the work you requested.\',\'I assembled the relevant packet and set my other duties aside for a while.\',\'I decline any responsibility for performing that work.\',\'undertaking\')]\n}\n# New validation/test domains/actions have never appeared in prior model evaluation.\nV2_SCENARIOS={\'train\':SCENARIOS[\'train\'][::2],\n \'validation\':[\n (\'prepare the aquarium feed inventory\',\'the aquarist approves the revised feeding amounts\',\'aquarium\'),(\'check the bookbinding repair register\',\'the conservator releases the damaged volumes\',\'bookbinding\'),(\'compile the planetarium demonstration rota\',\'the programme director confirms the equipment\',\'planetarium\'),(\'review the ceramic kiln schedule\',\'the workshop lead certifies the firing plan\',\'ceramics\'),(\'organize the language centre assessment packs\',\'the coordinator clears the assessment material\',\'language_centre\'),(\'revise the cable car servicing checklist\',\'the operations engineer approves the inspection\',\'cable_car\'),(\'summarize the cinema projection handover\',\'the chief projectionist releases the logs\',\'cinema\'),(\'check the glassworks dispatch sequence\',\'the furnace manager confirms the safe packing order\',\'glassworks\')],\n \'test\':[\n (\'compile the seed bank accession register\',\'the collections officer permits use of the accession records\',\'seed_bank\'),(\'review the puppet theatre touring checklist\',\'the touring manager confirms the transport arrangements\',\'puppet_theatre\'),(\'prepare the robotics lab equipment rotation\',\'the lab coordinator authorizes the robot allocation\',\'robotics\'),(\'check the quarry survey handover\',\'the survey supervisor signs the site clearance\',\'quarry\'),(\'summarize the ice rink maintenance record\',\'the rink technician approves the servicing details\',\'ice_rink\'),(\'organize the weaving class material packs\',\'the tutor confirms the thread allocations\',\'weaving_education\'),(\'revise the community garden tool schedule\',\'the garden steward approves the shared equipment plan\',\'community_garden\'),(\'review the sound archive restoration queue\',\'the archive engineer releases the audio masters\',\'sound_archive\')]\n}\n# Eight train context families, four validation, six test. A profile has:\n# U1 ordinary request, U2 another unqualified request, U3 fulfilled prerequisite,\n# U4 prerequisite removed, C1 pending condition, C2 multi-turn pending condition,\n# F affirmative factual confirmation, F indirect resource report, F undertaking quotation.\n# Same U/C contexts take direct, indirect and refusal replies with different labels.\nV2_CONTEXTS={\n \'train\':[\n (\'Can we agree that you will {a} {t}?\',\'I need you to {a} {t}. Are you taking responsibility?\',\n \'Earlier I said you could {a} {t} only if {c}. That has now happened, and the condition is satisfied. Will you take the work on?\',\n \'We considered making the plan to {a} dependent on {c}, but that restriction has been removed. The request now is for you to {a} {t}. Do you accept?\',\n \'Will you {a} {t} only if {c}? That decision is pending and the task will not proceed without it.\',\n \'[employee]: Is the work to {a} approved? [manager]: Not yet. You would {a} {t} provided {c}, which is still unresolved. Do you accept that arrangement?\',\n \'I stated that a request to {a} exists in the document. Is that account correct?\',\n \'For the capacity survey, describe your available time and the material on your desk. This is a factual check, not an assignment to {a}.\',\n \'We are rehearsing a line for a training recording, not assigning any work to {a}. Read the sample employee promise aloud; the recording creates no actual obligation.\'),\n (\'Are you agreeing to own the work to {a} {t}?\',\'May I record your agreement that you will {a} {t}?\',\n \'The earlier condition was that you would {a} if {c}. We have confirmation that it did occur. We now need you to {a} {t}. Are you agreeing?\',\n \'Approval is no longer a requirement: we dropped the proposal that {c} must happen first. Will you {a} {t} without that restriction?\',\n \'The offer is that you {a} {t} on condition that {c}. That has not occurred. Does that restricted offer have your agreement?\',\n \'[employee]: Can I begin the task to {a}? [manager]: We are still waiting. You may undertake it {t} contingent on {c}; otherwise there is no task. Are you accepting those terms?\',\n \'Our log records a proposal to {a}. Am I describing the log truthfully?\',\n \'I am asking for a description of your diary and reference material, not asking you to {a}. What is already available?\',\n \'For the staff workshop, quote a sample acceptance response from the script. We are only practicing speech, with no real assignment to {a}.\'),\n (\'Will you personally {a} {t}?\',\'The work to {a} needs an owner {t}. Is that you?\',\n \'We previously had to wait for {c} before anyone could {a}. That prerequisite has been met. Can we count on you to {a} {t}?\',\n \'The requirement that {c} has been waived completely. I am asking you to {a} {t}, with nothing left to wait for. Do you agree?\',\n \'Can you accept responsibility to {a} {t} provided {c}? We are still awaiting that event.\',\n \'[manager]: We need someone to {a}. [employee]: Is the permission settled? [manager]: No. The plan is that you do it {t} if {c}, and that remains open. Do you agree to that plan?\',\n \'Is it factually right that our notes mention a need to {a}?\',\n \'For a calendar and desk inventory, tell me what is open and when you are free. We are not choosing the person to {a} in this exchange.\',\n \'In this pronunciation exercise, say the fictional employee undertaking. Nobody is actually being asked to {a}; it is a line from an example dialogue.\'),\n (\'Please confirm you will {a} {t}.\',\'Would you accept the assignment to {a} {t}?\',\n \'The condition for the work to {a} was that {c}. It is now fulfilled, and the work is ready. Do you agree to {a} {t}?\',\n \'Our first plan tied the work to {c}, but we deleted that dependency. Are you now agreeing to {a} {t} as an ordinary assignment?\',\n \'Please confirm the conditional plan that you {a} {t} if {c}. We do not have that clearance yet.\',\n \'[employee]: Do we have clearance to {a}? [manager]: It is outstanding. I propose that you {a} {t} subject to {c}. Are those terms acceptable?\',\n \'I read that the task list contains an item to {a}. Do you agree that my reading is accurate?\',\n \'Before ownership of the task to {a} is discussed, report the resources you have at hand. I am collecting facts only.\',\n \'This is a simulated role-play. Deliver the employee acceptance line for the imaginary task to {a}; the exercise does not give you that real responsibility.\'),\n (\'Do you accept the proposal that you {a} {t}?\',\'Shall I put your name against the work to {a} {t}?\',\n \'We were waiting to {a} until {c}. The wait is over: that event has taken place. Will you {a} {t}?\',\n \'We decided not to require {c} after all. Can I record your undertaking to {a} {t}, with that prerequisite removed?\',\n \'Shall I put you down to {a} {t} in the event that {c}? The prerequisite remains unsatisfied.\',\n \'[manager]: There is a proposal to {a}. [employee]: Is it definite? [manager]: It will happen {t} only if {c}, which is uncertain. Will you accept the proposal on that basis?\',\n \'The planning document refers to a request to {a}. Does your understanding agree with that factual statement?\',\n \'This is a readiness census rather than a request to {a}. Describe your current preparation and free time.\',\n \'Read a promise as a quoted example for our communications class. It concerns an imaginary task to {a}, and reading it is not accepting that task.\'),\n (\'Can you confirm your undertaking to {a} {t}?\',\'Are you the person agreeing to {a} {t}?\',\n \'If {c}, we said the work to {a} could go ahead. It did happen; the requirement is complete. Are you taking the job {t}?\',\n \'The proposal originally depended on {c}; management has now cancelled that dependency. Are you agreeing to {a} {t}?\',\n \'Are you accepting the undertaking to {a} {t} dependent on {c}? We are waiting for that confirmation.\',\n \'[employee]: Is {c} confirmed? [manager]: No. Your proposed responsibility to {a} {t} would apply only after that happens. Are you taking on that limited arrangement?\',\n \'For a fact check, I said a request to {a} was registered. Is that description one you agree with?\',\n \'We are recording your existing work setup and calendar. No agreement to {a} is being sought in this question; describe what is there.\',\n \'During the microphone test, speak an example of an employee undertaking. The line is only scripted audio, not your actual acceptance of work to {a}.\'),\n (\'May we settle on you doing the task to {a} {t}?\',\'I propose you take charge and {a} {t}. Do you agree?\',\n \'The earlier restriction said we could {a} only once {c}. That event is now verified. Will you handle the job {t}?\',\n \'There is no longer any dependency on {c}. The earlier condition was withdrawn. Will you {a} {t}?\',\n \'I propose you {a} {t} as long as {c}. That remains pending; there will be no task if it fails.\',\n \'[manager]: Could you {a}? [employee]: Is approval available? [manager]: It is not. The arrangement would be for you to do it {t} once {c}. Does that describe what you accept?\',\n \'Please confirm whether our folder contains information about a proposal to {a}.\',\n \'For the workload notes, describe the time and materials currently available. We are not assigning you the work to {a}.\',\n \'Repeat a fictional promise for the acting exercise. This is not a live discussion of who will {a}; the words belong to the sample character.\'),\n (\'Are we agreed that you will {a} {t}?\',\'Will you accept responsibility for the request to {a} {t}?\',\n \'The request to {a} was subject to {c}. That has been completed, so the restriction no longer blocks it. Do you accept the work {t}?\',\n \'We removed the rule that {c} must come first. Are you agreeing to {a} {t} without that qualification?\',\n \'Would you agree to {a} {t} only after {c}? That prerequisite is not yet in place.\',\n \'[employee]: Is the plan to {a} approved? [manager]: It is a conditional proposal: you would do it {t} if {c}. Nothing has been decided about that yet. Is that proposal acceptable?\',\n \'I am checking the record, not allocating the task: does the record correctly say we have a request to {a}?\',\n \'For a factual resource report, tell me your current situation with time and documents. The decision about who will {a} will happen elsewhere.\',\n \'Please voice the sample employee promise in the rehearsal script. It has no effect on real responsibility to {a}; we only need the line for practice.\')],\n \'validation\':[\n (\'Is your answer an acceptance of the role to {a} {t}?\',\'Can the plan name you as the person who will {a} {t}?\',\n \'The trigger we had specified, namely {c}, has already occurred. The job to {a} can now proceed {t}; will you accept it?\',\n \'We abandoned the trigger involving {c}. Do you agree to {a} {t} now that no trigger limits the request?\',\n \'Would you accept the role to {a} {t}, pending {c}? The trigger has not occurred.\',\n \'[employee]: Is everything settled for the plan to {a}? [manager]: We still await the trigger: {c}. The role would apply {t} only on that basis. Do you accept?\',\n \'Do you support this account of the records: they discuss a proposal to {a}?\',\n \'In our descriptive survey, give your present diary and document status. Today we are not offering you the job to {a}.\',\n \'For a narrated example, say the character acceptance line. This narration does not establish an obligation for you to {a}.\'),\n (\'Would you endorse the proposal that you will {a} {t}?\',\'Do you consent to being the owner who will {a} {t}?\',\n \'We were prohibited from starting until {c}; that barrier has been cleared. Will you {a} {t}?\',\n \'A prior proposal required {c}. The requirement has been struck out. Would you endorse the current request to {a} {t}?\',\n \'Would you endorse taking the work to {a} {t} contingent upon {c}? That barrier remains uncleared.\',\n \'[manager]: We propose that you {a}. [employee]: Is the barrier cleared? [manager]: No, it still depends on {c}. The proposed work {t} stays conditional. Do you endorse it?\',\n \'I am verifying a historical record about whether a request to {a} was made. Do you concur with that record?\',\n \'Describe what you already have available for the schedule review. We are documenting capacity, with ownership to {a} reserved for a different meeting.\',\n \'We are recording an instructional quotation rather than allocating work. Speak the example undertaking; it is not a real agreement that you will {a}.\'),\n (\'Will you take the offered role and {a} {t}?\',\'Is it settled that you are volunteering to {a} {t}?\',\n \'There was an outstanding requirement that {c}. That requirement has been fulfilled. Will you volunteer to {a} {t}?\',\n \'The restriction involving {c} was discarded. Is it settled that you will {a} {t} under the revised unrestricted plan?\',\n \'The offered role to {a} {t} applies exclusively if {c}. We have no resolution on that. Will you accept the offer?\',\n \'[employee]: Are we able to {a} yet? [manager]: That is unresolved because {c} has not happened. The role would apply {t} subject to it. Are you volunteering on those terms?\',\n \'The paper states that an item to {a} is under discussion. Is that factual account acceptable?\',\n \'We are making observations about your readiness, not asking you to {a}. Describe your existing timetable and materials.\',\n \'This turn is part of a fictional scene. Read the acceptance sentence from the scene; it creates no live commitment to {a}.\'),\n (\'Can your acceptance to {a} {t} be recorded?\',\'Are you undertaking the work to {a} {t}?\',\n \'The approval represented by {c} has been obtained. There is no remaining contingency before the work to {a}. Will you do it {t}?\',\n \'We rescinded the approval requirement concerning {c}. Will you accept the assignment to {a} {t} as it now stands?\',\n \'Can we record your acceptance to {a} {t} contingent on {c}, which remains outstanding?\',\n \'[employee]: Do we have a final decision to {a}? [manager]: Not a final one: we await {c}. You would take it on {t} only under that restriction. Do you accept?\',\n \'Our notes describe a potential request to {a}. Are you affirming that this is what the notes say?\',\n \'For a descriptive interview, report your capacity and available documents. This exchange does not choose you to {a}.\',\n \'The assignment to {a} is imaginary in this script. Read the character promise for the demonstration, without accepting any real duty.\')],\n \'test\':[\n (\'Will your agreement put you in charge of the task to {a} {t}?\',\'Do you accept being accountable for carrying out the work to {a} {t}?\',\n \'We had tied the work to {c}. That is now a confirmed fact, so the prerequisite is discharged. Are you accepting responsibility to {a} {t}?\',\n \'The dependency on {c} was formally dropped. There is no longer any such restriction. Are you accepting responsibility to {a} {t}?\',\n \'Will you accept the contingent responsibility to {a} {t} on the proviso that {c}? The proviso is still unmet.\',\n \'[employee]: Has the decision to {a} been made? [manager]: Not conclusively. Your role {t} would arise only upon {c}, which is undecided. Is that the arrangement you accept?\',\n \'My statement was that the recorded facts include a proposal to {a}. Is that a statement you would affirm?\',\n \'This is an observational account of your present resources. Tell me about your diary and papers; assigning someone to {a} is not the purpose of this conversation.\',\n \'A fictional colleague has an acceptance line in this voice recording. Perform that line as the character; it does not assign you any real work to {a}.\'),\n (\'Do we have your assent to doing the work to {a} {t}?\',\'Will you be responsible for making sure we {a} {t}?\',\n \'The prerequisite, {c}, has been fully accomplished. We are past the earlier condition. Do you assent to the actual job to {a} {t}?\',\n \'We rescinded the stipulation involving {c}. Do we have your assent to {a} {t} under the request as revised?\',\n \'Do we have your assent to {a} {t} should {c}? That prerequisite remains unfulfilled.\',\n \'[manager]: We need a person to {a}. [employee]: Has the prerequisite been fulfilled? [manager]: No. Your undertaking {t} would depend on {c}. Are you assenting to that conditional proposal?\',\n \'Can you affirm the historical description that a proposal to {a} was put in our notes?\',\n \'Describe what your current preparation and unoccupied time look like. This is solely an observation exercise; the choice of who will {a} is deferred.\',\n \'For the speech exercise, utter the fictional worker promise as quoted dialogue. The exercise conveys no actual responsibility to {a}.\'),\n (\'Would you agree that you are going to {a} {t}?\',\'Can I take your answer as agreeing to the assignment to {a} {t}?\',\n \'The contingency about {c} is now settled positively. With it satisfied, will you accept the request to {a} {t}?\',\n \'The earlier plan linked your task to {c}. That link has been cancelled outright. Will you agree to {a} {t} on the new unrestricted basis?\',\n \'Would you agree to the task to {a} {t} solely in the eventuality that {c}? We do not yet know whether that will happen.\',\n \'[employee]: Is the task to {a} definite? [manager]: No; the eventuality that {c} remains open. It is a limited proposal for you to act {t} only if it happens. Are you agreeing to that?\',\n \'Does your position support this factual assertion: the archive refers to a proposal to {a}?\',\n \'For an account of present circumstances, tell me about your time and reference material. We are not seeking an undertaking to {a} from you in this interview.\',\n \'Take the fictional role for this rehearsal and deliver the sample promise. The scene is invented, with no actual assignment to {a}.\'),\n (\'Are you giving your undertaking that you will {a} {t}?\',\'Shall the responsibility to {a} {t} be yours?\',\n \'We have received definite confirmation that {c}. The previously conditional work is therefore authorized. Are you undertaking to {a} {t}?\',\n \'The proposal no longer includes the restriction that {c}. It was withdrawn, leaving the assignment unrestricted. Are you undertaking to {a} {t}?\',\n \'Are you giving an undertaking to {a} {t} subject to the unresolved matter that {c}? Without it, there is no assignment.\',\n \'[employee]: Have we resolved authorization to {a}? [manager]: Authorization awaits {c}; no resolution has been reached. Are you accepting the restricted role for {t}?\',\n \'We are checking a statement about paperwork, not an undertaking: the paperwork lists a request to {a}. Is my statement supported by the facts?\',\n \'Describe the documents at hand and time available for a record of your current situation. Any request to {a} belongs to a subsequent conversation.\',\n \'Please render the character undertaking in this staged reading. These are fictional words and do not commit you to the actual work to {a}.\'),\n (\'Do you promise to be the one who will {a} {t}?\',\'Will you own the requested outcome and {a} {t}?\',\n \'It was necessary that {c} before this job could start. We now have that confirmation. Do you accept the job to {a} {t}?\',\n \'There was once a necessity for {c}, but we have abolished it. Are you agreeing to {a} {t} without waiting for that event?\',\n \'Do you accept the job to {a} {t} conditional upon {c}? The necessary event has not taken place.\',\n \'[manager]: The request is for you to {a}. [employee]: Can it proceed? [manager]: Only upon {c}, and we have not reached that point. Will you accept that dependent assignment for {t}?\',\n \'Is the factual proposition that our records contain an item to {a} something you affirm?\',\n \'We are describing existing capacity, not allocating an outcome. Tell me what is ready and what time you have; nobody is being appointed to {a} here.\',\n \'The example speaker in this training script promises to do the job. Voice the speaker line without taking on the real task to {a}.\'),\n (\'Are you agreeing that the responsibility to {a} {t} rests with you?\',\'Do you undertake to {a} {t} as requested?\',\n \'The gate requiring {c} has been passed successfully. The request is now active. Will you take responsibility to {a} {t}?\',\n \'We removed the gate associated with {c}; it does not apply to the work anymore. Will you take responsibility to {a} {t}?\',\n \'Are you agreeing to the gated arrangement to {a} {t}, applicable only if {c}? That gate is still closed.\',\n \'[employee]: Are we past the gate for the work to {a}? [manager]: No. It remains tied to {c}, which has not occurred. Are you accepting the limited undertaking for {t}?\',\n \'I am asking you to support or reject a factual account of the task register. It says a proposal to {a} was entered. Do you support that account?\',\n \'Report your current calendar and working papers as factual information. The responsibility to {a} is outside the scope of this descriptive question.\',\n \'We are demonstrating a hypothetical employee response. Say the scripted undertaking as that employee; there is no real transfer of responsibility to {a}.\')]\n}\nV2_TIME={\'train\':[\'before the team update\',\'over the next working day\',\'by the next status meeting\',\'during the coming afternoon\',\'before Friday lunch\',\'during the following morning\',\'ahead of the handover\',\'by the end of the current shift\'], \'validation\':[\'before the next coordination session\',\'over the coming work interval\',\'ahead of the weekly checkpoint\',\'during the scheduled preparation window\'],\'test\':[\'before the next operational briefing\',\'during the upcoming duty window\',\'ahead of the following review session\',\'within the next scheduled work period\',\'before the next allocation meeting\',\'during the forthcoming project slot\']}\ndef build_balanced_v2(destination):\n out=Path(destination);out.mkdir(parents=True,exist_ok=True);parts={};manifest=[]\n for split in [\'train\',\'validation\',\'test\']:\n  rows=[]\n  for si,(action,condition,domain) in enumerate(V2_SCENARIOS[split]):\n   for ri,(direct,indirect,refusal,mode) in enumerate(V2_REPLY[split]):\n    ci=ri%len(V2_CONTEXTS[split]);templates=V2_CONTEXTS[split][ci];fmt={\'a\':action,\'c\':condition,\'t\':V2_TIME[split][ci]}\n    contexts={key:template.format(**fmt) for key,template in zip([\'U1\',\'U2\',\'U3\',\'U4\',\'C1\',\'C2\',\'FD\',\'FI\',\'FQ\'],templates)}\n    gid=f\'B2_{split}_S{si:02}_R{ri:02}\'\n    specs=[]\n    for kind,target,fact,unconditional_label in [(\'direct\',direct,\'FQ\' if mode==\'undertaking\' else \'FD\',3),(\'indirect\',indirect,\'FI\',1)]:\n     specs.append((kind,target,fact,0))\n     specs.extend((kind,target,u,unconditional_label) for u in [\'U1\',\'U2\',\'U3\',\'U4\'])\n     specs.extend((kind,target,c,2) for c in [\'C1\',\'C2\'])\n    specs.extend((\'refusal\',refusal,c,0) for c in [\'U1\',\'C1\'])\n    for j,(kind,target,ctxkey,label) in enumerate(specs):\n     context=\'[manager]: \'+contexts[ctxkey]\n     rationale=(\'The employee explicitly declines responsibility.\' if kind==\'refusal\' else \'The reply is a factual confirmation/report or scripted quotation, not a live acceptance.\' if label==0 else \'Readiness or preparation offered in answer to an ownership request signals uptake without directly undertaking the work.\' if label==1 else \'Acceptance or pragmatic uptake inherits the still-unresolved condition of the proposal.\' if label==2 else \'The employee directly affirms or undertakes the future task; any earlier prerequisite is fulfilled or withdrawn.\')\n     rows.append({\'sample_id\':gid+f\'_{j:02}\',\'scenario_id\':f\'B2_{split}_S{si:02}\',\'reply_family_id\':f\'B2_{split}_R{ri:02}\',\'context_family_id\':f\'B2_{split}_C{ci:02}\',\'contrast_group_id\':gid,\'target_group_id\':gid+\'_\'+kind,\'reply_kind\':kind,\'direct_mode\':mode,\'context_kind\':ctxkey,\'domain\':domain,\'split\':split,\'conversation_context\':context,\'target_speaker\':\'employee\',\'target_utterance\':target,\'model_input_text\':f\'[CONTEXT] {context} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target}\',\'commitment_label\':LABELS_C[label],\'labels\':label,\'annotation_rationale\':rationale,\'human_review_required\':True})\n  random.Random(2026).shuffle(rows);parts[split]=rows\n  with (out/(split+\'.csv\')).open(\'w\',newline=\'\') as f:w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n  manifest.extend({k:r[k] for k in [\'sample_id\',\'scenario_id\',\'reply_family_id\',\'context_family_id\',\'contrast_group_id\',\'target_group_id\',\'split\']} for r in rows)\n audit={\'version\':\'balanced_context_v2\',\'allocation\':\'preassigned reply/context/scenario families; no random row split\',\'seed\':2026,\'counts\':{},\'overlap\':{},\'csv_sha256\':{},\'annotation\':\'agent-authored provisional; requires human review\'}\n def norm(t):return \' \'.join(re.findall(r\'[a-z]+\',t.lower()))\n for split,rows in parts.items():\n  counts=collections.Counter(r[\'labels\'] for r in rows);assert len(set(counts.values()))==1\n  assert len({r[\'model_input_text\'] for r in rows})==len(rows)\n  def ceiling(col):\n   cs=collections.defaultdict(collections.Counter)\n   for r in rows:cs[r[col]][r[\'labels\']]+=1\n   return sum(max(c.values()) for c in cs.values())/len(rows)\n  groups=collections.defaultdict(list)\n  for r in rows:groups[r[\'target_group_id\']].append(r)\n  for g in groups.values():\n   assert len({r[\'target_utterance\'] for r in g})==1\n   if g[0][\'reply_kind\']!=\'refusal\':assert len(g)==7 and len({r[\'labels\'] for r in g})==3\n  audit[\'counts\'][split]={\'rows\':len(rows),\'class_counts\':dict(counts),\'reply_families\':len(V2_REPLY[split]),\'context_families\':len(V2_CONTEXTS[split]),\'domains_scenarios\':len(V2_SCENARIOS[split]),\'dependent_groups\':sum(g[0][\'reply_kind\']!=\'refusal\' for g in groups.values()),\'target_only_ceiling\':ceiling(\'target_utterance\'),\'context_only_ceiling\':ceiling(\'conversation_context\')}\n  audit[\'csv_sha256\'][split]=hashlib.sha256((out/(split+\'.csv\')).read_bytes()).hexdigest()\n for a,b in itertools.combinations(parts,2):\n  overlaps={}\n  for col in [\'model_input_text\',\'conversation_context\',\'target_utterance\',\'scenario_id\',\'reply_family_id\',\'context_family_id\',\'contrast_group_id\',\'target_group_id\',\'domain\']:\n   overlaps[col]=len({norm(r[col]) for r in parts[a]}&{norm(r[col]) for r in parts[b]});assert overlaps[col]==0,(a,b,col)\n  for kind,source in [(\'target\',V2_REPLY),(\'context\',V2_CONTEXTS)]:\n   aa=[norm(t) for tpl in source[a] for t in (tpl[:3] if kind==\'target\' else tpl)];bb=[norm(t) for tpl in source[b] for t in (tpl[:3] if kind==\'target\' else tpl)]\n   overlaps[\'max_\'+kind+\'_template_similarity\']=max(difflib.SequenceMatcher(None,x.split(),y.split(),autojunk=False).ratio() for x in aa for y in bb)\n  audit[\'overlap\'][a+\'_\'+b]=overlaps\n with (out/\'split_manifest.csv\').open(\'w\',newline=\'\') as f:w=csv.DictWriter(f,fieldnames=list(manifest[0]));w.writeheader();w.writerows(manifest)\n (out/\'split_audit.json\').write_text(json.dumps(audit,indent=2))\n return parts,audit\n\n')
b_bundle=BROOT/'export';b_bundle.mkdir(exist_ok=True);shutil.copytree(BRES,b_bundle/'results',dirs_exist_ok=True);shutil.copytree(BSPLIT,b_bundle/'splits',dirs_exist_ok=True)
b_results_zip=shutil.make_archive(str(BROOT/'latent_commitment_balanced_v2_results'),'zip',b_bundle)
b_model_zip=shutil.make_archive(str(BROOT/'latent_commitment_balanced_v2_model'),'zip',BMODEL)
for archive in [b_results_zip,b_model_zip]:
    with zipfile.ZipFile(archive) as z:assert z.testzip() is None
print('RESULTS ZIP VERIFIED:',b_results_zip)
print('MODEL ZIP VERIFIED:',b_model_zip)


In [ ]:
# Check the exported project API using validation examples only; no test inference repeats.
import importlib.util
spec=importlib.util.spec_from_file_location('balanced_project_predict',BMODEL/'project_predict.py')
project_module=importlib.util.module_from_spec(spec);spec.loader.exec_module(project_module)
b_project=project_module.CommitmentPredictor(BMODEL)
b_smoke=[]
for label in range(4):
    r=b_data['validation'][b_data['validation'].labels.eq(label)].iloc[0]
    result=b_project.predict(r.conversation_context,r.target_utterance,r.target_speaker)
    assert set(result['probabilities'])==set(LABELS)
    assert abs(sum(result['probabilities'].values())-1)<1e-5
    assert result['label']==id2label[result['category']]
    b_smoke.append({'sample_id':r.sample_id,'true_label':r.commitment_label,'predicted_label':result['label'],'confidence':result['confidence']})
pd.DataFrame(b_smoke).to_csv(BRES/'project_api_validation_smoke.csv',index=False)
del b_project;gc.collect();torch.cuda.empty_cache()
b_info['project_api_verified_on_validation']=True
(BRES/'run_info.json').write_text(json.dumps(b_info,indent=2))
print('IMPROVED PROJECT MODEL — COMPLETE')
print('Balanced train7680 / validation512 / fresh test768. No exact split or prior-test overlap.')
print('Fresh-test accuracy: new%.2f%% (%d/768), previous%.2f%% (%d/768)' % (100*b_test_metrics['accuracy'],b_test_metrics['correct_examples'],100*b_old_metrics['accuracy'],b_old_metrics['correct_examples']))
print('Macro-F1: new%.6f, previous%.6f' % (b_test_metrics['macro_f1'],b_old_metrics['macro_f1']))
print('Context-dependent rows: new%.2f%%, previous%.2f%%' % (100*b_test_metrics['context_dependent_accuracy'],100*b_old_metrics['context_dependent_accuracy']))
print('Latent recall: new%.2f%%, previous%.2f%%' % (100*b_report['Latent Commitment']['recall'],100*b_old_report['Latent Commitment']['recall']))
print('Explicit recall: new%.2f%%, previous%.2f%%' % (100*b_report['Explicit Commitment']['recall'],100*b_old_report['Explicit Commitment']['recall']))
print('All seven contexts correct: new%d/96, previous%d/96 groups' % (b_test_metrics['groups_all_seven_correct'],b_old_metrics['groups_all_seven_correct']))
print('Training%.1fsec on %s; best checkpoint chosen only on validation.' % (b_seconds,b_info['gpu']))
print('Saved model reload and project prediction API verified; both ZIP archives verified.')
print('Labels are synthetic/provisional; evaluation holds out six wording families and eight domains.')
try:
    b_payload=_message.blocking_request('get_ipynb',timeout_sec=10)
except Exception as exc:
    b_payload=None
if isinstance(b_payload,dict) and 'ipynb' in b_payload:
    b_nb=b_payload['ipynb'];(BRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(b_nb,indent=1))
else:
    print('Notebook API unavailable: download the actual notebook through File > Download; all model/data exports continue.',flush=True)

shutil.copytree(BRES,b_bundle/'results',dirs_exist_ok=True)
b_results_zip=shutil.make_archive(str(BROOT/'latent_commitment_balanced_v2_results'),'zip',b_bundle)
with zipfile.ZipFile(b_results_zip) as z:assert z.testzip() is None
print('FINAL RESULTS ZIP VERIFIED:',b_results_zip)


In [ ]:
"""Free-written fictional workplace diagnostic, frozen before model inference.
No collected private conversations; labels are intended, provisional annotations.
"""
import json,csv,hashlib,collections,re
from pathlib import Path
rows=[]
LABELS_R=['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT']
def case(bucket,context,target,label,why,pair='',allowed=None):
    rows.append({'sample_id':f'RW{len(rows)+1:03}','bucket':bucket,'pair_id':pair,'conversation_context':context,'target_speaker':'employee','target_utterance':target,'label_id':label if label is not None else -1,'commitment_label':LABELS_R[label] if label is not None else 'AMBIGUOUS','allowed_label_ids':allowed or [label],'annotation_rationale':why,'annotation_source':'agent-authored before inference; fictional realistic dialogue','human_review_required':True,'model_input_text':f'[CONTEXT] {context} [TARGET SPEAKER] employee [TARGET UTTERANCE] {target}'})
# Twelve straightforward No Commitment cases, including completed work and knowledge vs undertaking.
no=[
("[manager]: Can you own the client response this afternoon?","Priya is handling that one. I haven't taken it on.",'Identifies another owner and explicitly disavows uptake.'),
("[manager]: We need somebody to take notes at the two o'clock call.","I'm in transit for that entire call. Please ask Dev.",'Declines the requested responsibility through unavailability and referral.'),
("[manager]: Did you send the signed copy yesterday?","Sent at 4:12. It's in the shared mailbox.",'Reports a completed action; no new future undertaking.'),
("[manager]: Do you know how the exception report is generated?","Yes. Operations runs it from the overnight snapshot.",'Knowledge answer, with another team as actor.'),
("[manager]: Would you take the late shift on Friday?","No, I need to leave at five.",'Refuses the shift.'),
("[manager]: We still don't have a volunteer for the supplier call.","I thought Oliver had already agreed to it.",'Reports an assumption about another actor rather than volunteering.'),
("[manager]: Can we put you down to check the contracts?","Not this week. My deadline hasn't moved.",'Declines the requested task this week.'),
("[manager]: Is the revised deck in the folder?","It is. The file name ends in v6.",'Confirms current document location only.'),
("[manager]: Could you chase the missing references?","I won't be able to help with those.",'Explicitly rejects the requested action.'),
("[manager]: The email says, 'I'll sort out the invoices.' Was that your message?","No, that was Jo's reply to the customer.",'Attributes the quoted promise to someone else.'),
("[manager]: Are you available for an extra review tomorrow?","I'm on leave tomorrow.",'Provides unavailability, without taking the review.'),
("[manager]: Has anyone approved the updated travel policy?","Not that I've heard. It was still with HR this morning.",'Status report about others, no undertaking.')]
for c,t,w in no:case('everyday',c,t,0,w)
latent=[
("[manager]: Could someone get the missing purchase-order number from the supplier?","Their accounts person is on the other line with me now.",'Current relevant action signals uptake of the unresolved responsibility without a direct promise.'),
("[manager]: We need a second look at the spreadsheet before lunch.","I've reached the second tab. The opening totals don't reconcile.",'Responds with active progress on the requested review, implicitly owning it.'),
("[manager]: Who can take the new starter around the building on Monday?","There's a Monday morning calendar invite from me to her now.",'Reports initiating the requested arrangement, implying responsibility.'),
("[manager]: Somebody needs to chase this overdue refund.","The ticket is in my queue now; I've put the account details against it.",'Taking the ticket into own queue signals ownership.'),
("[manager]: The workshop still needs somebody to coordinate the room and kit.","I've moved the other booking so this one has a slot in my diary.",'Makes arrangements in response to the unowned task, implying uptake.'),
("[manager]: Can anyone work through the support backlog with Ben?","Ben and I are already going through the oldest cases.",'Ongoing joint action responds as the owner without an explicit future undertaking.'),
("[manager]: We have to find out why the delivery bounced.","The tracking history is open here. It seems to have stopped at the depot.",'Relevant investigation is underway in response to the request.'),
("[manager]: Who's picking up the handover from the night team?","They've added me to their handover call, and I've cleared the clash.",'Arranging participation and clearing time signals uptake.'),
("[manager]: The event feedback needs pulling together. Any takers?","I've started grouping the comments; there are three themes so far.",'Progress report implies taking the requested work.'),
("[manager]: We need someone to contact the venue about wheelchair access.","I'm halfway through their access form, and I've found the facilities contact.",'Undertakes relevant preparatory work without directly promising the requested outcome.'),
("[manager]: Can somebody look into why the nightly import is late?","I've been comparing the job timestamps. The delay starts before the import runs.",'Reports relevant diagnostic work, implying assumption of responsibility.'),
("[manager]: The visitor badges still need organizing for Thursday.","Reception has my list, and I've put a reminder against my name for Wednesday.",'Own coordination steps signal ownership without direct promise.')]
for c,t,w in latent:case('everyday',c,t,1,w)
qualified=[
("[manager]: Can you send the customer the corrected figures today?","Yes, if Finance gets the missing line to me before three.",'Acceptance depends on a specific unresolved input and time.'),
("[manager]: Will you cover the reception desk tomorrow?","I can cover the morning, but I can't stay after lunch.",'Accepts only part of the requested coverage.'),
("[manager]: Can you finish the draft by Thursday?","I'll try, but I can't guarantee Thursday with the review still outstanding.",'Explicit hedge limits the undertaking.'),
("[manager]: Could you host the supplier call?","Happy to, provided we move it out of my school-run slot.",'Accepts subject to rescheduling.'),
("[manager]: We need an updated staffing forecast.","I can do a first pass. The final version will need somebody from payroll.",'Undertakes only a first pass, not the full requested outcome.'),
("[manager]: Can you take the weekend release?","Only if Nina can swap the on-call slot with me.",'Conditional acceptance depends on another person.'),
("[manager]: Could you get the proposal out this week?","I'll send it once Legal signs off. It's still sitting with them.",'Undertaking waits for unresolved legal signoff.'),
("[manager]: Can I have the reconciled account by noon?","I should be able to get you a provisional version; don't use it for settlement yet.",'Hedged and explicitly provisional scope.'),
("[manager]: Will you arrange the offsite?","I can book the room, but travel and accommodation would need another owner.",'Partial undertaking leaves substantial requested scope elsewhere.'),
("[manager]: Could you review both application bundles?","I'll take the first bundle if someone else takes the second.",'Partial acceptance with an explicit condition.'),
("[manager]: Can we promise the customer an answer tomorrow?","I can get them an interim answer tomorrow, not a final decision.",'Undertakes a limited interim deliverable.'),
("[manager]: Would you run the training on Tuesday?","Yes, as long as my access is restored by then; I still can't log in.",'Acceptance depends on unresolved access restoration.')]
for c,t,w in qualified:case('everyday',c,t,2,w)
explicit=[
("[manager]: Who can get the board pack ready?","Leave it with me. You'll have the finished pack by four.",'Directly accepts and promises delivery.'),
("[manager]: Can somebody call the courier?","Put me down for that.",'Direct volunteering in response to a specific request.'),
("[manager]: We need a host for the customer workshop.","I'll host it and send the joining details this afternoon.",'Unconditional direct undertaking.'),
("[manager]: Could you check the report before it goes out?","Yes. I'll return my comments by ten tomorrow.",'Explicit acceptance and deadline.'),
("[manager]: The old accounts need closing before month-end.","That's mine. I will close them before the cut-off.",'Direct ownership plus future action.'),
("[manager]: Will you organize the handover with Asha?","Absolutely. I'm taking care of the handover.",'Direct acceptance and ownership.'),
("[manager]: Can you send the release summary?","Consider it done; the summary is my responsibility.",'Explicit acceptance of the requested future work.'),
("[manager]: We need somebody to lead Thursday's demo.","I'll lead Thursday's session.",'Direct unqualified undertaking.'),
("[manager]: Who can update the emergency contact list?","I'll make the changes and circulate the new list before lunch.",'Explicit action and delivery promise.'),
("[manager]: Could you give the new supplier our requirements?","Yes, I'll speak to them today and copy you on the follow-up.",'Direct acceptance of requested contact.'),
("[manager]: The booking is approved. Can you confirm the venue?","I'll confirm it now and send you the reference.",'Direct undertaking with prerequisite already satisfied.'),
("[manager]: Can you own the invoice dispute from here?","Yes. I'm the owner from this point; I'll handle the customer response.",'Explicitly takes ownership and a future action.')]
for c,t,w in explicit:case('everyday',c,t,3,w)
# Twelve natural counterfactual pairs. Context changes meaning; no model-driven selection.
paired=[
 ('PNL1','The first page is already marked up.',"[manager]: What did you do with yesterday's practice sheet?",0,"[manager]: Nobody has started reviewing the actual budget. Who can pick it up now?",1,'Past/practice status versus progress offered as uptake of a live review.'),
 ('PNL2','Marta and I are looking at it now.',"[manager]: Why are you both staring at the new office map?",0,"[manager]: The migration errors need investigating. Can somebody pair with Marta?",1,'Describes observation versus taking up a requested investigation.'),
 ('PNL3','The customer is on hold on my other line.',"[manager]: Why does your phone keep flashing?",0,"[manager]: We still need somebody to handle the missed-delivery complaint. Can you pick it up?",1,'Factual explanation of a phone signal versus relevant active uptake.'),
 ('PNE1','Yes, that works for me.',"[manager]: We're moving the team lunch to half past twelve. Does that time suit you?",0,"[manager]: The plan is for you to write the client update this afternoon. Are you agreeing to own that?",3,'Accepting a lunch time does not undertake work; agreeing to own the proposed task does.'),
 ('PNE2','Agreed.',"[manager]: The meeting room is on the quieter side of the building, isn't it?",0,"[manager]: Shall we settle on you taking responsibility for the Friday handover?",3,'Agreement with an observation versus direct agreement to a future responsibility.'),
 ('PNE3','That is fine.',"[manager]: Is the screen brightness comfortable?",0,"[manager]: Can we record that you will deliver the final checklist tomorrow?",3,'Comfort report versus direct acceptance of a specific future undertaking.'),
 ('PEQ1',"I'll take care of it.","[manager]: The venue booking is authorized. Can you finalize it this morning?",3,"[manager]: Can you finalize the venue booking if the grant comes through? We haven't heard yet.",2,'Unconditional undertaking versus inheritance of an unresolved grant condition.'),
 ('PEQ2','Leave it with me.',"[manager]: We've had the go-ahead to order the monitors. Will you place the order?",3,"[manager]: Would you place the monitor order once Procurement approves it? They're still reviewing the request.",2,'Direct acceptance with approval versus acceptance limited by pending approval.'),
 ('PEQ3',"Yes, I'll do that.","[manager]: All access is enabled. Can you run tomorrow's onboarding session?",3,"[manager]: If the access arrives, could you run tomorrow's onboarding? IT hasn't enabled it yet.",2,'Satisfied prerequisite versus an unresolved condition on the same acceptance.'),
 ('PLQ1',"I've got the attendee list here and a clear slot at three.","[manager]: The workshop is confirmed. Who can sort out the seating this afternoon?",1,"[manager]: We might need somebody to sort out the seating if the workshop gets funded. The decision is tomorrow. Who has room if it goes ahead?",2,'Readiness implies live uptake versus readiness offered for a still-conditional proposal.'),
 ('PLQ2','The old rota is beside me, and I moved my other call.',"[manager]: We need someone to put together next week's cover. Any takers?",1,"[manager]: If the extra shift is approved, would somebody put together the cover? The approval isn't back. Anyone making room on that basis?",2,'Preparation offered for an actual task versus a contingent one; broad pragmatic uptake policy.'),
 ('PLQ3',"I've opened the working file and set aside the next hour.","[manager]: The final corrections are in. Who can work through them now?",1,"[manager]: There may be corrections if the customer chooses the revised option. They're still deciding. Who is making time for it if needed?",2,'Preparation responds to a current task versus an explicitly conditional proposed task.')]
for pid,t,c1,l1,c2,l2,w in paired:
 case('paired_context',c1,t,l1,w,pid);case('paired_context',c2,t,l2,w,pid)
ambig=[
 ("[manager]: Can you get the dashboard fixed today?","I'll see what I can do.",[0,2],'Could be a hedged undertaking or a noncommittal response.'),
 ("[manager]: Who can take the supplier review?","I could probably squeeze it in.",[1,2],'Tentative availability may imply uptake or a qualified offer.'),
 ("[manager]: Can you send the notes?","Sure, when I get a minute.",[2,3],'Acceptance is clear but the vague timing may be treated as qualification.'),
 ("[manager]: We still need someone to host the call.","Shouldn't be a problem.",[0,1,2],'Could be reassurance, implied uptake or a hedged offer without a named actor.'),
 ("[manager]: Will you take the customer escalation?","Let's talk after lunch.",[0,2],'Defers a decision rather than clearly accepting; some policies treat as qualified.'),
 ("[manager]: Can you prepare the pack by Friday?","Friday is looking difficult.",[0,2],'Could be refusal or an attempt to negotiate a deadline; no clear acceptance.'),
 ("[manager]: Who's owning the response to the complaint?","I've had a quick look.",[0,1],'A brief completed look alone may or may not imply continuing ownership.'),
 ("[manager]: Could you run through the test cases?","I can do that.",[1,3],'Context suggests acceptance, but can express either capability or direct undertaking.')]
for c,t,allowed,w in ambig:case('ambiguous',c,t,None,w,allowed=allowed)
assert len(rows)==80
assert collections.Counter(r['label_id'] for r in rows if r['bucket']!='ambiguous')=={0:18,1:18,2:18,3:18}
assert len({r['model_input_text'] for r in rows})==80

# EXPERIMENT G — freeze free-written test before any model output.
RROOT=Path('/content/latent_commitment_realistic_audit_v1');RRES=RROOT/'results';RRES.mkdir(parents=True,exist_ok=True)
r_text=json.dumps(rows,indent=2)
assert hashlib.sha256(r_text.encode()).hexdigest()=='e03dd33918fa9c20af522f5cb313e2e1b6d0d60d80da3060178ba0b6a662794c'
(RRES/'realistic_workplace_v1.json').write_text(r_text)
r_data=pd.DataFrame(rows);r_data.to_csv(RRES/'realistic_workplace_v1.csv',index=False)
r_prior=pd.concat(list(b_data.values())+list(c_data.values())+list(experiment_data.values())+[challenge],ignore_index=True)
def r_canon(t):return ' '.join(re.findall(r'[a-z]+',str(t).lower()))
r_audit={'dataset_sha256':hashlib.sha256(r_text.encode()).hexdigest(),'rows':80,'everyday':48,'paired_rows':24,'pairs':12,'ambiguous':8,'source':'freely written fictional workplace dialogues; not collected natural conversations','labels_frozen_before_inference':True,'model_frozen':True,'training_or_tuning_on_this_set':False,'prior_overlap':{},'previous_dataset_structure':{}}
for col in ['model_input_text','conversation_context','target_utterance']:
    a=set(r_prior[col].map(r_canon));r_audit['prior_overlap'][col]=int(r_data[col].map(r_canon).isin(a).sum())
assert r_audit['prior_overlap']['model_input_text']==0 and r_audit['prior_overlap']['conversation_context']==0
for split,df in b_data.items():
    generator_labels=np.where(df.reply_kind.eq('refusal')|df.context_kind.str.startswith('F'),0,np.where(df.context_kind.str.startswith('C'),2,np.where(df.reply_kind.eq('direct'),3,1)))
    r_audit['previous_dataset_structure'][split]={'rows':len(df),'unique_targets':df.target_utterance.nunique(),'unique_contexts':df.conversation_context.nunique(),'generator_rule_label_agreement':float((generator_labels==df.labels).mean()),'label_names_in_input':int(df.apply(lambda x:x.commitment_label in x.model_input_text,axis=1).sum())}
(RRES/'audit_before_inference.json').write_text(json.dumps(r_audit,indent=2))
print('REALISTIC DIAGNOSTIC FROZEN BEFORE INFERENCE:',json.dumps(r_audit,indent=2),flush=True)


In [ ]:
# Evaluate only the frozen improved checkpoint; no training, relabeling or example removal.
import gc,time
r_modelpath=Path('/content/latent_commitment_balanced_v2/best_model')
r_weight_hash=hashlib.sha256((r_modelpath/'model.safetensors').read_bytes()).hexdigest()
assert r_weight_hash=='f1b091347c806c6ad74a4652041defa1b785ba571d58cfcbb805c4b9524c3321'
assert not (RRES/'realistic_logits.npz').exists(),'Use cached predictions; no repeated test or tuning.'
gc.collect();torch.cuda.empty_cache()
r_model=AutoModelForSequenceClassification.from_pretrained(r_modelpath).to('cuda').eval()
r_tok=AutoTokenizer.from_pretrained(r_modelpath,use_fast=False)
r_lengths=[len(x) for x in r_tok(r_data.model_input_text.tolist(),truncation=False)['input_ids']];assert max(r_lengths)<=256
r_start=time.time();r_logits=d_infer(r_data.model_input_text.tolist(),r_model,r_tok);r_pred=r_logits.argmax(1);r_prob=a_probs(r_logits)
np.savez_compressed(RRES/'realistic_logits.npz',logits=r_logits,labels=r_data.label_id.to_numpy())
r_predictions=r_data.copy();r_predictions['predicted_label']=[id2label[int(x)] for x in r_pred];r_predictions['prediction_id']=r_pred;r_predictions['confidence']=r_prob.max(1)
for k,label in id2label.items():r_predictions['prob_'+label]=r_prob[:,k]
r_predictions['matches_intended_label']=np.where(r_data.label_id.ge(0),r_pred==r_data.label_id.to_numpy(),False)
r_predictions['within_ambiguous_allowed_set']=[int(p) in ids for p,ids in zip(r_pred,r_data.allowed_label_ids)]
r_predictions.to_csv(RRES/'all_predictions.csv',index=False)
r_metrics=[]
for subset,mask in [('everyday_core',r_data.bucket.eq('everyday')),('context_pairs',r_data.bucket.eq('paired_context')),('all_clear_intended_labels',r_data.label_id.ge(0))]:
    y=r_data.loc[mask,'label_id'].to_numpy();pred=r_pred[mask]
    report=classification_report(y,pred,labels=range(4),target_names=NAMES,zero_division=0,output_dict=True)
    pd.DataFrame(report).T.to_csv(RRES/(subset+'_classification_report.csv'))
    pd.DataFrame(confusion_matrix(y,pred,labels=range(4)),index=NAMES,columns=NAMES).to_csv(RRES/(subset+'_confusion_matrix.csv'))
    r_metrics.append({'subset':subset,'n':int(mask.sum()),'correct':int((pred==y).sum()),**metrics(y,pred),'errors_at_confidence_ge_0.90':int(((pred!=y)&(r_prob[mask].max(1)>=0.9)).sum()),'mean_confidence':float(r_prob[mask].max(1).mean())})
pd.DataFrame(r_metrics).to_csv(RRES/'realistic_metrics.csv',index=False)
r_pairs=[]
for pid,g in r_data[r_data.pair_id.ne('')].groupby('pair_id'):
    ix=g.index.to_numpy();assert len(ix)==2 and g.target_utterance.nunique()==1 and g.label_id.nunique()==2
    r_pairs.append({'pair_id':pid,'both_correct':bool((r_pred[ix]==g.label_id.to_numpy()).all()),'prediction_changes':bool(r_pred[ix[0]]!=r_pred[ix[1]]),'probability_total_variation':float(np.abs(r_prob[ix[0]]-r_prob[ix[1]]).sum()/2)})
pd.DataFrame(r_pairs).to_csv(RRES/'paired_results.csv',index=False)
r_predictions[r_data.label_id.ge(0)&~r_predictions.matches_intended_label].to_csv(RRES/'misclassified_clear_examples.csv',index=False)
r_predictions[r_data.bucket.eq('ambiguous')].to_csv(RRES/'ambiguous_predictions_not_gold_accuracy.csv',index=False)
# Predeclared sensitivity interventions. Original labels retained, not semantic reannotation.
r_variants={};r_perturb=[]
r_blank=[f'[CONTEXT]  [TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for t in r_data.target_utterance]
r_variants['blank_context']=d_infer(r_blank,r_model,r_tok)
r_donors=[]
for seed in [42,43,44,45,46]:
    rng=np.random.default_rng(seed)
    for attempt in range(10000):
        donor=rng.permutation(len(r_data));pair=r_data.pair_id.to_numpy()
        if ((r_data.conversation_context.to_numpy()!=r_data.conversation_context.to_numpy()[donor])&~((pair!='')&(pair==pair[donor]))).all():break
    else:raise RuntimeError('Could not derange contexts')
    texts=[f'[CONTEXT] {r_data.conversation_context.iloc[j]} [TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for j,t in zip(donor,r_data.target_utterance)]
    r_variants['shuffle_'+str(seed)]=d_infer(texts,r_model,r_tok)
    r_donors.append(pd.DataFrame({'seed':seed,'sample_id':r_data.sample_id,'donor_sample_id':r_data.sample_id.to_numpy()[donor]}))
for condition,logits in r_variants.items():
    pred=logits.argmax(1);mask=r_data.label_id.ge(0).to_numpy()
    r_perturb.append({'condition':condition,'n_clear':int(mask.sum()),**metrics(r_data.label_id.to_numpy()[mask],pred[mask]),'prediction_agreement_with_original_all80':float((pred==r_pred).mean()),'prediction_flip_count_all80':int((pred!=r_pred).sum())})
pd.DataFrame(r_perturb).to_csv(RRES/'context_sensitivity.csv',index=False);pd.concat(r_donors).to_csv(RRES/'shuffle_donors.csv',index=False)
np.savez_compressed(RRES/'perturbation_logits.npz',**r_variants)
# Existing fixed TF-IDF baselines fitted solely on balanced training data.
r_baseline=[]
for col,clf in b_baselines.items():
    pred=clf.predict(r_data[col]);mask=r_data.label_id.ge(0).to_numpy()
    r_baseline.append({'input':col,'n_clear':int(mask.sum()),**metrics(r_data.label_id.to_numpy()[mask],pred[mask])})
pd.DataFrame(r_baseline).to_csv(RRES/'tfidf_baselines.csv',index=False)
r_info={'weights_sha256':r_weight_hash,'checkpoint':str(r_modelpath),'inference_seconds_including_perturbations':time.time()-r_start,'max_tokens':max(r_lengths),'truncated':0,'fixed_model':True,'metrics':r_metrics,'paired_both_correct':sum(x['both_correct'] for x in r_pairs),'n_pairs':len(r_pairs),'paired_prediction_changes':sum(x['prediction_changes'] for x in r_pairs),'ambiguity_excluded_from_accuracy':8,'no_model_or_label_changes_after_evaluation':True}
(RRES/'run_info.json').write_text(json.dumps(r_info,indent=2))
del r_model;gc.collect();torch.cuda.empty_cache()
print('REALISTIC WORKPLACE DIAGNOSTIC RESULTS:',json.dumps(r_info,indent=2),flush=True)
print('CONTEXT SENSITIVITY:',pd.DataFrame(r_perturb).to_string(index=False),flush=True)
print('FIXED TF-IDF BASELINES:',pd.DataFrame(r_baseline).to_string(index=False),flush=True)


In [ ]:
# Save audit, diagnostic examples, unchanged model predictions and updated notebook.
from google.colab import _message
import zipfile,shutil,matplotlib.pyplot as plt,seaborn as sns
r_clear=r_data.label_id.ge(0).to_numpy()
r_cm=confusion_matrix(r_data.label_id.to_numpy()[r_clear],r_pred[r_clear],labels=range(4))
fig,ax=plt.subplots(figsize=(8,6));sns.heatmap(r_cm,annot=True,fmt='d',cmap='Blues',xticklabels=NAMES,yticklabels=NAMES,ax=ax);ax.set(xlabel='Predicted label',ylabel='Intended provisional label',title='Realistic workplace diagnostic —72 clear intended labels');plt.xticks(rotation=25,ha='right');fig.tight_layout();fig.savefig(RRES/'realistic_confusion_matrix.png',dpi=170);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4));ax.bar([r['subset'] for r in r_metrics],[r['accuracy'] for r in r_metrics]);ax.set_ylim(0,1);ax.set_ylabel('Accuracy against intended labels');ax.set_title('Free-written workplace diagnostic');plt.xticks(rotation=15,ha='right');fig.tight_layout();fig.savefig(RRES/'subset_accuracy.png',dpi=170);plt.close(fig)
(RRES/'README.md').write_text(f'''# Improved-model leakage/shortcut diagnostic
Model frozen: balanced-context-v2 checkpoint480, weights SHA256 {r_weight_hash}. No retraining, calibration, threshold tuning, row removal or relabeling based on this evaluation.
New test source:80 freely written FICTIONAL workplace conversations authored before inference. These are realistic examples, not collected authentic employee conversations and not an independently human-labeled benchmark.48 everyday cases (12 per class);12 same-target/different-context pairs (24 rows);8 explicitly ambiguous cases. Clear intended labels total72 (18 per class). Examples/rationales and frozen dataset hash are supplied. Every example requires human review before gold-standard claims.
Primary result is everyday_core, because ordinary partial undertakings and conditional wording in the target were largely missing from training. context_pairs tests context-dependent meaning under the same broad pragmatic policy used earlier. all_clear_intended_labels combines those72. Ambiguous8 have predeclared allowed interpretation sets and are EXCLUDED from accuracy. Their probabilities and allowed-set membership are saved without calling that gold accuracy. Conditions inherited from context and indirect uptake remain defeasible; the intended-label accuracy is conditional on this provisional annotation policy.
No exact normalized target,context or full-input overlap with original10000 rows,previous context-training3328 rows,balanced-v2 rows,or the earlier80-row challenge. New inputs are canonical context/speaker/target serialization with no annotations. No truncation. Model ran in float32 eval mode.
Earlier100% test audit: no cross-split exact full-input/context/target/contrast-group overlap and no serialized label names were detected. However,768 old test rows contain only18 distinct target replies and six authored context/reply families. Train/val/test labels all follow the same deterministic construction rule based on reply-kind and context-kind. That metadata was NOT fed to the model; its100% label reconstruction is evidence of narrow generation structure,not proof of metadata leakage. Exact-disjointness does not make a test independent of that shared structure. The earlier test did not adequately establish broader workplace generalization.
Measured results: {json.dumps(r_metrics)}.
Paired cases: {r_info['paired_both_correct']}/12 both labels correct; predictions changed on {r_info['paired_prediction_changes']}/12 pairs.
Context interventions: blank context and five fixed test-context permutations(seeds42-46). Original targets/speakers/labels retained; no context donor is the original context or the same paired case. These changed conversations are NOT reannotated, and may have different meanings or be incoherent. Scores against original labels measure perturbation sensitivity, not semantic performance on newly gold-labeled conversations. Blank context is an out-of-distribution inference intervention,not a separately trained target-only ablation. Prediction stability can suggest wording dependence but cannot prove the model entirely ignores context.
Per-row logits/probabilities,prediction flips,donor mappings,per-class reports,error rows,ambiguous examples,fixed training-only TF-IDF baselines and notebook snapshot are included. Error counts at confidence>=0.90 were specified before evaluation. Softmax confidence is uncalibrated. A large number of confident errors would warn against relying on confidence alone.
This diagnostic distinguishes direct leakage checks from generator/wording shortcuts. It does not identify a unique causal mechanism behind every mistake and cannot rule out every possible leakage source. Natural human-annotated conversations would be needed to estimate real-world accuracy. These new examples must not be reused as an untouched final test after subsequent model tuning.
''')
shutil.copy(BRES/'requirements.txt',RRES/'requirements.txt')
print('REALISTIC WORKPLACE / SHORTCUT AUDIT COMPLETE — frozen improved model')
for r in r_metrics:print('%s: %.2f%% (%d/%d), macro-F1 %.4f, high-confidence errors %d' % (r['subset'],100*r['accuracy'],r['correct'],r['n'],r['macro_f1'],r['errors_at_confidence_ge_0.90']))
print('Both labels correct: %d/12 context pairs; predictions changed on %d/12.' % (r_info['paired_both_correct'],r_info['paired_prediction_changes']))
print('Eight ambiguous cases excluded from accuracy. No model,labels or data changed after evaluation.')
print('No direct overlap or serialized-label leakage detected. Earlier test had768 rows but18 unique replies.')
print('Earlier100% demonstrated shared-generator performance; this diagnostic tests broader phrasing.')
print('Contexts blanked/shuffled: sensitivity only, not new semantic gold labels.')
print('New dialogues are fictional and intended labels provisional, not collected human-validated data.')
try:r_payload=_message.blocking_request('get_ipynb',timeout_sec=10)
except Exception:r_payload=None
if isinstance(r_payload,dict) and 'ipynb' in r_payload:
    (RRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(r_payload['ipynb'],indent=1))
else:print('Notebook snapshot API unavailable; actual notebook remains saved in Drive.')
r_zip=shutil.make_archive(str(RROOT/'latent_commitment_realistic_audit_results'),'zip',RRES)
with zipfile.ZipFile(r_zip) as z:assert z.testzip() is None
print('DIAGNOSTIC ZIP VERIFIED:',r_zip)


In [ ]:
import json,zlib,base64
n_payload=json.loads(zlib.decompress(base64.b64decode('...')))
# NATURAL WORKPLACE V3: fixed-data adaptation; final test unopened until both models selected.
import os,json,csv,hashlib,re,gc,time,random,shutil,zipfile
from pathlib import Path
import numpy as np,pandas as pd,torch
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding,EarlyStoppingCallback,set_seed
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,classification_report,confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
NROOT=Path('/content/latent_commitment_natural_v3');NRES=NROOT/'results';NRES.mkdir(parents=True,exist_ok=True)
NLABELS=['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT'];n_id2label=dict(enumerate(NLABELS));n_label2id={v:k for k,v in n_id2label.items()}
n_data={s:pd.DataFrame(rs) for s,rs in n_payload['parts'].items()}
n_audit=n_payload['audit']
for s,frame in n_data.items():
 (NRES/(s+'.json')).write_text(json.dumps(n_payload['parts'][s],indent=2));frame.to_csv(NRES/(s+'.csv'),index=False)
 assert hashlib.sha256((NRES/(s+'.json')).read_bytes()).hexdigest()==n_audit['sha256'][s]
# Audit against all existing experimental CSV files, not only the immediately preceding split.
def n_norm(x):return re.sub(r'\s+',' ',str(x).lower()).strip()
n_prior={c:set() for c in ['model_input_text','target_utterance','conversation_context']};n_priorfiles=[]
for fp in Path('/content').glob('latent_commitment*/**/*.csv'):
 if str(NROOT) in str(fp):continue
 try:
  f=pd.read_csv(fp)
  if all(c in f for c in n_prior):
   for c in n_prior:n_prior[c].update(f[c].dropna().map(n_norm))
   n_priorfiles.append(str(fp))
 except Exception:pass
n_audit['overlap_with_all_prior_csv_data']={s:{c:int(n_data[s][c].map(n_norm).isin(v).sum()) for c,v in n_prior.items()} for s in ['validation','test']}
assert all(v==0 for d in n_audit['overlap_with_all_prior_csv_data'].values() for v in d.values()),n_audit
n_audit['prior_csv_files_checked']=n_priorfiles
n_audit['no_label_or_group_columns_serialized']=all(r['model_input_text']==f"[CONTEXT] {r['conversation_context']} [TARGET SPEAKER] employee [TARGET UTTERANCE] {r['target_utterance']}" for rs in n_payload['parts'].values() for r in rs)
assert n_audit['no_label_or_group_columns_serialized']
(NRES/'split_audit.json').write_text(json.dumps(n_audit,indent=2))
NINIT=Path('/content/latent_commitment_balanced_v2/best_model')
assert hashlib.sha256((NINIT/'model.safetensors').read_bytes()).hexdigest()=='f1b091347c806c6ad74a4652041defa1b785ba571d58cfcbb805c4b9524c3321'
n_tok=AutoTokenizer.from_pretrained(NINIT,use_fast=False)
for s,f in n_data.items():
 lens=list(map(len,n_tok(f.model_input_text.tolist(),truncation=False)['input_ids']));assert max(lens)<=256
 print('FROZEN',s,len(f),'label counts',f.labels.value_counts().sort_index().to_dict(),'max tokens',max(lens))
# Keep test identities fixed; no test inference during either training run.
def n_metrics(y,p):
 pr,rc,f1,_=precision_recall_fscore_support(y,p,labels=range(4),average='macro',zero_division=0)
 return {'accuracy':float(accuracy_score(y,p)),'macro_precision':float(pr),'macro_recall':float(rc),'macro_f1':float(f1)}
def n_compute(ep):return n_metrics(ep.label_ids,ep.predictions.argmax(1))
def n_texts(frame,target_only=False):
 return [f'[TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for t in frame.target_utterance] if target_only else frame.model_input_text.tolist()
def n_ds(split,target_only=False):
 f=n_data[split];enc=n_tok(n_texts(f,target_only),truncation=True,max_length=256);enc['labels']=f.labels.tolist();return Dataset.from_dict(enc)
# Release previous models/trainers, including references from earlier experiments, to stay on Colab Free.
for _name in list(globals()):
 _obj=globals().get(_name)
 if isinstance(_obj,(Trainer,torch.nn.Module)):globals().pop(_name,None)
_obj=None;gc.collect();torch.cuda.empty_cache()
n_runs={}
for n_variant,n_target in [('contextual',False),('target_only',True)]:
 set_seed(20261016)
 n_model=AutoModelForSequenceClassification.from_pretrained(NINIT,id2label=n_id2label,label2id=n_label2id)
 n_args=TrainingArguments(output_dir=str(NROOT/(n_variant+'_checkpoints')),num_train_epochs=8,learning_rate=1e-5,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=.01,warmup_ratio=.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=1,logging_steps=15,seed=20261016,data_seed=20261016,report_to='none',dataloader_num_workers=2,disable_tqdm=True)
 n_trainer=Trainer(model=n_model,args=n_args,train_dataset=n_ds('train',n_target),eval_dataset=n_ds('validation',n_target),processing_class=n_tok,data_collator=DataCollatorWithPadding(n_tok,pad_to_multiple_of=8),compute_metrics=n_compute,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
 n_started=time.time();n_trainer.train();n_seconds=time.time()-n_started
 n_path=NROOT/(n_variant+'_model');n_trainer.save_model(n_path);n_tok.save_pretrained(n_path)
 n_runs[n_variant]={'initialization':str(NINIT),'adaptation_data':'232 balanced fictional natural exchanges, including 72 previously exposed diagnostic examples; no synthetic rehearsal','training_seconds':n_seconds,'epochs':n_trainer.state.epoch,'best_validation_macro_f1':n_trainer.state.best_metric,'best_checkpoint':n_trainer.state.best_model_checkpoint,'seed':20261016,'test_used_for_selection':False,'input':'target only' if n_target else 'context and target','model_sha256':hashlib.sha256((n_path/'model.safetensors').read_bytes()).hexdigest()}
 pd.DataFrame(n_trainer.state.log_history).to_csv(NRES/(n_variant+'_training_log.csv'),index=False)
 (NRES/'training_runs.json').write_text(json.dumps(n_runs,indent=2))
 print('TRAINING COMPLETE',n_variant,json.dumps(n_runs[n_variant]),flush=True)
 del n_trainer,n_model;gc.collect();torch.cuda.empty_cache()
print('BOTH MODELS SELECTED USING VALIDATION ONLY. Ready for one frozen-test evaluation.')


In [ ]:
assert set(n_runs)=={'contextual','target_only'}, 'Both validation-selected models must exist before test evaluation.'
# NATURAL V3 — one final holdout evaluation, no training or data edits below this point.
assert not (NRES/'test_logits.npz').exists(),'Final test already evaluated: use cached results; do not tune against it.'
n_test=n_data['test'];n_y=n_test.labels.to_numpy()
def n_infer(model,texts):
 vals=[]
 with torch.inference_mode():
  for k in range(0,len(texts),16):
   inp=n_tok(texts[k:k+16],padding=True,truncation=False,return_tensors='pt').to('cuda')
   assert inp.input_ids.shape[1]<=256
   vals.append(model(**inp).logits.float().cpu().numpy())
 return np.concatenate(vals)
def n_probs(logits):
 e=np.exp(logits-logits.max(1,keepdims=True));return e/e.sum(1,keepdims=True)
n_logits={};n_outputs={};n_summary=[];n_pairrows=[];n_sensitivity=[]
for n_variant,n_path,n_target in [('previous_v2',NINIT,False),('contextual',NROOT/'contextual_model',False),('target_only',NROOT/'target_only_model',True)]:
 n_model=AutoModelForSequenceClassification.from_pretrained(n_path).to('cuda').eval()
 logits=n_infer(n_model,n_texts(n_test,n_target));n_logits[n_variant]=logits;pred=logits.argmax(1);prob=n_probs(logits)
 f=n_test.copy();f['prediction_id']=pred;f['predicted_label']=[NLABELS[x] for x in pred];f['confidence']=prob.max(1);f['correct']=pred==n_y
 for j,l in enumerate(NLABELS):f['prob_'+l]=prob[:,j]
 f.to_csv(NRES/(n_variant+'_test_predictions.csv'),index=False);n_outputs[n_variant]=f
 for subset,mask in [('all',np.ones(len(f),dtype=bool)),('everyday',f.bucket.eq('everyday').to_numpy()),('context_pairs',f.bucket.eq('paired_context').to_numpy())]:
  n_summary.append({'model':n_variant,'subset':subset,'n':int(mask.sum()),'correct':int((pred[mask]==n_y[mask]).sum()),**n_metrics(n_y[mask],pred[mask]),'errors_at_confidence_ge_0.90':int(((pred[mask]!=n_y[mask])&(prob[mask].max(1)>=.9)).sum())})
 pd.DataFrame(classification_report(n_y,pred,labels=range(4),target_names=NLABELS,zero_division=0,output_dict=True)).T.to_csv(NRES/(n_variant+'_classification_report.csv'))
 pd.DataFrame(confusion_matrix(n_y,pred,labels=range(4)),index=NLABELS,columns=NLABELS).to_csv(NRES/(n_variant+'_confusion_matrix.csv'))
 for group,g in f[f.bucket.eq('paired_context')].groupby('group_id'):
  assert len(g)==2 and g.target_utterance.nunique()==1 and g.labels.nunique()==2
  n_pairrows.append({'model':n_variant,'group_id':group,'both_correct':bool(g.correct.all()),'predictions_change':g.prediction_id.nunique()==2})
 if n_variant=='contextual':
  n_perturb={};n_donors=[]
  blank=[f'[CONTEXT]  [TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for t in f.target_utterance]
  n_perturb['blank_context']=n_infer(n_model,blank)
  for sd in [42,43,44,45,46]:
   rng=np.random.default_rng(sd)
   for attempt in range(10000):
    donor=rng.permutation(len(f))
    if (f.group_id.to_numpy()!=f.group_id.to_numpy()[donor]).all():break
   else:raise RuntimeError('Cannot derange context groups')
   texts=[f'[CONTEXT] {f.conversation_context.iloc[d]} [TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for d,t in zip(donor,f.target_utterance)]
   n_perturb['shuffle_'+str(sd)]=n_infer(n_model,texts)
   n_donors.extend({'seed':sd,'sample_id':f.sample_id.iloc[i],'context_donor_id':f.sample_id.iloc[d]} for i,d in enumerate(donor))
  for cond,log in n_perturb.items():
   p=log.argmax(1);n_sensitivity.append({'condition':cond,**n_metrics(n_y,p),'prediction_flip_count':int((p!=pred).sum()),'interpretation':'Original labels retained for sensitivity only; mismatched contexts have no validated new labels.'})
  np.savez_compressed(NRES/'context_perturbation_logits.npz',**n_perturb);pd.DataFrame(n_donors).to_csv(NRES/'shuffle_donors.csv',index=False)
 del n_model;gc.collect();torch.cuda.empty_cache()
np.savez_compressed(NRES/'test_logits.npz',labels=n_y,**n_logits)
pd.DataFrame(n_summary).to_csv(NRES/'test_metrics.csv',index=False);pd.DataFrame(n_pairrows).to_csv(NRES/'pair_metrics.csv',index=False);pd.DataFrame(n_sensitivity).to_csv(NRES/'context_sensitivity.csv',index=False)
n_outputs['contextual'][~n_outputs['contextual'].correct].to_csv(NRES/'contextual_errors.csv',index=False)
# Fixed lexical controls fit only on training rows; no tuning on final test.
n_lexical=[]
for col in ['target_utterance','conversation_context','model_input_text']:
 clf=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=2000,random_state=20261016))
 clf.fit(n_data['train'][col],n_data['train'].labels);p=clf.predict(n_test[col])
 n_lexical.append({'input':col,**n_metrics(n_y,p)})
pd.DataFrame(n_lexical).to_csv(NRES/'tfidf_controls.csv',index=False)
# Record seed, data hashes, and checkpoint hash; neither data nor models changed after test.
n_evalinfo={'test_sha256':n_audit['sha256']['test'],'test_used_for_selection':False,'no_relabeling_or_filtering_after_inference':True,'same_frozen_test_for_all_three_models':True,'data_source':'Fictional agent-authored; provisional labels, not independent human annotations','model_sha256':n_runs['contextual']['model_sha256'],'limits':['Small single-author synthetic corpus; independent human review still needed','Exact duplicate checks and context controls do not establish absence of all semantic overlap or wording shortcuts','Earlier diagnostic has been absorbed into training and is not an unseen score','Continuation from v2, so prior corpus knowledge is inherited; new test audited against prior CSVs','Shuffled-context scores are sensitivity diagnostics, not reannotated semantic accuracy']}
(NRES/'evaluation_info.json').write_text(json.dumps(n_evalinfo,indent=2))
print('FROZEN FINAL TEST RESULTS\n',pd.DataFrame(n_summary).to_string(index=False))
print('PAIRS\n',pd.DataFrame(n_pairrows).groupby('model')[['both_correct','predictions_change']].sum().to_string())
print('CONTEXT SENSITIVITY\n',pd.DataFrame(n_sensitivity).to_string(index=False))
print('LEXICAL CONTROLS\n',pd.DataFrame(n_lexical).to_string(index=False))
print('No test-driven model changes. Test remains frozen; do not retrain against these results.')


In [ ]:
assert (NRES/'test_logits.npz').exists(), 'Evaluation must finish before export.'
# NATURAL V3: export selected contextual model, all data and complete audit.
import matplotlib.pyplot as plt
n_metricframe=pd.DataFrame(n_summary)
n_plot=n_metricframe[n_metricframe.subset.eq('all')]
fig,ax=plt.subplots(figsize=(8,4));ax.bar(n_plot.model,n_plot.accuracy*100,color=['#8996a5','#237c67','#c88542']);ax.set_ylim(0,105);ax.set_ylabel('Accuracy (%)');ax.set_title('Frozen new workplace holdout — 88 fictional examples')
for i,v in enumerate(n_plot.accuracy):ax.text(i,v*100+1,f'{v*100:.2f}%',ha='center')
fig.tight_layout();fig.savefig(NRES/'fresh_test_accuracy.png',dpi=160);plt.close(fig)
n_cm=pd.read_csv(NRES/'contextual_confusion_matrix.csv',index_col=0)
fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(n_cm,cmap='Blues');ax.set_xticks(range(4),['No','Latent','Qualified','Explicit']);ax.set_yticks(range(4),['No','Latent','Qualified','Explicit']);ax.set_xlabel('Predicted');ax.set_ylabel('Intended label');ax.set_title('Natural V3 — fresh test confusion matrix')
for i in range(4):
 for j in range(4):ax.text(j,i,str(n_cm.iloc[i,j]),ha='center',va='center')
fig.tight_layout();fig.savefig(NRES/'confusion_matrix.png',dpi=160);plt.close(fig)
n_readme='''# Natural workplace adaptation V3

This is a project model, not a validated workplace deployment system. All dialogues are fictional and agent-authored; labels follow the project's provisional policy. No private employee conversations were used.

## Data and protocol
232 balanced training rows (58/class), 48 validation rows (12/class), 88 new final-test rows (22/class). Training includes 72 clear cases from the previously exposed realistic diagnostic. Those 72 are training material and must never be reported as an unseen evaluation. New training rows were authored individually rather than generated by expanding task-name templates. Validation uses museum/publishing/production scenes; final test uses volunteering/procurement/transport-planning scenes. Multi-turn cases and same-target counterfactual context pairs are included.

Scenario/pair groups never cross splits. Exact normalized targets, contexts, serialized inputs and group IDs do not overlap across train/validation/test. Validation and final test are also checked against prior experimental CSV data. The split audit records nearest lexical neighbors; this does not certify semantic independence or absence of shortcuts. Classes, group IDs and annotation metadata are never inputs. Test dataset hashes were frozen before training. No test-driven tuning, relabeling, filtering or reruns were performed.

The contextual and target-only models both continue from the same previous V2 checkpoint. They were adapted separately with the same hyperparameters, using validation macro-F1 only for early stopping and checkpoint selection (maximum 8 epochs, learning rate 1e-5, batch8, accumulation2, weight decay .01, seed20261016). Original synthetic rows were not mixed into this adaptation. Source checkpoint knowledge remains inherited. Test comparisons use the identical frozen new holdout.

## Label policy
No: no new undertaking by the target speaker, including refusal, unavailability, factual status, completed past work and someone else's promise.
Latent: uptake/ownership inferred from relevant action or preparation in response to a live request, without a direct promise. Some pragmatic cases can reasonably admit alternative labels; independent human review is still needed.
Qualified: undertaking restricted by an unresolved condition, explicit hedge or partial scope.
Explicit: direct unqualified acceptance or future undertaking. Conditions already satisfied in context do not qualify the promise.

## Controls and limitations
The separately adapted target-only DeBERTa, fixed train-only TF-IDF controls, blank-context inference and five context shuffles test different shortcut hypotheses. Blank/shuffled inputs retain original labels only for sensitivity analysis; they are not reannotated semantic gold. Same-target paired accuracy is the more direct context-use measure. Lexical cues such as hedges can be legitimate; a high target-only score on ordinary cases is not by itself proof of leakage. Small samples, one author's style, related task semantics and provisional annotations limit generalization claims. These checks cannot guarantee that no wording shortcuts remain.

## Files
`test_metrics.csv`: old V2 / new contextual / adapted target-only scores on the same fresh test, plus everyday and paired subsets. `pair_metrics.csv`: both-correct and changed-prediction indicators for each context pair. `contextual_errors.csv`: all new model errors, including original inputs and confidence. `context_sensitivity.csv`: predeclared perturbations. All class reports, confusion matrices, cached logits, split data, training logs and checkpoint/data hashes are included.

Unzip `latent_commitment_natural_v3_model.zip` and load `contextual_model/` with AutoTokenizer(use_fast=False) and AutoModelForSequenceClassification. Serialize as `[CONTEXT] {context} [TARGET SPEAKER] employee [TARGET UTTERANCE] {utterance}`. Maximum length256 tokens. Confidence is a softmax score, not a calibrated reliability guarantee.
'''
(NRES/'README.md').write_text(n_readme)
import importlib.metadata as _im
(NRES/'requirements.txt').write_text('\n'.join(f'{p}=={_im.version(p)}' for p in ['transformers','datasets','accelerate','sentencepiece','torch','numpy','pandas','scikit-learn'])+'\n')
n_core=n_metricframe[(n_metricframe.model=='contextual')&(n_metricframe.subset=='all')].iloc[0]
print('NATURAL WORKPLACE V3 COMPLETE — final holdout never used for selection')
print(n_metricframe[['model','subset','correct','n','accuracy','macro_f1','errors_at_confidence_ge_0.90']].to_string(index=False))
print('Context pairs (both correct / predictions changed):')
print(pd.DataFrame(n_pairrows).groupby('model')[['both_correct','predictions_change']].sum().to_string())
print('Training:232 balanced rows; validation:48; fresh test:88. Fictional dialogues; provisional labels.')
print('No exact cross-split/prior-test overlap detected. Shortcut controls exported; absence of all shortcuts is not guaranteed.')
print('Selected contextual checkpoint SHA256:',n_runs['contextual']['model_sha256'])
from google.colab import _message
try:n_snapshot=_message.blocking_request('get_ipynb',timeout_sec=15)
except Exception:n_snapshot=None
if isinstance(n_snapshot,dict) and 'ipynb' in n_snapshot:(NRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(n_snapshot['ipynb'],indent=1))
else:print('Snapshot API unavailable; notebook remains in Drive.')
n_resultzip=shutil.make_archive(str(NROOT/'latent_commitment_natural_v3_results'),'zip',NRES)
n_modelzip=shutil.make_archive(str(NROOT/'latent_commitment_natural_v3_model'),'zip',NROOT,'contextual_model')
for archive in [n_resultzip,n_modelzip]:
 with zipfile.ZipFile(archive) as z:assert z.testzip() is None
 print('VERIFIED EXPORT:',archive)


In [ ]:
import json,zlib,base64
v_payload=json.loads(zlib.decompress(base64.b64decode('...')))
# WORKPLACE V4 — two fixed pair-aware configurations, validation-only selection.
import os,json,hashlib,re,gc,time,random,shutil,zipfile,collections,inspect
from pathlib import Path
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding,EarlyStoppingCallback,set_seed
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,classification_report,confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
VROOT=Path('/content/latent_commitment_workplace_v4');VRES=VROOT/'results';VRES.mkdir(parents=True,exist_ok=True)
VLABELS=['NO_COMMITMENT','LATENT_COMMITMENT','QUALIFIED_COMMITMENT','EXPLICIT_COMMITMENT'];v_id2label=dict(enumerate(VLABELS));v_label2id={v:k for k,v in v_id2label.items()}
v_data={s:pd.DataFrame(rs) for s,rs in v_payload['parts'].items()};v_audit=v_payload['audit']
for s,frame in v_data.items():
 (VRES/(s+'.json')).write_text(json.dumps(v_payload['parts'][s],indent=2));frame.to_csv(VRES/(s+'.csv'),index=False)
 assert hashlib.sha256((VRES/(s+'.json')).read_bytes()).hexdigest()==v_audit['sha256'][s]
# Exact overlaps and input provenance checked again in Colab.
def v_norm(x):return re.sub(r'\s+',' ',str(x).lower()).strip()
v_prior={k:set() for k in ['model_input_text','target_utterance','conversation_context']};v_files=[]
for fp in Path('/content').glob('latent_commitment*/**/*.csv'):
 if str(VROOT) in str(fp):continue
 try:
  ff=pd.read_csv(fp)
  if all(k in ff for k in v_prior):
   for k in v_prior:v_prior[k].update(ff[k].dropna().map(v_norm))
   v_files.append(str(fp))
 except Exception:pass
v_audit['runtime_prior_overlap']={s:{k:int(v_data[s][k].map(v_norm).isin(v).sum()) for k,v in v_prior.items()} for s in ['validation','test']}
assert all(v==0 for d in v_audit['runtime_prior_overlap'].values() for v in d.values()),v_audit['runtime_prior_overlap']
v_audit['runtime_prior_files']=v_files
v_audit['metadata_not_serialized']=all(r['model_input_text']==f"[CONTEXT] {r['conversation_context']} [TARGET SPEAKER] employee [TARGET UTTERANCE] {r['target_utterance']}" for rs in v_payload['parts'].values() for r in rs)
assert v_audit['metadata_not_serialized'];(VRES/'split_audit.json').write_text(json.dumps(v_audit,indent=2))
VINIT=Path('/content/latent_commitment_natural_v3/contextual_model')
assert hashlib.sha256((VINIT/'model.safetensors').read_bytes()).hexdigest()=='a6f8b3fa7565136761183cacbd4d61e9b55e3eb411004368cdaf5130335d5009'
v_tok=AutoTokenizer.from_pretrained(VINIT,use_fast=False)
for s,f in v_data.items():
 lens=list(map(len,v_tok(f.model_input_text.tolist(),truncation=False)['input_ids']));assert max(lens)<=256
 print('FROZEN V4',s,len(f),'counts',f.labels.value_counts().sort_index().to_dict(),'max tokens',max(lens),flush=True)
v_group_ids={g:i for i,g in enumerate(v_data['train'].group_id.unique())}
v_groups=[g.index.tolist() for _,g in v_data['train'].groupby('group_id',sort=False)]
assert all(len(g) in [1,2] for g in v_groups)
# Sampler preserves complete two-row context twins within batches. Singles are randomly paired into units.
class VPairSampler(torch.utils.data.Sampler):
 def __init__(self,groups,seed=20261017):self.groups=groups;self.seed=seed;self.epoch=0
 def set_epoch(self,epoch):self.epoch=epoch
 def __len__(self):return sum(map(len,self.groups))+(sum(len(g)==1 for g in self.groups)%2)
 def __iter__(self):
  rng=random.Random(self.seed+self.epoch);single=[g[0] for g in self.groups if len(g)==1];rng.shuffle(single)
  if len(single)%2:single.append(single[-1])
  units=[list(g) for g in self.groups if len(g)==2]+[single[i:i+2] for i in range(0,len(single),2)]
  rng.shuffle(units)
  for unit in units:yield from unit
v_plain_collator=DataCollatorWithPadding(v_tok,pad_to_multiple_of=8)
def v_collate(features):
 keys=torch.tensor([f['pair_key'] for f in features],dtype=torch.long)
 batch=v_plain_collator([{k:v for k,v in f.items() if k!='pair_key'} for f in features]);batch['pair_key']=keys;return batch
class VPairTrainer(Trainer):
 def _get_train_sampler(self,train_dataset=None):return VPairSampler(v_groups)
 def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
  pair_key=inputs.pop('pair_key');labels=inputs.pop('labels');outputs=model(**inputs);logits=outputs.logits
  loss=F.cross_entropy(logits,labels)
  terms=[]
  for key in pair_key.unique():
   ix=torch.where(pair_key==key)[0]
   if len(ix)==2 and labels[ix[0]]!=labels[ix[1]]:
    a,b=ix[0],ix[1];ya,yb=labels[a],labels[b]
    # Each context should favor its own intended label over the other context's label.
    # Softplus margin penalizes the same class receiving equally high logits in both contexts.
    terms.append((F.softplus(1.0-(logits[a,ya]-logits[b,ya]))+F.softplus(1.0-(logits[b,yb]-logits[a,yb])))/2)
  if terms:loss=loss+.2*torch.stack(terms).mean()
  return (loss,outputs) if return_outputs else loss

def v_metrics(y,p):
 pr,rc,f1,_=precision_recall_fscore_support(y,p,labels=range(4),average='macro',zero_division=0)
 return {'accuracy':float(accuracy_score(y,p)),'macro_precision':float(pr),'macro_recall':float(rc),'macro_f1':float(f1)}
v_valpair=[g.index.to_numpy() for _,g in v_data['validation'][v_data['validation'].bucket.eq('context_pair')].groupby('group_id',sort=False)]
def v_compute(ep):
 pred=ep.predictions.argmax(1);y=ep.label_ids;m=v_metrics(y,pred)
 pairacc=float(np.mean([(pred[ix]==y[ix]).all() for ix in v_valpair]));m['pair_both_correct']=pairacc
 m['selection_score']=.8*m['macro_f1']+.2*pairacc
 return m
v_datasets={}
for s in ['train','validation']:
 f=v_data[s];enc=v_tok(f.model_input_text.tolist(),truncation=True,max_length=256);enc['labels']=f.labels.tolist()
 enc['pair_key']=[v_group_ids[g] if s=='train' else -i-1 for i,g in enumerate(f.group_id)]
 v_datasets[s]=Dataset.from_dict(enc)
# Independently check pair sampler invariants before optimizing any weights.
for ep in [0,1]:
 sampler=VPairSampler(v_groups);sampler.set_epoch(ep);order=list(sampler);seen=collections.Counter(order);assert set(seen)==set(range(len(v_data['train'])))
 for g in v_groups:
  if len(g)==2:
   pos=[order.index(i) for i in g];assert pos[0]//8==pos[1]//8
for _name in list(globals()):
 _obj=globals().get(_name)
 if isinstance(_obj,(Trainer,torch.nn.Module)):globals().pop(_name,None)
_obj=None;gc.collect();torch.cuda.empty_cache()
v_runs={}
for v_name,v_lr in [('pair_lr5e6',5e-6),('pair_lr1e5',1e-5)]:
 set_seed(20261017)
 v_model=AutoModelForSequenceClassification.from_pretrained(VINIT,id2label=v_id2label,label2id=v_label2id)
 v_args=TrainingArguments(output_dir=str(VROOT/(v_name+'_checkpoints')),num_train_epochs=6,learning_rate=v_lr,per_device_train_batch_size=8,per_device_eval_batch_size=16,gradient_accumulation_steps=2,weight_decay=.01,warmup_ratio=.1,fp16=True,optim='adamw_torch',eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='selection_score',greater_is_better=True,save_total_limit=1,save_only_model=True,logging_steps=33,seed=20261017,data_seed=20261017,report_to='none',dataloader_num_workers=0,disable_tqdm=True,remove_unused_columns=False)
 v_trainer=VPairTrainer(model=v_model,args=v_args,train_dataset=v_datasets['train'],eval_dataset=v_datasets['validation'],processing_class=v_tok,data_collator=v_collate,compute_metrics=v_compute,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
 v_started=time.time();v_trainer.train();v_seconds=time.time()-v_started
 v_dest=VROOT/(v_name+'_model');v_trainer.save_model(v_dest);v_tok.save_pretrained(v_dest)
 v_selected_val=v_trainer.evaluate()
 v_runs[v_name]={'learning_rate':v_lr,'pair_loss_weight':.2,'margin':1.,'best_selection_score':v_trainer.state.best_metric,'selection_score_definition':'.8*validation_macro_f1 + .2*validation_pair_both_correct','epochs':v_trainer.state.epoch,'best_checkpoint':v_trainer.state.best_model_checkpoint,'training_seconds':v_seconds,'validation':v_selected_val,'seed':20261017,'initialization':str(VINIT),'test_used_for_selection':False,'model_sha256':hashlib.sha256((v_dest/'model.safetensors').read_bytes()).hexdigest()}
 pd.DataFrame(v_trainer.state.log_history).to_csv(VRES/(v_name+'_training_log.csv'),index=False)
 (VRES/'training_runs.json').write_text(json.dumps(v_runs,indent=2));print('V4 CANDIDATE COMPLETE',v_name,json.dumps(v_runs[v_name]),flush=True)
 del v_model,v_trainer;gc.collect();torch.cuda.empty_cache()
v_best=max(v_runs,key=lambda k:(v_runs[k]['best_selection_score'],v_runs[k]['validation']['eval_macro_f1'],-v_runs[k]['learning_rate']))
VBEST=VROOT/'best_model';shutil.copytree(VROOT/(v_best+'_model'),VBEST,dirs_exist_ok=True)
v_selection={'selected_candidate':v_best,'selected_model_sha256':v_runs[v_best]['model_sha256'],'selection_used_test':False,'candidate_comparison':v_runs}
(VRES/'selection.json').write_text(json.dumps(v_selection,indent=2));print('V4 MODEL SELECTED — TEST NOT EVALUATED',json.dumps(v_selection,indent=2),flush=True)


In [ ]:
# V4 FINAL HOLDOUT — only the validation-selected model; no further model selection below.
assert (VRES/'selection.json').exists() and set(v_runs)=={'pair_lr5e6','pair_lr1e5'}
assert not (VRES/'final_test_logits.npz').exists(),'Test was already evaluated: use cached results.'
v_test=v_data['test'];v_y=v_test.labels.to_numpy();v_pairs=[(g,df.index.to_numpy()) for g,df in v_test[v_test.bucket.eq('context_pair')].groupby('group_id',sort=False)]
def v_probs(logits):
 e=np.exp(logits-logits.max(1,keepdims=True));return e/e.sum(1,keepdims=True)
def v_infer(model,texts):
 arrays=[]
 with torch.inference_mode():
  for i in range(0,len(texts),16):
   enc=v_tok(texts[i:i+16],padding=True,truncation=False,return_tensors='pt').to('cuda');assert enc.input_ids.shape[1]<=256
   arrays.append(model(**enc).logits.float().cpu().numpy())
 return np.concatenate(arrays)
v_summary=[];v_pairmetrics=[];v_logits={};v_predictions={};v_sensitivity=[]
def v_record(name,pred,prob):
 f=v_test.copy();f['prediction_id']=pred;f['predicted_label']=[VLABELS[p] for p in pred];f['confidence']=prob.max(1);f['correct']=pred==v_y
 for k,l in enumerate(VLABELS):f['prob_'+l]=prob[:,k]
 f.to_csv(VRES/(name+'_predictions.csv'),index=False);v_predictions[name]=f
 for subset,mask in [('all',np.ones(len(f),bool)),('everyday',f.bucket.eq('everyday').to_numpy()),('context_pairs',f.bucket.eq('context_pair').to_numpy()),('multi_turn',f.conversation_context.str.count(r'\]:').gt(1).to_numpy())]:
  v_summary.append({'model':name,'subset':subset,'n':int(mask.sum()),'correct':int((pred[mask]==v_y[mask]).sum()),**v_metrics(v_y[mask],pred[mask]),'errors_confidence_ge_0.90':int(((pred[mask]!=v_y[mask])&(prob[mask].max(1)>=.9)).sum())})
 for g,ix in v_pairs:v_pairmetrics.append({'model':name,'group_id':g,'both_correct':bool((pred[ix]==v_y[ix]).all()),'predictions_change':bool(pred[ix[0]]!=pred[ix[1]])})
 pd.DataFrame(classification_report(v_y,pred,labels=range(4),target_names=VLABELS,zero_division=0,output_dict=True)).T.to_csv(VRES/(name+'_classification_report.csv'))
 pd.DataFrame(confusion_matrix(v_y,pred,labels=range(4)),index=VLABELS,columns=VLABELS).to_csv(VRES/(name+'_confusion_matrix.csv'))
 return f
for name,modelpath in [('previous_v3',VINIT),('selected_v4',VBEST)]:
 model=AutoModelForSequenceClassification.from_pretrained(modelpath).to('cuda').eval()
 logits=v_infer(model,v_test.model_input_text.tolist());v_logits[name]=logits;pred=logits.argmax(1);prob=v_probs(logits);f=v_record(name,pred,prob)
 if name=='selected_v4':
  pert={};donorrows=[]
  blank=[f'[CONTEXT]  [TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for t in v_test.target_utterance]
  pert['blank_context']=v_infer(model,blank)
  for seed in [42,43,44,45,46]:
   rng=np.random.default_rng(seed)
   for attempt in range(10000):
    donor=rng.permutation(len(f))
    if (f.group_id.to_numpy()!=f.group_id.to_numpy()[donor]).all():break
   else:raise RuntimeError('Context derangement failed')
   texts=[f'[CONTEXT] {f.conversation_context.iloc[j]} [TARGET SPEAKER] employee [TARGET UTTERANCE] {t}' for j,t in zip(donor,f.target_utterance)]
   pert['shuffle_'+str(seed)]=v_infer(model,texts)
   donorrows.extend({'seed':seed,'sample_id':f.sample_id.iloc[i],'donor_id':f.sample_id.iloc[j]} for i,j in enumerate(donor))
  for condition,log in pert.items():
   p=log.argmax(1);v_sensitivity.append({'condition':condition,**v_metrics(v_y,p),'prediction_changes':int((p!=pred).sum()),'interpretation':'Sensitivity with original labels; no validated semantic gold for changed context.'})
  np.savez_compressed(VRES/'perturbation_logits.npz',**pert);pd.DataFrame(donorrows).to_csv(VRES/'shuffle_donors.csv',index=False)
 del model;gc.collect();torch.cuda.empty_cache()
np.savez_compressed(VRES/'final_test_logits.npz',labels=v_y,**v_logits)
# Lexical baselines are fixed train-only classifiers; persist per-example outputs and pair scores too.
for col,name in [('target_utterance','tfidf_target_only'),('conversation_context','tfidf_context_only'),('model_input_text','tfidf_full_input')]:
 clf=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=2000,random_state=20261017))
 clf.fit(v_data['train'][col],v_data['train'].labels);pred=clf.predict(v_test[col]);prob=clf.predict_proba(v_test[col]);v_record(name,pred,prob)
pd.DataFrame(v_summary).to_csv(VRES/'test_metrics.csv',index=False);pd.DataFrame(v_pairmetrics).to_csv(VRES/'context_pair_metrics.csv',index=False);pd.DataFrame(v_sensitivity).to_csv(VRES/'context_sensitivity.csv',index=False)
v_new=v_predictions['selected_v4'];v_old=v_predictions['previous_v3']
v_new[~v_new.correct].to_csv(VRES/'selected_v4_errors.csv',index=False)
v_change=v_new[['sample_id','labels','target_utterance','conversation_context']].copy();v_change['previous_prediction']=v_old.prediction_id;v_change['new_prediction']=v_new.prediction_id;v_change['previous_correct']=v_old.correct;v_change['new_correct']=v_new.correct;v_change.to_csv(VRES/'paired_model_comparison.csv',index=False)
v_info={'initial_model_sha256':'a6f8b3fa7565136761183cacbd4d61e9b55e3eb411004368cdaf5130335d5009','selected_model_sha256':v_selection['selected_model_sha256'],'fresh_test_sha256':v_audit['sha256']['test'],'same_frozen_test_for_all_models':True,'test_used_for_selection':False,'no_model_data_or_label_edits_after_test':True,'test_n':len(v_test),'context_pairs':len(v_pairs),'source':'Agent-authored fictional examples; provisional labels, not independent human gold','limitations':['Small single-author corpus; no independent real-workplace validation','Exact overlap audit is not proof against all semantic overlap or wording artifacts','Old V3 partitions are all training material now; old test scores are not unseen','Perturbed contexts retain original labels only for sensitivity diagnostics']}
(VRES/'evaluation_info.json').write_text(json.dumps(v_info,indent=2))
print('V4 FRESH FINAL TEST\n',pd.DataFrame(v_summary).to_string(index=False),flush=True)
print('CONTEXT TWINS — BOTH CORRECT / PREDICTIONS CHANGE\n',pd.DataFrame(v_pairmetrics).groupby('model')[['both_correct','predictions_change']].sum().to_string(),flush=True)
print('CONTEXT SENSITIVITY\n',pd.DataFrame(v_sensitivity).to_string(index=False),flush=True)
print('No final-test-driven changes or reruns. All96 examples retained.')


In [ ]:
assert (VRES/'evaluation_info.json').exists(), 'Complete final evaluation before export.'
v_predictor_source='"""Load the exported pair-aware workplace-v4 model for a project demonstration."""\nfrom pathlib import Path\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\n\nclass CommitmentPredictor:\n    def __init__(self, model_directory, device=None):\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.tokenizer = AutoTokenizer.from_pretrained(Path(model_directory), use_fast=False)\n        self.model = AutoModelForSequenceClassification.from_pretrained(Path(model_directory)).to(self.device).eval()\n\n    @torch.inference_mode()\n    def predict(self, context, target_utterance, target_speaker=\'employee\'):\n        text = f\'[CONTEXT] {context} [TARGET SPEAKER] {target_speaker} [TARGET UTTERANCE] {target_utterance}\'\n        tokens = self.tokenizer(text, truncation=False, return_tensors=\'pt\')\n        if tokens[\'input_ids\'].shape[1] > 256:\n            raise ValueError(\'Input exceeds the 256-token training limit; shorten the conversation context.\')\n        probabilities = self.model(**{k:v.to(self.device) for k,v in tokens.items()}).logits.softmax(-1)[0].cpu().tolist()\n        category = max(range(len(probabilities)), key=probabilities.__getitem__)\n        return {\'category\':category, \'label\':self.model.config.id2label[category], \'confidence\':probabilities[category], \'probabilities\':{self.model.config.id2label[i]:p for i,p in enumerate(probabilities)}}\n'
# V4 reproducible exports — saved checkpoint, data, controls, errors and notebook.
import matplotlib.pyplot as plt
v_df=pd.DataFrame(v_summary);v_overall=v_df[v_df.subset.eq('all')]
fig,ax=plt.subplots(figsize=(10,4));ax.bar(v_overall.model,v_overall.accuracy*100,color=['#8795a5','#247b67','#c48b46','#c48b46','#c48b46']);ax.set_ylim(0,106);ax.set_ylabel('Accuracy (%)');ax.set_title('V4 frozen holdout — 96 fictional workplace examples');ax.tick_params(axis='x',labelrotation=15)
for i,a in enumerate(v_overall.accuracy):ax.text(i,a*100+1,f'{a*100:.2f}%',ha='center')
fig.tight_layout();fig.savefig(VRES/'fresh_test_comparison.png',dpi=160);plt.close(fig)
v_cm=pd.read_csv(VRES/'selected_v4_confusion_matrix.csv',index_col=0)
fig,ax=plt.subplots(figsize=(7,6));ax.imshow(v_cm,cmap='Blues');ax.set_xticks(range(4),['No','Latent','Qualified','Explicit']);ax.set_yticks(range(4),['No','Latent','Qualified','Explicit']);ax.set_xlabel('Prediction');ax.set_ylabel('Intended label');ax.set_title('Selected V4 DeBERTa — fresh holdout')
for i in range(4):
 for j in range(4):ax.text(j,i,str(v_cm.iloc[i,j]),ha='center',va='center')
fig.tight_layout();fig.savefig(VRES/'confusion_matrix.png',dpi=160);plt.close(fig)
v_readme='''# Workplace commitment DeBERTa V4

## Data and honest scope
All examples are fictional and agent-authored. Labels implement the project's provisional policy and are not independent human gold. No collected employee conversations or private communications were used. This is a project model with limited empirical validation.

Training:528 balanced rows,132/class;448 distinct target replies;80 same-reply/different-context pairs. All368 V3 rows, including its old validation and test partitions, are now training material. An additional160 rows target observed errors. Old V3 test scores must not be reported as unseen results for V4.
Validation:64 new rows with16 context pairs; label counts20/14/16/14. Final test:96 new rows,24/class,80 distinct target replies,16 context pairs and64 everyday rows. Both are checked for exact normalized target/context/input/group overlap with training and each other. Prior local and runtime CSV data are also audited. Full hashes and nearest textual neighbors are included. Exact overlap checks do not establish absence of semantic similarity, author's style, or wording shortcuts.

The new data includes full refusal versus partial acceptance, social preference versus task ownership, quoted/practice/past statements versus live uptake, conditional versus already-authorized commitments, and multi-turn clarifications. Same reply pairs remain in one split. Labels and group metadata are not serialized into model inputs.

## Training and selection
Both candidates initialize from the previous contextual V3 checkpoint (SHA256 a6f8b3fa7565136761183cacbd4d61e9b55e3eb411004368cdaf5130335d5009). Two predetermined learning rates:5e-6 and1e-5. Other settings:maximum6epochs,seed20261017,batch8,accumulation2,AdamW,weight decay.01,warmup.1,FP16,max256tokens. Complete pairs are kept within training batches by a group sampler; singleton rows are shuffled in two-row units. Training pairs use gold group identities only to construct a loss, never as input tokens.

Loss = ordinary four-class cross-entropy + .2 times a softplus margin penalty for each context twin. For labels a/b and twin logits A/B, penalty = [softplus(1-(A[a]-B[a]))+softplus(1-(B[b]-A[b]))]/2. It encourages each label's logit to be higher under its own context. Validation has ordinary cross-entropy only; the group keys are not model inputs. Checkpoint score is predetermined as .8*validation macro-F1 + .2*validation fraction of context pairs with both labels correct. Early stopping patience2. The best candidate is chosen by this score, then macro-F1, then smaller learning rate for a complete tie. Only this selected candidate is evaluated on the new final test.

## Evaluation and controls
The previous V3 contextual model, selected V4 model and three fixed TF-IDF+logistic-regression controls use the identical96-row final test. Lexical controls fit only training, with no held-out hyperparameter search. Predictions, class reports, all confusion matrices and both-correct pair scores are saved for every model. Same accuracy need not mean the same errors.

Blank-context inference and five seeded context shuffles are sensitivity checks. Their retained original labels are not validated semantic gold under altered context. Context-pair both-correct scores are the more direct test of context dependence. Neither these controls nor pair-aware training guarantee freedom from shortcuts.

The test was frozen before either candidate trained. No model, label, data or threshold changes were made after final-test inference. Every final row is retained. Confidence is uncalibrated softmax, not a reliability guarantee. All errors and high-confidence errors are exported.

## Label policy
No Commitment:no new work undertaking by the target speaker; includes complete refusal, factual knowledge, finished past work, another person's promise and merely social agreement.
Latent Commitment:live uptake inferred from relevant ongoing action or preparation in response to a request without direct acceptance wording.
Qualified Commitment:actual uptake limited by unresolved condition, uncertainty or partial scope. Refusing the remainder after accepting a part does not cancel that partial undertaking.
Explicit Commitment:direct unqualified acceptance of the requested work. A fully satisfied prerequisite is not an unresolved condition. Some pragmatic cases can admit alternate interpretations and need independent review.

## Loading the model
Unzip the model ZIP and use `best_model/`. AutoTokenizer.from_pretrained(path,use_fast=False), AutoModelForSequenceClassification.from_pretrained(path). Input serialization:`[CONTEXT] {context} [TARGET SPEAKER] employee [TARGET UTTERANCE] {utterance}`. Limit256tokens. The ordinary classifier can be used for inference without the custom training class or pair sampler. `project_predict.py` provides a small loader.

## Artifacts
Data/split audit and hashes; training logs and candidate selection; frozen final-test per-example predictions; context-pair metrics; previous/new error comparison; all selected-model errors; full lexical control predictions; perturbation logits/donors; model SHA256; notebook snapshot; runtime package versions.
'''
(VRES/'README.md').write_text(v_readme);(VBEST/'README.md').write_text(v_readme)
(VBEST/'project_predict.py').write_text(v_predictor_source);(VRES/'project_predict.py').write_text(v_predictor_source)
import importlib.metadata as v_im
(VRES/'requirements.txt').write_text('\n'.join(f'{p}=={v_im.version(p)}' for p in ['transformers','datasets','accelerate','sentencepiece','torch','numpy','pandas','scikit-learn'])+'\n')
print('WORKPLACE V4 COMPLETE — MODEL SELECTED USING VALIDATION ONLY')
print('Selected candidate:',v_best,'; fresh test frozen before training, all96 rows retained.')
print(v_df[['model','subset','correct','n','accuracy','macro_f1','errors_confidence_ge_0.90']].to_string(index=False))
print('Context pairs: BOTH CORRECT / PREDICTIONS CHANGE (out of16)')
print(pd.DataFrame(v_pairmetrics).groupby('model')[['both_correct','predictions_change']].sum().to_string())
print('Training528 balanced; validation64; fresh test96. Fictional examples; provisional labels.')
print('No exact cross-split or prior overlap detected; no test-based tuning. Shortcut freedom is not guaranteed.')
print('Selected weights SHA256:',v_selection['selected_model_sha256'])
from google.colab import _message
try:v_snapshot=_message.blocking_request('get_ipynb',timeout_sec=15)
except Exception:v_snapshot=None
if isinstance(v_snapshot,dict) and 'ipynb' in v_snapshot:(VRES/'Latent_Commitment_DeBERTa_Training.ipynb').write_text(json.dumps(v_snapshot['ipynb'],indent=1))
else:print('Notebook snapshot API unavailable; notebook remains in Drive.')
v_results_zip=shutil.make_archive(str(VROOT/'latent_commitment_workplace_v4_results'),'zip',VRES)
v_model_zip=shutil.make_archive(str(VROOT/'latent_commitment_workplace_v4_model'),'zip',VROOT,'best_model')
for archive in [v_results_zip,v_model_zip]:
 with zipfile.ZipFile(archive) as z:assert z.testzip() is None
 print('VERIFIED EXPORT:',archive)


In [ ]:
# Final read-only verification: cached results only; no test rerun.
assert (VRES/'final_test_logits.npz').exists() and (VRES/'evaluation_info.json').exists()
v_final_table=pd.read_csv(VRES/'test_metrics.csv')
print('V4 COMPLETE: SAME FROZEN96-EXAMPLE TEST, VALIDATION-ONLY MODEL SELECTION')
print(v_final_table[v_final_table.subset.eq('all')][['model','correct','n','accuracy','macro_f1']].to_string(index=False))
print('Context pairs, both labels correct out of16:')
print(pd.read_csv(VRES/'context_pair_metrics.csv').groupby('model').both_correct.sum().to_string())
print('528 balanced training rows;64 validation;96 fresh test. No post-test tuning or relabeling.')
print('No exact cross-split/prior overlap found. Fictional, provisionally labeled examples; shortcuts not ruled out.')
print('Model:',VBEST,'; selected SHA256:',v_selection['selected_model_sha256'])


In [ ]:
#@title Restore the saved V4 model and test a workplace conversation — run only this cell
#@markdown Edit the two fields below, then run this cell. **Do not use Run all**, which includes the earlier training experiments. A fresh runtime may ask you to authorize Google Drive. The verified model is restored into temporary runtime storage; your Drive backup stays intact.
workplace_context = "[manager]: Can you send the revised client summary by three this afternoon?" #@param {type:"string"}
employee_reply = "No, I can't take that on today." #@param {type:"string"}

# Self-contained inference: no Trainer, optimizer, dataset loading, or training calls.
import sys, subprocess, importlib.metadata, hashlib, json, zipfile, os
from pathlib import Path

_demo_requirements = {'transformers': '4.57.1', 'sentencepiece': '0.2.1'}
_demo_install = []
for _pkg, _ver in _demo_requirements.items():
    try:
        if importlib.metadata.version(_pkg) != _ver:
            _demo_install.append(f'{_pkg}=={_ver}')
    except importlib.metadata.PackageNotFoundError:
        _demo_install.append(f'{_pkg}=={_ver}')
if _demo_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_demo_install])

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_DEMO_WEIGHT_SHA = 'fc3f77b652ee7231430eb95d90d300b6496983fc37a1c42670017d8936b2a633'
_DEMO_ARCHIVE_SHA = 'b8ee949d2e5def582a3f6a9ed2f055e7f3ec844f2c87c74cbea3b795950137c1'
_DEMO_DRIVE_FOLDER = Path('/content/drive/MyDrive/Latent Commitment DeBERTa V4 - 95.83% - 2026-10-16')
_DEMO_BACKUP_URL = 'https://drive.google.com/drive/folders/11H35YyKLoY4QfVTjczrMemFetTx_Hsy5'
_DEMO_RUNTIME = Path('/content/latent_commitment_v4_restored')
_DEMO_FILE_HASHES = {'added_tokens.json': 'dc046d04c9b0ada7ae6f1dc89c465801799acdf0c9a6aab8c15a1b2d5ca4e91f', 'spm.model': 'c679fbf93643d19aab7ee10c0b99e460bdbc02fedf34b92b05af343b4af586fd', 'special_tokens_map.json': 'b2f1b2f15f29a6b6d9d6ea4eca1675d2c231a71477f151d48f79cc83a625ba21', 'tokenizer_config.json': 'e6c4c771911c211618a2d46488dc7e9e499f051773cd4ff4faeca6fe55fdf569', 'config.json': '0d08223a9f262d665df20b78fcacd8d675c027773cb9e6fd502d814540655055', 'model.safetensors': 'fc3f77b652ee7231430eb95d90d300b6496983fc37a1c42670017d8936b2a633'}
_DEMO_PARTS = [{'name': 'latent_commitment_workplace_v4_model.zip.part01', 'bytes': 94371840, 'sha256': '45a981b393204f45c94cab9f97e75680b17907de198a06a09e76f3869f5a7aa0'}, {'name': 'latent_commitment_workplace_v4_model.zip.part02', 'bytes': 94371840, 'sha256': '122451e863a7dbf3c9da64f7277f837a0f1a8f4c8eee224837a712c53a3ad64e'}, {'name': 'latent_commitment_workplace_v4_model.zip.part03', 'bytes': 94371840, 'sha256': 'd7a2ee3f6855863e5293e1ed044da036df4d1319e347d359c7b00d4ff7eeb30d'}, {'name': 'latent_commitment_workplace_v4_model.zip.part04', 'bytes': 94371840, 'sha256': '23c4af645d2284cbdb8cf2afc17e6ae6510409a7ec77bc73a85979e69f9d2802'}, {'name': 'latent_commitment_workplace_v4_model.zip.part05', 'bytes': 94371840, 'sha256': 'fe498d0cb8a2561f214fa5d491b1e156df39cae7ae5ee7849f7d144b0cef6f6c'}, {'name': 'latent_commitment_workplace_v4_model.zip.part06', 'bytes': 89925406, 'sha256': '8baa0c10d1aee2aa138a0f12df07d829e8a4c50b2d9da71e086b1aabaf613383'}]

def _demo_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        while chunk := source.read(8 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def _demo_verified_model(path):
    path = Path(path)
    return all((path / name).is_file() and _demo_sha256(path / name) == sha
               for name, sha in _DEMO_FILE_HASHES.items())

def _demo_restore_from_folder(backup_folder, runtime_folder):
    """Read immutable backup parts, verify, and extract only into runtime_folder."""
    backup_folder, runtime_folder = Path(backup_folder), Path(runtime_folder)
    missing = [p['name'] for p in _DEMO_PARTS if not (backup_folder / p['name']).is_file()]
    if missing:
        raise FileNotFoundError(
            'Drive backup parts were not found at ' + str(backup_folder) + '. '
            'Mount the Google account containing ' + _DEMO_BACKUP_URL +
            ' and keep all six parts in that folder. Missing: ' + ', '.join(missing))
    runtime_folder.mkdir(parents=True, exist_ok=True)
    partial = runtime_folder / 'verified_model.zip.incomplete'
    combined_hash = hashlib.sha256()
    try:
        with partial.open('wb') as destination:
            for number, part in enumerate(_DEMO_PARTS, 1):
                source_path = backup_folder / part['name']
                if source_path.stat().st_size != part['bytes']:
                    raise ValueError('Incomplete backup part: ' + part['name'])
                part_hash = hashlib.sha256()
                with source_path.open('rb') as source:
                    while chunk := source.read(8 * 1024 * 1024):
                        part_hash.update(chunk)
                        combined_hash.update(chunk)
                        destination.write(chunk)
                if part_hash.hexdigest() != part['sha256']:
                    raise ValueError('Checksum failed for ' + part['name'] + '. Download the original backup part again.')
                print(f'Verified Drive backup part {number}/6', flush=True)
        if combined_hash.hexdigest() != _DEMO_ARCHIVE_SHA:
            raise ValueError('Combined model archive checksum failed.')
        archive = runtime_folder / 'verified_model.zip'
        partial.replace(archive)
        with zipfile.ZipFile(archive) as saved:
            if saved.testzip() is not None:
                raise ValueError('The saved ZIP failed its integrity check.')
            base = runtime_folder.resolve()
            for entry in saved.infolist():
                target = (runtime_folder / entry.filename).resolve()
                if not target.is_relative_to(base) or not entry.filename.startswith('best_model/'):
                    raise ValueError('Unexpected archive path: ' + entry.filename)
            saved.extractall(runtime_folder)
        model_dir = runtime_folder / 'best_model'
        if not _demo_verified_model(model_dir):
            raise ValueError('Restored model or tokenizer failed verification.')
        print('Restored the exact saved V4 model and tokenizer; no training performed.', flush=True)
        return model_dir
    finally:
        # Only remove our own unfinished runtime ZIP after an interrupted transfer.
        if partial.exists():
            partial.unlink()

# Reuse a verified in-memory model when rerunning this cell with a new conversation.
if (globals().get('_commitment_v4_loaded_sha') != _DEMO_WEIGHT_SHA or
        '_commitment_v4_demo_model' not in globals() or '_commitment_v4_demo_tokenizer' not in globals()):
    _demo_model_dir = None
    for _candidate in [_DEMO_RUNTIME / 'best_model', Path('/content/latent_commitment_workplace_v4/best_model')]:
        if _demo_verified_model(_candidate):
            _demo_model_dir = _candidate
            print('Using verified V4 model already in this runtime:', _candidate, flush=True)
            break
    if _demo_model_dir is None:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').is_dir():
            print('Authorize Google Drive using the account that contains the saved backup.', flush=True)
            drive.mount('/content/drive')
        _demo_model_dir = _demo_restore_from_folder(_DEMO_DRIVE_FOLDER, _DEMO_RUNTIME)
    _commitment_v4_demo_tokenizer = AutoTokenizer.from_pretrained(
        _demo_model_dir, use_fast=False, local_files_only=True, trust_remote_code=False)
    _commitment_v4_demo_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    _commitment_v4_demo_model = AutoModelForSequenceClassification.from_pretrained(
        _demo_model_dir, local_files_only=True, trust_remote_code=False, use_safetensors=True
    ).to(_commitment_v4_demo_device).eval()
    _commitment_v4_loaded_sha = _DEMO_WEIGHT_SHA

@torch.inference_mode()
def predict_workplace(context, reply):
    """Predict the employee's new reply using earlier conversation as context."""
    if not isinstance(context, str) or not isinstance(reply, str) or not context.strip() or not reply.strip():
        raise ValueError('Enter both the earlier conversation and the employee reply.')
    text = f'[CONTEXT] {context.strip()} [TARGET SPEAKER] employee [TARGET UTTERANCE] {reply.strip()}'
    inputs = _commitment_v4_demo_tokenizer(text, return_tensors='pt', truncation=False)
    tokens = int(inputs['input_ids'].shape[1])
    if tokens > 256:
        raise ValueError(f'This conversation has {tokens} tokens; the limit is 256. Shorten the earlier context. Nothing was silently truncated.')
    inputs = {k: v.to(_commitment_v4_demo_device) for k, v in inputs.items()}
    probabilities = _commitment_v4_demo_model(**inputs).logits.float().softmax(-1)[0].cpu().tolist()
    label_id = max(range(4), key=probabilities.__getitem__)
    labels = _commitment_v4_demo_model.config.id2label
    return {'label': labels[label_id], 'label_id': label_id,
            'score': probabilities[label_id], 'tokens': tokens,
            'probabilities': {labels[i]: p for i, p in enumerate(probabilities)}}

_demo_result = predict_workplace(workplace_context, employee_reply)
print('\nPREDICTION:', _demo_result['label'].replace('_', ' '))
print('Class scores:')
for _label, _score in _demo_result['probabilities'].items():
    print(f'  {_label.replace("_", " "):24s} {_score:7.2%}')
print('Scores are uncalibrated; review ambiguous conversations. This cell performs inference only.')
print('Edit the two fields above and rerun this cell for another conversation.')
print('You can also call predict_workplace(context, reply) in a new code cell.')
